<a href="https://colab.research.google.com/github/Maverick-Ansh/recurrent_looped_transformer_scratch/blob/main/scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recurrent Looped Transformer — a computational dissection

**Paper:** *Recurrent Looped Transformer*, Yifan Zhang, Jichen Feng, Shihan Qin — September 12, 2026.

We are not implementing this paper. We are **opening it up**: reconstructing the computation from first principles so that questions like *"what exactly is in the recurrent state at token t"*, *"what carries from t to t+1"*, *"what does the gate do numerically"* become things we **measure**, not things we assert.

---

### The hard constraint

**Pure Python.** No PyTorch, TensorFlow, JAX, Flax, Keras, HuggingFace, autograd — and **no NumPy**. Lists, floats, ints, dicts. The only import anywhere is `math`.

Everything is written inline in this notebook. There is no hidden module to jump to.

---

### Status

| Phase | What | State |
|---|---|---|
| **0** | Read the paper as a scientist | ✅ this notebook, §0 |
| **1** | Turn it into a computation graph | ✅ §1 |
| **2** | Build the mathematical primitives | ✅ §2, 67 invariants passing |
| 3 | Tiny RLT (d=8, V=10, L=1+1) | next |
| 4–9 | Full forward trace, shape tracker, trajectories, recurrence / cache / attention dissection | |
| 10–12 | Manual gradients, BPTT through the recurrence, parameter + gradient dissection | |
| 13–18 | Information flow, claim verification, ablation lab, write-up | |

Mirror repo (notes + a `.py` copy of §2): **[Maverick-Ansh/recurrent_looped_transformer_scratch](https://github.com/Maverick-Ansh/recurrent_looped_transformer_scratch)**

---

## ⚠️ Read this before anything else: **the paper has no experiments**

It says so three times:

- §1 — *"The report develops these mechanisms; **it does not report measured efficiency or scaling results**."*
- §3.2 — *"no reduced-prefill speedup is claimed."*
- §8 — *"The computational definitions are explicit; realized reasoning quality, hardware efficiency, and scaling behavior **require future validation**."*

Grep of the full 19-page text confirms it:

| searched | hits |
|---|---|
| `512`, `1365`, `4+4`, `accuracy`, `we train`, `we evaluate` | **0** |
| `parity` | 1 — and it is *"does not prove numerical kernel **parity**"* (§5.3), about kernel numerics, **not** the parity task |

There is no task, no dataset, no baseline, no results table, no hyperparameter appendix.

**So the config "width 512, FFN 1365, 4 heads, 8 layers, SWA 8, α=0.1, splits 4+4…8+0" is *not* from this paper.** It matches `C:\Users\ANSH\rlt-reproduce\rlt\config.py` — an earlier reproduction, i.e. a *previously chosen* experimental design. Recorded here so it never gets laundered into "the paper says".

**What this changes:** there is nothing to re-run. There are **propositions to verify** (§3.1, App. B.1) and a set of stated structural non-equivalences (§2.4, §2.6, App. B, App. C). Those are *exactly* what a pure-Python implementation can check **exactly** — we control every float. Any task or config in this notebook is **ours**, and is labelled as ours.

---

### Tagging convention, used everywhere below

- **[PAPER]** — stated in the paper, with section/equation.
- **[UNSPECIFIED]** — the paper does not say.
- **[ASSUMPTION]** — our choice, because we need a number to run code.

Paper facts and our choices are never mixed.

# §0 — Phase 0: the paper, extracted

## 0.1 Notation

| Symbol | Type | Meaning | §
|---|---|---|---|
| $x_{1:S}$ | token ids, $x_1=\text{BOS}$ | one independent sequence | 2.1 |
| $T$ | int | serving boundary — *"marks a serving boundary, **not a change in the conditional model**"* | 2.1 |
| $d$ | int | residual width | 2.1 |
| $L_E, L_D$ | int | encoder / decoder depth | 2.1 |
| $e_t \in \mathbb{R}^d$ | vector | encoder representation | 2.1 |
| $s_t \in \mathbb{R}^d$ | vector | recurrent decoder output | 2.1 |
| $C^D_t$ | per-layer (k,v) lists | retained decoder SWA KV at **every** layer | 2.1 |
| $H_t=(s_t, C^D_t)$ | pair | **complete decoder state** | 2.1 |
| $W \ge 1$ | int | SWA window — **includes the current token** | 2.1 |
| $G \in [1, L_D]$ | int | encoder-memory groups; $G{=}1$ shares, $G{=}L_D$ per-layer | 2.2 |
| $M^g_{\le t}$ | growing (k,v) set | encoder-derived cross-attention memory | 2.2 |
| $s_\star$ | vector | **learned** initial state, part of $\Theta$ | 2.3 |
| $\alpha$ | scalar | feedback scale | 2.5 |
| $\mu$ | dist | **behavior** policy (sampler) ≠ $p_\Theta$ | 5.3 |

⚠️ The paper overloads $W$: the scalar window **and** the matrices $W_g, W_s, W_V, W_o$. We always subscript matrices.

---

## 0.2 Every equation

**Encoder + memory**

$$e_{1:T} = E_\theta(x_{1:T}) \tag{2.1}$$

$$k^g_t = \mathcal{P}^g_K(e_t, t), \qquad v^g_t = W^g_V\,\mathrm{RMSNorm}_E(e_t), \qquad M^g_{\le t} = \{(k^g_j, v^g_j)\}_{j=1}^{t} \tag{2.2}$$

> [PAPER] *"The key map includes normalization, projection, and any positional transformation."* — so $\mathcal{P}_K$ = norm → linear → positional. Values get norm + linear but **no** positional op.
> [PAPER] *"This global memory depends on encoder representations, **not decoder states**."* ← the crucial asymmetry.

**The recurrence**

$$H_0 = (s_\star, \varnothing) \tag{2.3}$$
$$u_t = \mathrm{Merge}(e_t, s_{t-1}) \tag{2.4}$$
$$H_t = (s_t, C^D_t) = D_\phi(u_t;\, M_{\le t},\, C^D_{t-1},\, t), \quad t \ge 1 \tag{2.5}$$
$$p_\Theta(x_{t+1}\mid x_{1:t}) = \mathrm{softmax}\!\left(W_o\,\mathrm{RMSNorm}_o(s_t)\right)_{x_{t+1}} \tag{2.6}$$
$$H_T = F_T \circ F_{T-1} \circ \cdots \circ F_1(H_0) \tag{2.7}$$

> [PAPER] §2.3: *"**Neither component** of $H_T$ is reset at the serving boundary."*

**The merge (the gate)**

$$r_{t-1} = \mathrm{RMSNorm}_s(s_{t-1}) \tag{2.9}$$
$$g_t = \sigma\!\left(W_g[e_t; r_{t-1}] + b_g\right) \tag{2.10}$$
$$u_t = e_t + \alpha\, g_t \odot W_s r_{t-1} \tag{2.11}$$

with $W_g \in \mathbb{R}^{d\times 2d}$, $W_s \in \mathbb{R}^{d\times d}$, $g_t \in (0,1)^d$.

> ⚠️ **Read (2.11) again.** $W_s$ multiplies $r_{t-1}$ — the **normalized** state — not $s_{t-1}$. A trace that prints "$W_s s_{t-1}$" is printing a different quantity. We implement the paper and will print both.
> [PAPER] *"$\alpha$ controls the feedback scale. A modest nonzero initial feedback scale is a **candidate** initialization, **not an established stability prescription**."* — the paper declines to pick a value.

**The decoder block** — order is SWA → cross-attention → FFN, with $z^0_t = u_t$:

$$q^{D,\ell}_t = \mathcal{P}^{D,\ell}_Q(z^{\ell-1}_t, t) \tag{2.12}$$
$$k^{D,\ell}_t = \mathcal{P}^{D,\ell}_K(z^{\ell-1}_t, t), \qquad v^{D,\ell}_t = W^{D,\ell}_V \mathrm{RMSNorm}_{S,\ell}(z^{\ell-1}_t) \tag{2.13}$$
$$b^\ell_t = z^{\ell-1}_t + \mathrm{Attn}^D_\ell\!\left(q^{D,\ell}_t,\ \{(k^{D,\ell}_j, v^{D,\ell}_j)\}_{j=\max(1,\,t-W+1)}^{t}\right) \tag{2.14}$$
$$a^\ell_t = b^\ell_t + \mathrm{Attn}^M_\ell\!\left(\mathcal{P}^{M,\ell}_Q(b^\ell_t, t),\ M^{g(\ell)}_{\le t}\right) \tag{2.15}$$
$$z^\ell_t = a^\ell_t + \mathrm{FFN}_\ell\!\left(\mathrm{RMSNorm}_{D,\ell}(a^\ell_t)\right), \qquad s_t = z^{L_D}_t \tag{2.16}$$

> [PAPER] *"The attention operators include their output projections; query/key maps include their respective normalizations and positional transformations."*
> [PAPER] *"At each layer, current KV is formed **before** SWA, using the layer input, so current-position attention introduces **no circular dependency**."* — no fixpoint, no iteration.
> [PAPER] retention: *"After the update, retain positions $\max(1, t-W+2), \dots, t$ in $C^D_t$; this set is **empty for $W=1$**."*

**What "looped" means** — §2.6: the reference tied config sets $L_E = L_D = L$ and **shares** Q/K/V/O and FFN between encoder layer $\ell$ and decoder SWA layer $\ell$. Cross-attention gets separate query/output projections. So: **one backbone, two logical passes**, $2L$ block evaluations per token.

> [PAPER] *"This is parameter reuse with **different attention wiring**, not activation copying. No decoder output is identified with an encoder output."*

## 0.3 The four stores — and the fact that decides everything downstream

| # | Object | Shape | Bound | Evicts? | Depends on |
|---|---|---|---|---|---|
| 1 | $s_t$ | $[d]$ | one vector | overwritten | $x_{1:t}, \Theta$ |
| 2 | $C^D_t$ | $L_D \times (\le W{-}1) \times (k,v)$ | **bounded** | **yes** | $x_{1:t}, \Theta$ |
| 3 | $M^g_{\le t}$ | $G \times t \times (k,v)$ | unbounded | no | $x_{1:t}, \theta$ **only** |
| 4 | $C^E_t$ | encoder causal KV | unbounded | no | $x_{1:t}, \theta$ only |

> [PAPER] §4.2: *"Decoder layers read this memory and separately append decoder-derived KV to their bounded layerwise SWA caches. **These caches cannot be replaced by encoder memory.**"*

---

### ⚡ There are **two** recurrent channels, not one

Almost every mental model of this architecture has one backward arrow: $s_t \to s_{t+1}$. That is **wrong**, and Appendix B says so with a $2\times2$ Jacobian:

$$J_t = \frac{\partial H_t}{\partial H_{t-1}} = \begin{bmatrix} \dfrac{\partial s_t}{\partial s_{t-1}} & \dfrac{\partial s_t}{\partial C^D_{t-1}} \\[2ex] \dfrac{\partial C^D_t}{\partial s_{t-1}} & \dfrac{\partial C^D_t}{\partial C^D_{t-1}} \end{bmatrix} \tag{B.3}$$

$$\frac{\partial H_t}{\partial H_j} = J_t J_{t-1}\cdots J_{j+1}, \qquad j < t \tag{B.2}$$

> [PAPER] **"A product involving only $\partial s_t/\partial s_{t-1}$ generally misses paths through decoder KV. Normalization alone does not bound products of these Jacobians."**

So the state crossing the token boundary is $(s_t, C^D_t)$ — the vector **and** every decoder layer's sliding window.

```
   s_{t-1} ──────────────► MERGE ──► u_t ──► decoder ──► s_t ────────►  channel 1
                                       ▲         │
   C^D_{t-1} ──────────────────────────┘         └─────► C^D_t ───────►  channel 2
```

### 🔧 Design correction this forces on us

**Setting $\alpha = 0$ does *not* give a non-recurrent model.** It severs channel 1 (equation 2.11 collapses to $u_t = e_t$) and leaves channel 2 running at full strength.

A genuine non-recurrence control needs **both**:

$$\alpha = 0 \quad\textbf{and}\quad W = 1$$

because the paper states the retained set is *empty* at $W=1$. So the Phase-7 ablation is a **2×2 over $(\alpha, W)$**, not a single knob. (The earlier `rlt-reproduce` run used $\alpha=0$ alone as "the" recurrence ablation — that is the channel-1 ablation only.)

Similarly:
- **Zeroing $s_{t-1}$ does not erase history** — the window still holds $W-1$ decoder-derived KV entries.
- **Eviction is not forgetting.** [PAPER] App. B: *"Evicted entries can still influence later computation through states or retained activations that previously consumed them."*
- **Eviction is not a stop-gradient.** [PAPER] App. C: *"SWA eviction limits future direct access to old KV; **it is not itself a stop-gradient operation**. Full BPTT still differentiates computations that consumed those entries before eviction. The inference cache size therefore **does not bound** full-BPTT activation storage."*

---

## 0.4 The two propositions

**Prop 3.1 — invariance to the serving split.** Batched prefill + recurrent decoding ≡ fully incremental processing; *"Moving the prompt–response split does not change the conditional distribution for a fixed token history."* Proof is a one-line induction: both start at $H_0$; identical ops on identical inputs. *"The serving split never appears in the transition."*
→ In pure Python with one code path, the only way this fails is if **we** leak the split. That is what makes it a good test.

**Prop B.1 — causality.** $H_t$ depends only on $x_{1:t}$ and $\Theta$.
→ Checkable by perturbation: change $x_{t+1}$, assert $H_t$ is **bit-identical**. The cleanest experiment in the paper.

---

## 0.5 Claims extracted for Phase 14 — to be **tested**, not assumed

| ID | Claim | Source |
|---|---|---|
| C1 | State path composes $t$ transitions / $t\,L_D$ blocks; per-token count fixed at $L_E{+}L_D$ | §3.3, Fig 2 |
| C2 | Prompt and response use the same transition; moving $T$ changes nothing | Prop 3.1 |
| C3 | $H_t$ depends only on $x_{1:t}$ | Prop B.1 |
| C4 | **Both** $s$ and $C^D$ cross the boundary unreset | §2.3 |
| C5 | Encoder memory is prefix-restricted **per decoder position** | §2.4 |
| C6 | Cached states under old parameters are not current-policy states | §5.4, App C |
| C7 | Detaching $s$ alone is **not** full BPTT | App B, C |
| C8 | retention $[\max(1,t{-}W{+}2), t]$ + current == next read window $[\max(1,t{-}W{+}1), t{+}1]$ | §2.5 |
| C9 | An $\partial s/\partial s$-only product **misses** the KV paths | App B (B.3) |
| C10 | Gradient reach **>** cache reach | App C |
| C11 | $M$ never depends on decoder states | §2.2 |

C9 and C10 are the best targets: both are stated **without proof** and both are easy to get wrong in an implementation.

**C8 is already verified below in §2.5.**

---

## 0.6 What the paper does **not** specify

Every one of these needs a labelled choice from us.

| | Quantity | Paper's constraint | **[ASSUMPTION]** (tiny model) |
|---|---|---|---|
| A1 | $d$ | none | `8` |
| A2 | $V$ | none | `10` |
| A3 | $L_E, L_D$ | none (48+48 is *"illustrative"*, Fig 2) | `1, 1` |
| A4 | heads | **never mentioned at all** | `2` |
| A5 | $d_{ff}$ | none | `16` |
| A6 | FFN activation | none | GELU (both exact + tanh built) |
| A7 | $W$ | $W\ge1$ | `3` |
| A8 | $G$ | $1\le G\le L_D$ | `1` |
| A9 | $\alpha$ | *"modest nonzero"* | `0.1`, swept in Phase 7 |
| A10 | RMSNorm $\epsilon$ | none | `1e-5` |
| A11 | attention scale | none | $1/\sqrt{d_\text{head}}$ |

**Structural choices left open — implement *both* where practical:**

| | Choice | Paper's words | Plan |
|---|---|---|---|
| B1 | positional transform in $\mathcal{P}_Q,\mathcal{P}_K$ | *"any positional transformation"* | NoPE **and** RoPE; NoPE default so position doesn't confound state effects |
| B2 | tied vs untied $E/D$ | §2.6 tied is "reference"; untied *"preserves the complete-state recurrence"* | untied first (fewer confounds), tied as a flag |
| B3 | gate shape | vector $g_t$, or *"scalar gating"* | vector; scalar as a flag |
| B4 | $W_s$ rank | full, or *"low-rank"* | full |
| B5 | RMSNorm learned gain | unstated | with gain, init to 1 → no-gain variant **is** the init |

**Entirely absent:** optimizer, LR, schedule, batch size, init scheme, dropout, tokenizer, data, sequence length, $s_\star$ init. All become logged config fields so no result is ever read as "the paper's".

**Genuine ambiguities, flagged not hidden:**
1. $M_{\le t}$ must be **sliced per decoder position**, not just appended. §4.2: *"a faster kernel that reads future entries **changes the model**."*
2. Head split in cross-attention — [UNSPECIFIED]. [ASSUMPTION] same head count as SWA.
3. Does BOS get a decoder update? **Yes** — $t{=}1$ is BOS, produces $H_1$, predicts $x_2$. Consistent with (5.1) summing from $t{=}1$.
4. Is $\alpha$ trainable? [UNSPECIFIED]. [ASSUMPTION] fixed, so $\alpha{=}0$ is an exact clean ablation.

# §1 — Phase 1: the computation graph

Shapes below are for the **tiny config** of Phase 3, so every number is one you can print:

```
V = 10    d = 8    L_E = 1    L_D = 1    H = 2    d_head = 4
d_ff = 16    W = 3    G = 1    alpha = 0.1
```

All [ASSUMPTION] — see §0.6.

---

## 1.1 Encoder path (per token t)

```
x_t                                    int in [0,10)
  ↓ embedding lookup E_tok[x_t]
h_t^0                                  [8]    initial residual stream
  ↓ RMSNorm  →  W_Q/W_K/W_V  →  split heads
q,k,v                                  [2,4]  each
  ↓ append (k,v) to encoder cache C^E          ← STORE 4, grows forever
  ↓ causal attention over j = 1..t
score[h][j] = <q_t[h], k_j[h]> / sqrt(4)   [2,t]
  ↓ softmax over j
p[h][j]                                [2,t]  sums to 1 along j
  ↓ sum_j p[h][j] · v_j[h] → flatten → W_O
  ↓ residual add
b_t                                    [8]
  ↓ RMSNorm → FFN (8→16→8, GELU) → residual add
h_t^1  =  e_t                          [8]    ENCODER REPRESENTATION   (2.1)
```

**Meaning of $e_t$:** a causal, **non-recurrent** feature of $x_{1:t}$. It has *never seen a decoder state*. That asymmetry is what makes the encoder parallelizable (§4.1) and is what Prop B.1's proof leans on.

**Memory projection (2.2):**
```
e_t → RMSNorm_E → P_K^g (norm,linear,positional) → k_t^g   [2,4]
                → W_V^g  (no positional op)       → v_t^g   [2,4]
                → append to M^g                             ← STORE 3, unbounded
```

---

## 1.2 The merge — the entire recurrence for channel 1, in three lines

```
s_{t-1}                                [8]   (s_* at t=1)
  ↓ RMSNorm_s :  r_i = s_i / sqrt(mean(s²)+eps) · gain_i        (2.9)
r_{t-1}                                [8]

e_t, r_{t-1}
  ↓ concat                                                      (2.10)
[e_t ; r_{t-1}]                        [16] = [2d]
  ↓ W_g @ .  + b_g          W_g is [8,16]
gate preactivation                     [8]
  ↓ sigma elementwise
g_t                                    [8]   ∈ (0,1)^8   ← THE GATE

r_{t-1}
  ↓ W_s @ .                 W_s is [8,8]                        (2.11)
W_s r_{t-1}                            [8]
  ↓ ⊙ g_t
g_t ⊙ W_s r_{t-1}                      [8]
  ↓ × alpha
alpha · g_t ⊙ W_s r_{t-1}              [8]   ← FEEDBACK CONTRIBUTION
  ↓ + e_t
u_t                                    [8]   ← MERGED DECODER INPUT
```

$\alpha = 0$ kills the entire third block. $u_t = e_t$ exactly. $g_t$ is still computed and still depends on $r_{t-1}$, but is multiplied by zero.

---

## 1.3 Decoder block ($z^0_t = u_t$)

**A — causal SWA (2.12–2.14).** The step to watch:

```
z_t^{l-1}  →  P_Q (norm,W_Q,pos)  →  q   [2,4]
           →  P_K (norm,W_K,pos)  →  k   [2,4]
           →  RMSNorm_S → W_V     →  v   [2,4]        ← note: no positional op on v

C^D_{t-1}[l]        ≤ W-1 = 2 entries                 ← STORE 2
  ↓ read window = C^D_{t-1}[l] ++ [(k_t,v_t)]
read window         j ∈ [max(1,t-W+1), t], ≤ 3 entries

  ↓ score[h][j] = <q[h],k_j[h]>/sqrt(4) → softmax over j
  (no mask needed — THE WINDOW *IS* THE MASK)
  ↓ sum_j p·v → flatten → W_O → residual add
b_t^l = z_t^{l-1} + swa_out            [8]                      (2.14)

  ↓ retain [max(1,t-W+2), t]  (drop oldest if full)
C^D_t[l]            ≤ W-1 = 2 entries
```

**B — cross-attention to encoder memory (2.15).**

```
b_t^l → P_Q^M → cross query    [2,4]
M_{<=t}^{g(l)}                 [t][2,4]×2    ← PREFIX-RESTRICTED, sliced to t
  ↓ scores over j=1..t → softmax → weighted V → W_O^M → residual add
a_t^l = b_t^l + cross_out      [8]                              (2.15)
```

The prefix restriction is a **model property, not an optimization**. §2.4: *"At decoder position t, attention is restricted to $M_{\le t}$ even though the entire prompt memory is available."*

**C — FFN (2.16).**
```
a_t^l → RMSNorm_D → W_1 [16,8] → GELU → W_2 [8,16] → residual add
z_t^l                          [8]
                     s_t = z_t^{L_D}    ← THE RECURRENT STATE
```

**Readout (2.6).** `s_t → RMSNorm_o → W_o [10,8] → logits [10] → softmax → p(x_{t+1}|x_{1:t})`

---

## 1.4 The recurrent path, unrolled — **both** channels, nothing hidden

```
H_0 = (s_*, ∅)
 │  F_1 : u_1 = Merge(e_1, s_*)   → decoder →
 ▼
H_1 = (s_1, C^D_1)     C^D_1[l] = [ (k_1,v_1) ]                     W=3
 │  F_2 : u_2 = Merge(e_2, s_1)   → decoder →
 ▼
H_2 = (s_2, C^D_2)     C^D_2[l] = [ (k_1,v_1), (k_2,v_2) ]
 │  F_3 : u_3 = Merge(e_3, s_2)   → decoder →
 ▼
H_3 = (s_3, C^D_3)     C^D_3[l] = [ (k_2,v_2), (k_3,v_3) ]   ← (k_1,v_1) EVICTED
 │  F_4 : u_4 = Merge(e_4, s_3)   → decoder →
 ▼
H_4 = (s_4, C^D_4)     C^D_4[l] = [ (k_3,v_3), (k_4,v_4) ]
```

### Where does $x_1$ survive at $t=4$? Three routes — enumerating them **is** Phase 13.

1. ❌ **Not** in $C^D_4$ — $(k_1,v_1)$ was evicted at $t=3$.
2. ✅ In $M_{\le 4}$ — encoder memory never evicts.
3. ✅ In $s_4$, **indirectly**: $(k_1,v_1)$ was read by SWA at $t=1,2,3$, shaping $s_1,s_2,s_3$; and $s_3$ feeds $u_4$.

Route 3 is exactly App. C's *"evicted entries can still influence later computation through states … that previously consumed them."* Route 3 is **also** why eviction is not a stop-gradient (C10).

---

## 1.5 Block-count accounting (claim C1)

```
per token:      L_E + L_D  blocks          ← FIXED, independent of t
along chain:    t · L_D    decoder blocks  ← GROWS with t
```

Paper's illustrative $L_E{=}L_D{=}48$: 96 blocks/token, $48t$ along the chain — exactly Fig. 2. Our tiny config: 2 blocks/token, $t$ along the chain. Phase 14 tests this by **counting actual block invocations**, not by re-deriving the formula.

# §2 — Phase 2: the mathematical primitives

Everything below is written from scratch. The **only** import in this entire section is `math`.

Two rules held throughout:

1. **`attention()` returns `(out, scores, probs)` — all three.** A function that returned only `out` would be exactly the opaque helper this project exists to avoid. Phase 9 inspects every intermediate, so the intermediates must escape.
2. **Initialization has no hidden machinery.** A 10-line linear congruential generator + Box–Muller, rather than inheriting a Mersenne Twister — so the path from bits to weights is readable, and the same seed gives the same weights on any platform.

Data types, fixed here for the whole project:

| name | Python type | shape |
|---|---|---|
| scalar | `float` | |
| vector | `list[float]` | `d` |
| matrix | `list[list[float]]` | `[d_out][d_in]` — paper uses **column vectors**, so `matvec(M,x)` is $Mx$ |
| heads | `list[list[float]]` | `[H][d_head]` |
| kv entry | `(pos, k, v)` | |
| swa cache | `list[list[kv]]` | `[layer][slot]` |
| attn scores | `list[list[float]]` | `[H][n_keys]` |

Run the cells in order — each builds on the previous.

In [2]:
# =============================================================================
# 2.1  THE PURITY GUARD, then vectors, matrices, reductions
# =============================================================================
import math          # <-- the ONLY import in all of section 2
import sys

# A guard on sys.modules would measure the HOST, not this code: Colab pre-imports
# numpy before we ever run. So we snapshot what is ALREADY loaded, and later assert
# that *we* never bind any of it.
_PRELOADED = set(sys.modules)
BANNED = {"numpy", "torch", "tensorflow", "jax", "flax", "keras",
          "transformers", "scipy", "sklearn", "pandas"}

def assert_pure(namespace):
    """Fail if any banned library leaked into OUR namespace (not the host's)."""
    leaked = []
    for name, obj in list(namespace.items()):
        mod = getattr(obj, "__module__", None) or getattr(obj, "__name__", "")
        root = str(mod).split(".")[0]
        if root in BANNED:
            leaked.append((name, root))
    assert not leaked, f"PURITY VIOLATION: {leaked}"
    return (f"pure: none of {sorted(BANNED)} is bound in this notebook "
            f"(host had {len(_PRELOADED)} modules preloaded -- irrelevant, that is Colab's)")

NEG_INF = float("-inf")

# ------------------------------------------------------------------ vectors --
def vadd(a, b):
    """(a+b)_i = a_i + b_i"""
    assert len(a) == len(b), f"vadd mismatch {len(a)} vs {len(b)}"
    return [a[i] + b[i] for i in range(len(a))]

def vsub(a, b):
    """(a-b)_i = a_i - b_i"""
    assert len(a) == len(b), f"vsub mismatch {len(a)} vs {len(b)}"
    return [a[i] - b[i] for i in range(len(a))]

def vscale(c, a):
    """(c*a)_i = c*a_i"""
    return [c * a[i] for i in range(len(a))]

def vmul(a, b):
    """(a (*) b)_i = a_i*b_i  -- the elementwise product of eq (2.11)."""
    assert len(a) == len(b), f"vmul mismatch {len(a)} vs {len(b)}"
    return [a[i] * b[i] for i in range(len(a))]

def vneg(a):        return [-x for x in a]
def vzeros(n):      return [0.0] * n

def vsum(a):
    """sum_i a_i -- written as a loop, not sum(), so the accumulation is visible."""
    tot = 0.0
    for x in a:
        tot += x
    return tot

def dot(a, b):
    """<a,b> = sum_i a_i*b_i"""
    assert len(a) == len(b), f"dot mismatch {len(a)} vs {len(b)}"
    tot = 0.0
    for i in range(len(a)):
        tot += a[i] * b[i]
    return tot

# ----------------------------------------------------------------- matrices --
def matvec(M, x):
    """y = Mx,  y_i = sum_j M_ij x_j.   M is [d_out][d_in], x is [d_in]."""
    assert len(M) > 0 and len(M[0]) == len(x), \
        f"matvec mismatch: M=[{len(M)},{len(M[0])}] x=[{len(x)}]"
    return [dot(M[i], x) for i in range(len(M))]

def matmul(A, B):
    """C = AB,  C_ij = sum_k A_ik B_kj."""
    assert len(A[0]) == len(B), f"matmul mismatch: A cols {len(A[0])} vs B rows {len(B)}"
    n, k, m = len(A), len(B), len(B[0])
    C = [[0.0] * m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            acc = 0.0
            for p in range(k):
                acc += A[i][p] * B[p][j]
            C[i][j] = acc
    return C

def transpose(M):   return [[M[i][j] for i in range(len(M))] for j in range(len(M[0]))]
def mzeros(r, c):   return [[0.0] * c for _ in range(r)]

# --------------------------------------------------- reductions / statistics --
def mean(a):  return vsum(a) / len(a)

def var(a):
    m = mean(a)
    return vsum([(x - m) ** 2 for x in a]) / len(a)

def rms(a):
    """rms(a) = sqrt( (1/n) sum_i a_i^2 )

    This is the UNCENTERED second moment -- it does NOT subtract the mean.
    That is precisely what separates RMSNorm from LayerNorm.
    """
    return math.sqrt(vsum([x * x for x in a]) / len(a))

def l2(a):
    """||a||_2 = sqrt(sum_i a_i^2).   Relation: ||a|| = sqrt(n) * rms(a)."""
    return math.sqrt(vsum([x * x for x in a]))

def vmin(a):
    m = a[0]
    for x in a:
        if x < m: m = x
    return m

def vmax(a):
    m = a[0]
    for x in a:
        if x > m: m = x
    return m


print(assert_pure(globals()))
print()
u = [3.0, -1.0, 0.0, 4.0]
w = [1.0, 2.0, -2.0, 0.5]
print(f"u          = {u}")
print(f"w          = {w}")
print(f"u + w      = {vadd(u, w)}")
print(f"u (*) w    = {vmul(u, w)}        <- eq (2.11) uses this")
print(f"<u,w>      = {dot(u, w)}")
print(f"mean(u)    = {mean(u)}")
print(f"rms(u)     = {rms(u):.6f}   (uncentered -- mean is NOT subtracted)")
print(f"l2(u)      = {l2(u):.6f}   == sqrt(4)*rms(u) = {math.sqrt(4)*rms(u):.6f}")
M = [[1.0, 0.0, 2.0, 0.0], [0.0, 1.0, 0.0, -1.0]]
print(f"\nM ({len(M)}x{len(M[0])}) @ u = {matvec(M, u)}    <- row i is <M[i], u>")

pure: none of ['flax', 'jax', 'keras', 'numpy', 'pandas', 'scipy', 'sklearn', 'tensorflow', 'torch', 'transformers'] is bound in this notebook (host had 1311 modules preloaded -- irrelevant, that is Colab's)

u          = [3.0, -1.0, 0.0, 4.0]
w          = [1.0, 2.0, -2.0, 0.5]
u + w      = [4.0, 1.0, -2.0, 4.5]
u (*) w    = [3.0, -2.0, -0.0, 2.0]        <- eq (2.11) uses this
<u,w>      = 3.0
mean(u)    = 1.5
rms(u)     = 2.549510   (uncentered -- mean is NOT subtracted)
l2(u)      = 5.099020   == sqrt(4)*rms(u) = 5.099020

M (2x4) @ u = [3.0, -5.0]    <- row i is <M[i], u>


In [3]:
# =============================================================================
# 2.2  NONLINEARITIES  -- softmax, log-softmax, sigma (eq 2.10), GELU
# =============================================================================

def softmax(z):
    """p_i = exp(z_i) / sum_j exp(z_j)

    Computed as exp(z_i - max z) / sum_j exp(z_j - max z). Algebraically
    IDENTICAL (the exp(max) factor cancels top and bottom) but it cannot
    overflow. Entries equal to -inf (masked positions) map to exactly 0.0.
    """
    m = vmax(z)
    if m == NEG_INF:
        raise ValueError("softmax over an all-masked row: every logit is -inf")
    exps = [0.0 if x == NEG_INF else math.exp(x - m) for x in z]
    denom = vsum(exps)
    return [e / denom for e in exps]


def log_softmax(z):
    """log p_i = z_i - log sum_j exp(z_j)

    Kept separate from log(softmax(z)): taking the log of a tiny probability
    loses precision, this form does not. The Phase-10 training loss uses it.
    """
    m = vmax(z)
    if m == NEG_INF:
        raise ValueError("log_softmax over an all-masked row")
    shifted = [(x - m) if x != NEG_INF else NEG_INF for x in z]
    lse = math.log(vsum([math.exp(x) if x != NEG_INF else 0.0 for x in shifted]))
    return [(x - lse) if x != NEG_INF else NEG_INF for x in shifted]


def sigmoid(x):
    """sigma(x) = 1 / (1 + exp(-x))   -- the 'sigma' of eq (2.10).

    Branch on the sign so neither exp() overflows:
        x >= 0 :  1 / (1 + exp(-x))
        x <  0 :  exp(x) / (1 + exp(x))      same value, multiplied through by exp(x)
    """
    if x >= 0.0:
        return 1.0 / (1.0 + math.exp(-x))
    e = math.exp(x)
    return e / (1.0 + e)


def gelu_exact(x):
    """GELU(x) = x * Phi(x) = x * 0.5 * (1 + erf(x/sqrt(2)))    [Hendrycks & Gimpel]"""
    return x * 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def gelu_tanh(x):
    """GELU(x) ~= 0.5x(1 + tanh( sqrt(2/pi)(x + 0.044715 x^3) ))  -- tanh approximation.

    [UNSPECIFIED] the paper writes only 'FFN_l' and never names an activation.
    Both forms are here so the choice is visible and swappable (see 0.6, A6).
    """
    c = math.sqrt(2.0 / math.pi)
    return 0.5 * x * (1.0 + math.tanh(c * (x + 0.044715 * x ** 3)))


def apply_mask(scores_row, mask_row):
    """Set disallowed logits to -inf so softmax sends them to exactly 0."""
    assert len(scores_row) == len(mask_row)
    return [scores_row[j] if mask_row[j] else NEG_INF for j in range(len(scores_row))]


# ------------------------------------------------------------------ numbers --
z = [1.0, -2.0, 0.5, 3.0, -0.25]
p = softmax(z)
print("softmax")
print(f"  z            = {z}")
print(f"  p            = {[round(v,6) for v in p]}")
print(f"  sum p        = {vsum(p):.15f}")
print(f"  softmax(z+17.3) == softmax(z)?  max|diff| = "
      f"{max(abs(a-b) for a,b in zip(softmax([v+17.3 for v in z]), p)):.3e}   (shift invariance)")

big = [1000.0, 999.0, 1001.0]
print(f"\n  logits of 1e3 -> p = {[round(v,6) for v in softmax(big)]}   "
      f"(naive exp(1000) would be inf)")

masked = apply_mask([1.0, 2.0, 3.0], [True, False, True])
print(f"  masked row {masked} -> p = {[round(v,6) for v in softmax(masked)]}   "
      f"(masked entry is EXACTLY 0.0)")

print("\nsigma  -- eq (2.10) squashes the gate preactivation into (0,1)")
for x in [-1000, -6, -2, 0, 2, 6, 1000]:
    print(f"  sigma({x:>6}) = {sigmoid(x):.9f}")

print("\nGELU")
worst = max(abs(gelu_exact(-4 + 0.05*i) - gelu_tanh(-4 + 0.05*i)) for i in range(161))
print(f"  gelu(0)                 = {gelu_exact(0.0)}")
print(f"  gelu(-0.5)              = {gelu_exact(-0.5):+.6f}   <- NOT monotone, it dips below 0")
print(f"  max|exact - tanh| on [-4,4] = {worst:.3e}")

softmax
  z            = [1.0, -2.0, 0.5, 3.0, -0.25]
  p            = [0.10716, 0.005335, 0.064996, 0.791808, 0.030702]
  sum p        = 1.000000000000000
  softmax(z+17.3) == softmax(z)?  max|diff| = 0.000e+00   (shift invariance)

  logits of 1e3 -> p = [0.244728, 0.090031, 0.665241]   (naive exp(1000) would be inf)
  masked row [1.0, -inf, 3.0] -> p = [0.119203, 0.0, 0.880797]   (masked entry is EXACTLY 0.0)

sigma  -- eq (2.10) squashes the gate preactivation into (0,1)
  sigma( -1000) = 0.000000000
  sigma(    -6) = 0.002472623
  sigma(    -2) = 0.119202922
  sigma(     0) = 0.500000000
  sigma(     2) = 0.880797078
  sigma(     6) = 0.997527377
  sigma(  1000) = 1.000000000

GELU
  gelu(0)                 = 0.0
  gelu(-0.5)              = -0.154269   <- NOT monotone, it dips below 0
  max|exact - tanh| on [-4,4] = 4.732e-04


In [4]:
# =============================================================================
# 2.3  RMSNORM  -- eq (2.9), and the seven other norm sites in the model
# =============================================================================

def rmsnorm_raw(x, eps=1e-5):
    """The normalization alone, no learned gain:

        r_i = x_i / sqrt( mean(x^2) + eps )
            = x_i / sqrt( (1/n) sum_j x_j^2 + eps )

    eps sits INSIDE the sqrt (the usual convention). The mean is over the
    SQUARES -- the vector's own mean is never subtracted.
    """
    ms = vsum([v * v for v in x]) / len(x)
    return [v / math.sqrt(ms + eps) for v in x]


def rmsnorm(x, gain=None, eps=1e-5):
    """Full RMSNorm:   out_i = gain_i * x_i / sqrt(mean(x^2) + eps)

    [UNSPECIFIED] whether the paper's RMSNorm carries a learned gain. We include
    one, initialized to all-ones, so the no-gain variant IS the initialization
    (see 0.6, B5).
    """
    r = rmsnorm_raw(x, eps)
    if gain is None:
        return r
    assert len(gain) == len(x), f"gain {len(gain)} != input {len(x)}"
    return vmul(gain, r)


# ------------------------------------------------------------------ numbers --
v = [2.0, -1.0, 0.5, 3.0, -0.25, 1.5, -2.0, 0.75]

r0 = rmsnorm_raw(v, eps=0.0)
print("with eps = 0")
print(f"  x                    = {v}")
print(f"  rmsnorm(x)           = {[round(q,6) for q in r0]}")
print(f"  rms(output)          = {rms(r0):.15f}      <- exactly 1")
print(f"  mean(output)         = {mean(r0):+.6f}      <- NOT 0: RMSNorm does not centre")
print(f"  rmsnorm(37x) == rmsnorm(x)?  max|diff| = "
      f"{max(abs(a-b) for a,b in zip(rmsnorm_raw(vscale(37.0, v), eps=0.0), r0)):.3e}"
      f"   <- scale invariant")

print("\n" + "="*72)
print("FINDING: eps breaks scale invariance, and it breaks it exactly where a")
print("         decaying recurrent state lives.")
print("="*72)
print(f"  {'scale c':>10} {'rms(c*x)':>12} {'rms(rmsnorm(c*x))':>20}   eps=1e-5")
for c in [1e2, 1e0, 1e-1, 1e-2, 1e-3, 1e-4]:
    xc = vscale(c, v)
    print(f"  {c:>10.0e} {rms(xc):>12.3e} {rms(rmsnorm(xc, eps=1e-5)):>20.6f}")

print("""
RMSNorm is usually described as scale-invariant. At eps = 0 it is. At eps = 1e-5
it is not, and the failure is severe precisely when ||x|| is small -- which is the
regime a CONTRACTING recurrent state s_t lives in.

So a state that decays toward zero does NOT get renormalized back to unit RMS by
eq (2.9). It keeps shrinking THROUGH the norm, and r_{t-1} shrinks with it, which
scales down the whole feedback term  alpha * g_t (*) W_s r_{t-1}  in eq (2.11).

This is a concrete mechanism behind the paper's own caveat, section 3.3:
    "Gates, contraction, and learned projections may suppress the practical
     contribution of long paths; structural depth alone is not a reasoning
     guarantee."

Phase 6 measures whether ||s_t|| actually enters this regime. Noted now, not
assumed either way.""")

with eps = 0
  x                    = [2.0, -1.0, 0.5, 3.0, -0.25, 1.5, -2.0, 0.75]
  rmsnorm(x)           = [1.230769, -0.615385, 0.307692, 1.846154, -0.153846, 0.923077, -1.230769, 0.461538]
  rms(output)          = 1.000000000000000      <- exactly 1
  mean(output)         = +0.346154      <- NOT 0: RMSNorm does not centre
  rmsnorm(37x) == rmsnorm(x)?  max|diff| = 0.000e+00   <- scale invariant

FINDING: eps breaks scale invariance, and it breaks it exactly where a
         decaying recurrent state lives.
     scale c     rms(c*x)    rms(rmsnorm(c*x))   eps=1e-5
       1e+02    1.625e+02             1.000000
       1e+00    1.625e+00             0.999998
       1e-01    1.625e-01             0.999811
       1e-02    1.625e-02             0.981586
       1e-03    1.625e-03             0.457056
       1e-04    1.625e-04             0.051319

RMSNorm is usually described as scale-invariant. At eps = 0 it is. At eps = 1e-5
it is not, and the failure is severe precisely when ||x|| is sm

In [5]:
# =============================================================================
# 2.4  SHAPE PLUMBING + MASKS, and the SWA window identity (claim C8)
# =============================================================================

def concat(a, b):
    """[a ; b] -- the concatenation of eq (2.10), giving length 2d."""
    return list(a) + list(b)

def slice_(a, start, stop):
    """a[start:stop] -- explicit, because Phase 8 slices caches constantly."""
    return a[start:stop]

def split_heads(x, n_heads):
    """[d] -> [H][d_head], contiguous. Head h owns dims [h*dh, (h+1)*dh).

    This is a VIEW decision, not mathematics -- the standard contiguous
    convention, chosen so merge_heads(split_heads(x,H)) == x exactly.
    """
    d = len(x)
    assert d % n_heads == 0, f"d={d} not divisible by n_heads={n_heads}"
    dh = d // n_heads
    return [x[h*dh:(h+1)*dh] for h in range(n_heads)]

def merge_heads(heads):
    """[H][d_head] -> [d], the exact inverse of split_heads."""
    out = []
    for h in heads:
        out.extend(h)
    return out

def causal_mask(T):
    """mask[i][j] = True iff query i may attend to key j.  Causal: j <= i."""
    return [[j <= i for j in range(T)] for i in range(T)]

def sliding_window_mask(T, W):
    """Causal AND within W positions INCLUDING the current one:

        mask[i][j] = True  iff  (i - W + 1) <= j <= i

    The paper (2.1) is explicit that W includes the current token, so W=1 means
    'attend to yourself only'.
    """
    assert W >= 1, f"window W must be >= 1, got {W}"
    return [[(j <= i) and (j >= i - W + 1) for j in range(T)] for i in range(T)]


# ------------------------------------------------------------------ numbers --
y = list(range(12))
y = [float(q) for q in y]
print(f"split_heads(0..11, H=3) = {split_heads(y, 3)}")
print(f"round-trips for H in 1,2,3,4,6,12: "
      f"{all(merge_heads(split_heads(y, H)) == y for H in [1,2,3,4,6,12])}")

def show_mask(m, title):
    print(f"\n  {title}")
    print("        j: " + " ".join(str(j) for j in range(len(m[0]))))
    for i, row in enumerate(m):
        print(f"      i={i}:  " + " ".join("T" if q else "." for q in row))

show_mask(causal_mask(5), "causal_mask(5)")
show_mask(sliding_window_mask(6, 3), "sliding_window_mask(6, W=3)  <- position 0 leaves at i=3")
show_mask(sliding_window_mask(5, 1), "sliding_window_mask(5, W=1)  <- diagonal only; NO history at all")

print("\n" + "="*72)
print("CLAIM C8  (paper section 2.5) -- the one that looks like an off-by-one")
print("="*72)
print("""The paper states TWO different intervals:

    read window at t   :  j in [ max(1, t-W+1), t ]     eq (2.14)
    retain after t     :  j in [ max(1, t-W+2), t ]     'After the update, retain...'

They differ by one. That is easy to misread as a bug. The claim to test is that
they are exactly consistent:

    retained(t)  UNION  {t+1}   ==   read_window(t+1)

no slack, no gap. Checking it for W in {1,2,3,8}, all t (0-indexed here):""")

c8_ok, rows = True, []
for W in [1, 2, 3, 8]:
    T = 12
    m = sliding_window_mask(T, W)
    for t in range(T - 1):
        read_t    = {j for j in range(T) if m[t][j]}
        retained  = {j for j in read_t if j >= t - W + 2}
        read_next = {j for j in range(T) if m[t+1][j]}
        if retained | {t+1} != read_next:
            c8_ok = False
            rows.append(f"    MISMATCH W={W} t={t}")
        if W == 3 and t < 6:
            rows.append(f"    W=3 t={t}:  read={sorted(read_t)!s:<12} "
                        f"retained={sorted(retained)!s:<10} -> next read={sorted(read_next)}")
print()
print("\n".join(rows))
print(f"\n  |retained| <= W-1 always:  "
      f"{all(len({j for j in range(T) if sliding_window_mask(T,W)[t][j] and j >= t-W+2}) <= W-1 for W in [1,2,3,8] for t in range(11))}")
print(f"  at W=1 the retained set is EMPTY (paper says so explicitly):  "
      f"{ {j for j in range(12) if sliding_window_mask(12,1)[5][j] and j >= 5-1+2} == set() }")
print(f"\n  ==> C8 VERIFIED: {c8_ok}")

split_heads(0..11, H=3) = [[0.0, 1.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0], [8.0, 9.0, 10.0, 11.0]]
round-trips for H in 1,2,3,4,6,12: True

  causal_mask(5)
        j: 0 1 2 3 4
      i=0:  T . . . .
      i=1:  T T . . .
      i=2:  T T T . .
      i=3:  T T T T .
      i=4:  T T T T T

  sliding_window_mask(6, W=3)  <- position 0 leaves at i=3
        j: 0 1 2 3 4 5
      i=0:  T . . . . .
      i=1:  T T . . . .
      i=2:  T T T . . .
      i=3:  . T T T . .
      i=4:  . . T T T .
      i=5:  . . . T T T

  sliding_window_mask(5, W=1)  <- diagonal only; NO history at all
        j: 0 1 2 3 4
      i=0:  T . . . .
      i=1:  . T . . .
      i=2:  . . T . .
      i=3:  . . . T .
      i=4:  . . . . T

CLAIM C8  (paper section 2.5) -- the one that looks like an off-by-one
The paper states TWO different intervals:

    read window at t   :  j in [ max(1, t-W+1), t ]     eq (2.14)
    retain after t     :  j in [ max(1, t-W+2), t ]     'After the update, retain...'

They differ by one. Th

In [6]:
# =============================================================================
# 2.5  ATTENTION  -- the complete chain, nothing collapsed into one call
# =============================================================================

def attention(q, keys, values, scale=None):
    """Single-head scaled dot-product attention over an EXPLICIT key/value list.

        score_j = <q, k_j> / sqrt(d_head)
        p_j     = softmax(score)_j
        out     = sum_j p_j * v_j

    Returns (out, scores, probs) -- ALL THREE. A function returning only `out`
    would be exactly the opaque helper this project exists to avoid; Phase 9
    inspects every intermediate, so every intermediate must escape.

    For SWA the caller passes ONLY the window, so no mask is needed:
    the window IS the mask.
    """
    assert len(keys) == len(values), f"{len(keys)} keys vs {len(values)} values"
    assert len(keys) > 0, "attention over an empty key set"
    if scale is None:
        scale = 1.0 / math.sqrt(len(q))

    scores = [dot(q, k) * scale for k in keys]
    probs  = softmax(scores)

    out = vzeros(len(values[0]))
    for j in range(len(values)):
        out = vadd(out, vscale(probs[j], values[j]))
    return out, scores, probs


def multihead_attention(q_heads, k_heads_list, v_heads_list, scale=None):
    """Run `attention` per head, concatenate.

        q_heads       [H][d_head]
        k_heads_list  [n_keys][H][d_head]        one entry per key position
        v_heads_list  [n_keys][H][d_head]

    Returns (out [d], scores [H][n_keys], probs [H][n_keys]). Stacking the
    per-query scores gives the [heads, query_pos, key_pos] tensor of Phase 9.
    """
    H, n = len(q_heads), len(k_heads_list)
    assert n == len(v_heads_list) and n > 0
    out_heads, all_scores, all_probs = [], [], []
    for h in range(H):
        keys_h = [k_heads_list[j][h] for j in range(n)]
        vals_h = [v_heads_list[j][h] for j in range(n)]
        o, sc, pr = attention(q_heads[h], keys_h, vals_h, scale)
        out_heads.append(o); all_scores.append(sc); all_probs.append(pr)
    return merge_heads(out_heads), all_scores, all_probs


# ------------------------------------------------------------------ numbers --
# A worked example, printed as the full chain rather than as one number.
q    = [1.0, 0.0, 0.5, -0.5]
keys = [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.5, 0.5, 0.5, 0.5], [2.0, 0.0, 1.0, -1.0]]
vals = [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]]

out, scores, probs = attention(q, keys, vals)
dh = len(q)

print(f"q  = {q}          d_head = {dh},  scale = 1/sqrt({dh}) = {1/math.sqrt(dh):.6f}\n")
print("  j   k_j                        <q,k_j>    score=<q,k_j>/sqrt(d)   p_j")
for j in range(len(keys)):
    print(f"  {j}   {str(keys[j]):<26} {dot(q,keys[j]):+7.3f}   {scores[j]:+12.6f}        {probs[j]:.6f}")
print(f"\n  sum_j p_j = {vsum(probs):.15f}")

print("\n  out_i = sum_j p_j * v_j[i]:")
for i in range(4):
    terms = " + ".join(f"{probs[j]:.4f}*{vals[j][i]:.0f}" for j in range(len(vals)))
    print(f"    out[{i}] = {terms} = {out[i]:.6f}")
print(f"\n  out = {[round(o,6) for o in out]}")

# Invariants that catch sign/index errors a 'sums to 1' check would miss.
hull = all(min(vl[i] for vl in vals) - 1e-12 <= out[i] <= max(vl[i] for vl in vals) + 1e-12
           for i in range(4))
print(f"\n  output inside the convex hull of the values?  {hull}")
print(f"  identical keys -> uniform?  {[round(x,6) for x in attention(q,[keys[0]]*4,vals)[2]]}")
print(f"  one dominant score -> one-hot?  "
      f"{[round(x,9) for x in attention(q,[vscale(80.0,q),keys[1],keys[2]],vals[:3])[2]]}")

print("\nmulti-head, H=2 on d=8 (heads must NOT mix):")
q2 = [1.0,0.0,0.5,-0.5, 0.0,1.0,-1.0,0.25]
k2 = [[1.0,0,0,0, 0,0,0,0],[0,1.0,0,0, 0,0,1.0,0],[0,0,1.0,0, 1.0,0,0,0]]
v2 = [[1.0,0,0,0, 0,0,0,0],[0,1.0,0,0, 0,1.0,0,0],[0,0,1.0,0, 0,0,1.0,0]]
mo, ms, mp = multihead_attention(split_heads(q2,2), [split_heads(k,2) for k in k2],
                                                    [split_heads(v,2) for v in v2])
print(f"  scores shape [H][n_keys] = ({len(ms)}, {len(ms[0])})")
for h in range(2):
    print(f"  head {h}: scores={[round(x,4) for x in ms[h]]}  probs={[round(x,4) for x in mp[h]]}"
          f"  sum={vsum(mp[h]):.12f}")
h0 = attention(split_heads(q2,2)[0], [split_heads(k,2)[0] for k in k2],
                                     [split_heads(v,2)[0] for v in v2])[0]
print(f"  head-0 output computed alone == first half of merged output?  {mo[:4] == h0}")
print(f"  scale is 1/sqrt(d_head)=1/2, not 1/sqrt(d):  "
      f"{abs(ms[0][0] - dot(split_heads(q2,2)[0], split_heads(k2[0],2)[0])/2.0) < 1e-15}")

q  = [1.0, 0.0, 0.5, -0.5]          d_head = 4,  scale = 1/sqrt(4) = 0.500000

  j   k_j                        <q,k_j>    score=<q,k_j>/sqrt(d)   p_j
  0   [1.0, 0.0, 0.0, 0.0]        +1.000      +0.500000        0.195940
  1   [0.0, 1.0, 0.0, 0.0]        +0.000      +0.000000        0.118843
  2   [0.5, 0.5, 0.5, 0.5]        +0.500      +0.250000        0.152598
  3   [2.0, 0.0, 1.0, -1.0]       +3.000      +1.500000        0.532619

  sum_j p_j = 1.000000000000000

  out_i = sum_j p_j * v_j[i]:
    out[0] = 0.1959*1 + 0.1188*0 + 0.1526*0 + 0.5326*0 = 0.195940
    out[1] = 0.1959*0 + 0.1188*1 + 0.1526*0 + 0.5326*0 = 0.118843
    out[2] = 0.1959*0 + 0.1188*0 + 0.1526*1 + 0.5326*0 = 0.152598
    out[3] = 0.1959*0 + 0.1188*0 + 0.1526*0 + 0.5326*1 = 0.532619

  out = [0.19594, 0.118843, 0.152598, 0.532619]

  output inside the convex hull of the values?  True
  identical keys -> uniform?  [0.25, 0.25, 0.25, 0.25]
  one dominant score -> one-hot?  [1.0, 0.0, 0.0]

multi-head, H=2 on d=8 (

In [7]:
# =============================================================================
# 2.6  RANDOM INITIALIZATION + INSPECTION HELPERS (seed of the Phase 5 tracker)
# =============================================================================

class LCG:
    """A linear congruential generator, so initialization has no hidden machinery.

        state <- (a*state + c) mod m      Numerical Recipes constants
        a = 1664525,  c = 1013904223,  m = 2^32

    Why not `random`? Reproducibility here should be readable in ten lines rather
    than inherited from a Mersenne Twister. Same seed -> same weights on any
    platform, any Python version.

    LCG low bits are famously poor, so `uniform` uses only the top 24 bits.
    """
    A, C, M = 1664525, 1013904223, 2 ** 32

    def __init__(self, seed=0):
        self.state = seed % self.M

    def next_u32(self):
        self.state = (self.A * self.state + self.C) % self.M
        return self.state

    def uniform(self):
        """Uniform on the OPEN interval (0,1) -- never exactly 0, which would
        make log(u) in Box-Muller blow up."""
        return ((self.next_u32() >> 8) + 0.5) / (2 ** 24)

    def normal(self):
        """Box-Muller:  z = sqrt(-2 ln u1) * cos(2*pi*u2)  is N(0,1)
        for u1,u2 iid Uniform(0,1). We discard the sin(...) partner for clarity."""
        u1, u2 = self.uniform(), self.uniform()
        return math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)


def randn_vector(n, rng, std=1.0):
    return [rng.normal() * std for _ in range(n)]

def randn_matrix(rows, cols, rng, std=1.0):
    """[UNSPECIFIED] the paper gives NO initialization scheme at all. Callers pass
    `std` explicitly (e.g. 1/sqrt(fan_in)) so the choice always lives at the call
    site and is never buried here."""
    return [[rng.normal() * std for _ in range(cols)] for _ in range(rows)]


# ------------------------------------------------ inspection (Phase 5 seed) --
def shape(x):
    """Recursive shape of nested lists, as a tuple. shape(3.0) == ()."""
    if isinstance(x, (int, float)):   return ()
    if isinstance(x, (list, tuple)):  return (0,) if len(x) == 0 else (len(x),) + shape(x[0])
    raise TypeError(f"shape() does not handle {type(x)}")

def _flatten(x):
    if isinstance(x, (int, float)): return [float(x)]
    out = []
    for item in x: out.extend(_flatten(item))
    return out

def summarize(name, x):
    """The Phase-5 record for one value: name, shape, dtype, min, max, mean, rms, l2.
    Returns a DICT, not a string -- trajectory.json wants raw numbers; formatting
    belongs in the tracer, not here."""
    flat = _flatten(x)
    finite = [v for v in flat if v != NEG_INF and v == v]
    rec = {"name": name, "shape": list(shape(x)), "dtype": "float",
           "n": len(flat), "n_masked": len(flat) - len(finite)}
    if finite:
        rec.update({"min": vmin(finite), "max": vmax(finite), "mean": mean(finite),
                    "rms": rms(finite), "l2": l2(finite)})
    return rec

def fmt(rec):
    """One-line rendering of a summarize() record."""
    if "min" not in rec:
        return f"{rec['name']:<22} shape={str(tuple(rec['shape'])):<10} (all masked)"
    return (f"{rec['name']:<22} shape={str(tuple(rec['shape'])):<10} "
            f"min={rec['min']:+9.4f} max={rec['max']:+9.4f} "
            f"mean={rec['mean']:+9.4f} rms={rec['rms']:8.4f} l2={rec['l2']:8.4f}")

def allclose(a, b, atol=1e-9, rtol=1e-7):
    """Elementwise |a-b| <= atol + rtol*|b| over arbitrarily nested lists.
    Returns (ok, max_abs_diff, index_of_worst)."""
    fa, fb = _flatten(a), _flatten(b)
    assert len(fa) == len(fb), f"allclose shape mismatch: {len(fa)} vs {len(fb)}"
    worst, worst_i = 0.0, -1
    for i in range(len(fa)):
        d = abs(fa[i] - fb[i])
        if d > worst: worst, worst_i = d, i
        if d > atol + rtol * abs(fb[i]): return False, worst, worst_i
    return True, worst, worst_i


# ------------------------------------------------------------------ numbers --
# NOTE: draw from ONE generator. [LCG(99).normal() for _ in range(5)] rebuilds the
# generator every iteration and returns the SAME number five times -- a vacuous
# test that passes. This bit the first draft of the test suite.
ga, gb, gc = LCG(99), LCG(99), LCG(100)
sa = [ga.normal() for _ in range(5)]
sb = [gb.normal() for _ in range(5)]
sc = [gc.normal() for _ in range(5)]
print(f"LCG(99)  stream = {[round(x,6) for x in sa]}")
print(f"LCG(99)  again  = {[round(x,6) for x in sb]}    identical? {sa == sb}")
print(f"LCG(100) stream = {[round(x,6) for x in sc]}    differs?   {sa != sc}")
print(f"stream does not repeat itself: {len(set(sa)) == 5}")

g = LCG(2024); N = 20000
samples = [g.normal() for _ in range(N)]
print(f"\nBox-Muller over N={N}:  mean={mean(samples):+.4f} (want 0)   "
      f"std={math.sqrt(var(samples)):.4f} (want 1)")

g5 = LCG(5); us = [g5.uniform() for _ in range(1000)]
bins = [0]*10
for t in us: bins[min(9, int(t*10))] += 1
print(f"uniform: min={min(us):.6f} max={max(us):.6f} distinct={len(set(us))}/1000")
print(f"         10 equal bins of 1000 draws = {bins}   (want ~100 each)")

print("\nsummarize() -- the Phase 5 record:")
rng = LCG(7)
print("  " + fmt(summarize("s_t",              randn_vector(8, rng))))
print("  " + fmt(summarize("attn_scores[H,n]", [randn_vector(5, rng) for _ in range(2)])))
print("  " + fmt(summarize("W_g",              randn_matrix(8, 16, rng, std=1/math.sqrt(16)))))
print(f"\n  masked entries counted: {summarize('m',[1.0,NEG_INF,2.0])['n_masked']}")
print(f"  allclose([1,2],[1,2.5]) -> {allclose([1.0,2.0],[1.0,2.5])}   (ok, worst, index)")

LCG(99)  stream = [-0.083218, -0.496106, -0.869789, -1.523911, -0.063356]
LCG(99)  again  = [-0.083218, -0.496106, -0.869789, -1.523911, -0.063356]    identical? True
LCG(100) stream = [-0.936065, -1.072978, -2.064, -0.042479, -0.644961]    differs?   True
stream does not repeat itself: True

Box-Muller over N=20000:  mean=+0.0052 (want 0)   std=0.9977 (want 1)
uniform: min=0.002502 max=0.997903 distinct=1000/1000
         10 equal bins of 1000 draws = [104, 95, 94, 107, 105, 109, 96, 98, 89, 103]   (want ~100 each)

summarize() -- the Phase 5 record:
  s_t                    shape=(8,)       min=  -1.2040 max=  +2.3020 mean=  +0.7797 rms=  1.3146 l2=  3.7184
  attn_scores[H,n]       shape=(2, 5)     min=  -1.9687 max=  +1.8432 mean=  -0.1724 rms=  1.1802 l2=  3.7321
  W_g                    shape=(8, 16)    min=  -0.6983 max=  +0.7792 mean=  +0.0029 rms=  0.2595 l2=  2.9363

  masked entries counted: 1
  allclose([1,2],[1,2.5]) -> (False, 0.5, 1)   (ok, worst, index)


In [8]:
# =============================================================================
# 2.7  THE INVARIANT SUITE -- every primitive above, checked
# =============================================================================
# No pytest, no unittest: a 10-line harness, zero dependencies.

_PASS, _FAIL = [], []

def check(name, cond, detail=""):
    (_PASS if cond else _FAIL).append(name)
    print(("  ok   " if cond else "  FAIL ") + name + (f"   {detail}" if detail else ""))

def close(a, b, atol=1e-9, rtol=1e-7):
    ok, worst, _ = allclose(a, b, atol, rtol)
    return ok, f"max|diff|={worst:.3e}"

def section(t):
    print(f"\n--- {t} " + "-" * max(0, 60 - len(t)))

print("=" * 70); print("core primitives -- invariant suite"); print("=" * 70)

section("purity")
check("0  no banned framework bound in our namespace", "PURITY" not in assert_pure(globals()))

rng = LCG(1234)

section("vector / matrix algebra")
a, b = randn_vector(6, rng), randn_vector(6, rng)
check("1  (a+b)-b == a",            *close(vsub(vadd(a, b), b), a))
check("2  a (*) 1 == a",            *close(vmul(a, [1.0]*6), a))
check("3  <a,b> == <b,a>",          *close([dot(a,b)], [dot(b,a)]))
check("4  <a,a> == ||a||^2",        *close([dot(a,a)], [l2(a)**2]))
check("5  ||a|| == sqrt(n)*rms(a)", *close([l2(a)], [math.sqrt(6)*rms(a)]))
A, B, Cm = randn_matrix(4,5,rng), randn_matrix(5,3,rng), randn_matrix(3,2,rng)
xv = randn_vector(5, rng)
check("6  matvec(M,x)[i] == <M[i],x>", *close(matvec(A,xv), [dot(A[i],xv) for i in range(4)]))
check("7a transpose(transpose(A)) == A", *close(transpose(transpose(A)), A))
check("7b (AB)^T == B^T A^T", *close(transpose(matmul(A,B)), matmul(transpose(B),transpose(A)), 1e-12, 1e-9))
check("7c A(BC) == (AB)C  [associativity]", *close(matmul(A,matmul(B,Cm)), matmul(matmul(A,B),Cm), 1e-12, 1e-9))
xv3 = randn_vector(3, LCG(7))
check("7d (AB)x == A(Bx)", *close(matvec(matmul(A,B),xv3), matvec(A,matvec(B,xv3)), 1e-12, 1e-9))

section("softmax and friends")
z = [1.0, -2.0, 0.5, 3.0, -0.25]; p = softmax(z)
check("8  softmax sums to 1",           *close([vsum(p)], [1.0]))
check("9  softmax strictly positive",   all(v > 0 for v in p))
check("10 shift invariance",            *close(softmax([v+17.3 for v in z]), p, 1e-12, 1e-9))
pb = softmax([1000.0, 999.0, 1001.0])
check("11 survives logits of 1e3",      abs(vsum(pb)-1.0) < 1e-12 and all(v==v for v in pb))
pm = softmax(apply_mask([1.0,2.0,3.0], [True,False,True]))
check("12 masked -> exactly 0.0",       pm[1] == 0.0 and abs(vsum(pm)-1.0) < 1e-12)
ls = log_softmax(z)
check("13a exp(log_softmax) == softmax", *close([math.exp(v) for v in ls], p))
check("13b log_softmax == log(softmax)", *close(ls, [math.log(v) for v in p]))
try:
    softmax([NEG_INF]*3); ok_allmask = False
except ValueError:
    ok_allmask = True
check("13c all-masked row raises, not NaN", ok_allmask)

section("nonlinearities")
check("14a sigma(0) == 0.5",                *close([sigmoid(0.0)], [0.5]))
check("14b sigma(-x) == 1 - sigma(x)",      *close([sigmoid(-2.7)], [1.0-sigmoid(2.7)]))
check("14c no overflow at +/-1000",         sigmoid(1000.0)==1.0 and sigmoid(-1000.0) < 1e-300)
check("14d sigma in [0,1] on a sweep",      all(0.0 <= sigmoid(v) <= 1.0 for v in [-50,-5,-1,0,1,5,50]))
check("15a gelu(0) == 0",                   gelu_exact(0.0)==0.0 and gelu_tanh(0.0)==0.0)
wg = max(abs(gelu_exact(-4+0.05*i)-gelu_tanh(-4+0.05*i)) for i in range(161))
check("15b tanh approx within 1e-2",        wg < 1e-2, f"max|diff|={wg:.3e}")
check("15c gelu dips below 0 for x<0",      gelu_exact(-0.5) < 0.0)

section("RMSNorm -- eq (2.9)")
vv = randn_vector(8, rng); r0 = rmsnorm_raw(vv, eps=0.0)
check("17a rms(output) == 1 at eps=0",      *close([rms(r0)], [1.0]))
check("17b does NOT centre (mean != 0)",    abs(mean(r0)) > 1e-6, f"mean={mean(r0):+.6f}")
check("17c scale invariant at eps=0",       *close(rmsnorm_raw(vscale(37.0,vv),eps=0.0), r0, 1e-12, 1e-9))
check("17d gain of ones is identity",       *close(rmsnorm(vv, gain=[1.0]*8, eps=0.0), r0))
rbig, rsml = rms(rmsnorm(vscale(1e2,vv),eps=1e-5)), rms(rmsnorm(vscale(1e-3,vv),eps=1e-5))
check("17e eps breaks invariance near 0",   rbig > 0.999 and rsml < 0.9,
      f"rms@1e2={rbig:.6f}  rms@1e-3={rsml:.6f}  <- the finding in 2.3")

section("shape plumbing")
yv = randn_vector(12, rng)
check("18a merge(split(x,H)) == x, H=1..12", all(merge_heads(split_heads(yv,H))==yv for H in [1,2,3,4,6,12]))
check("18b split gives H blocks of d/H",     shape(split_heads(yv,3)) == (3,4))
check("19a concat lengths add",              len(concat(a,b)) == len(a)+len(b))
check("19b concat puts e first",             concat([1.0,2.0],[9.0,9.0])[:2] == [1.0,2.0])

section("masks + claim C8")
check("20a causal row i allows i+1",  [sum(r) for r in causal_mask(4)] == [1,2,3,4])
cmk = causal_mask(4)
check("20b causal is lower-triangular", all(not cmk[i][j] for i in range(4) for j in range(4) if j>i))
for W in [1,2,3,8]:
    check(f"21 W={W}: row i allows min(W,i+1)",
          [sum(r) for r in sliding_window_mask(6,W)] == [min(W,i+1) for i in range(6)])
check("21b W=1 is the diagonal only",
      all(sliding_window_mask(5,1)[i][j] == (i==j) for i in range(5) for j in range(5)))
c8 = True
for W in [1,2,3,8]:
    m = sliding_window_mask(12, W)
    for t in range(11):
        rd  = {j for j in range(12) if m[t][j]}
        ret = {j for j in rd if j >= t-W+2}
        if ret | {t+1} != {j for j in range(12) if m[t+1][j]}: c8 = False
check("22 C8: retained(t) + current == read(t+1)", c8, "all W in {1,2,3,8}, all t")

section("attention")
q = randn_vector(4, rng)
ks = [randn_vector(4, rng) for _ in range(5)]
vs = [randn_vector(4, rng) for _ in range(5)]
o, sc, pr = attention(q, ks, vs)
check("23a probs sum to 1",                 *close([vsum(pr)], [1.0]))
check("23b scores == <q,k>/sqrt(d_head)",   *close(sc, [dot(q,k)/math.sqrt(4) for k in ks]))
check("23c out == sum_j p_j v_j by hand",   *close(o, [sum(pr[j]*vs[j][i] for j in range(5)) for i in range(4)]))
check("24 out inside convex hull of values",
      all(min(x[i] for x in vs)-1e-12 <= o[i] <= max(x[i] for x in vs)+1e-12 for i in range(4)))
check("25 identical keys -> uniform",       *close(attention(q,[ks[0]]*4,vs[:4])[2], [0.25]*4))
o1,_,p1 = attention(q,[ks[0]],[vs[0]])
check("26a single key -> p=1, out==value",  p1==[1.0] and allclose(o1, vs[0])[0])
os_,_,ps_ = attention(q,[vscale(80.0,q),ks[1],ks[2]],vs[:3])
check("26b dominant score -> one-hot",      ps_[0] > 1-1e-9 and allclose(os_, vs[0], 1e-8)[0])
mo1,ms1,_ = multihead_attention(split_heads(q,1), [split_heads(k,1) for k in ks], [split_heads(x,1) for x in vs])
check("27a H=1 reduces to single-head",     *close(mo1, o))
check("27b scores shape [H][n_keys]",       shape(ms1) == (1,5))
q2 = randn_vector(8, rng); k2=[randn_vector(8,rng) for _ in range(3)]; v2=[randn_vector(8,rng) for _ in range(3)]
mo2,ms2,mp2 = multihead_attention(split_heads(q2,2),[split_heads(k,2) for k in k2],[split_heads(x,2) for x in v2])
check("27c each head's probs sum to 1",     all(abs(vsum(r)-1.0) < 1e-12 for r in mp2))
check("27d heads do not mix",               *close(mo2[:4], attention(split_heads(q2,2)[0],
          [split_heads(k,2)[0] for k in k2], [split_heads(x,2)[0] for x in v2])[0]))
check("27e scale is 1/sqrt(d_head) not 1/sqrt(d)",
      *close([ms2[0][0]], [dot(split_heads(q2,2)[0], split_heads(k2[0],2)[0])/2.0]))

section("random generator")
ga2, gb2, gc2 = LCG(99), LCG(99), LCG(100)
s1 = [ga2.normal() for _ in range(5)]; s2 = [gb2.normal() for _ in range(5)]; s3 = [gc2.normal() for _ in range(5)]
check("28a same seed -> same stream",  s1 == s2)
check("28b different seeds differ",    s1 != s3)
check("28c stream does not repeat",    len(set(s1)) == 5)
gN = LCG(2024); smp = [gN.normal() for _ in range(20000)]
mu_, sd_ = mean(smp), math.sqrt(var(smp))
check("29 Box-Muller mean~0 std~1",    abs(mu_) < 0.05 and abs(sd_-1.0) < 0.05, f"mean={mu_:+.4f} std={sd_:.4f}")
g5b = LCG(5); uu = [g5b.uniform() for _ in range(1000)]
bn = [0]*10
for t in uu: bn[min(9,int(t*10))] += 1
check("30a uniform strictly inside (0,1)", all(0.0 < t < 1.0 for t in uu))
check("30b uniform actually varies",       len(set(uu)) > 990, f"{len(set(uu))}/1000 distinct")
check("30c 10 bins within 100+/-40",       all(60 <= c <= 140 for c in bn), f"{bn}")

section("inspection helpers")
check("31a shape(scalar) == ()",       shape(3.0) == ())
check("31b shape nests [H][n]",        shape([[1.0,2.0],[3.0,4.0],[5.0,6.0]]) == (3,2))
rec = summarize("s_t", [3.0,-1.0,0.0,4.0])
check("31c summarize stats correct",   rec["shape"]==[4] and rec["min"]==-1.0 and rec["max"]==4.0
      and abs(rec["mean"]-1.5)<1e-12 and abs(rec["l2"]-math.sqrt(26))<1e-12)
check("31d counts masked entries",     summarize("m",[1.0,NEG_INF,2.0])["n_masked"] == 1)
okd, wd, idx = allclose([1.0,2.0],[1.0,2.5])
check("31e allclose reports worst index", (not okd) and idx == 1)

print("\n" + "=" * 70)
print(f"PASSED {len(_PASS)}   FAILED {len(_FAIL)}")
for n in _FAIL: print("   FAILED:", n)
print("all primitives hold." if not _FAIL else "SUITE RED")
print("=" * 70)

core primitives -- invariant suite

--- purity ------------------------------------------------------
  ok   0  no banned framework bound in our namespace

--- vector / matrix algebra -------------------------------------
  ok   1  (a+b)-b == a   max|diff|=5.551e-17
  ok   2  a (*) 1 == a   max|diff|=0.000e+00
  ok   3  <a,b> == <b,a>   max|diff|=0.000e+00
  ok   4  <a,a> == ||a||^2   max|diff|=4.441e-16
  ok   5  ||a|| == sqrt(n)*rms(a)   max|diff|=2.220e-16
  ok   6  matvec(M,x)[i] == <M[i],x>   max|diff|=0.000e+00
  ok   7a transpose(transpose(A)) == A   max|diff|=0.000e+00
  ok   7b (AB)^T == B^T A^T   max|diff|=0.000e+00
  ok   7c A(BC) == (AB)C  [associativity]   max|diff|=8.882e-16
  ok   7d (AB)x == A(Bx)   max|diff|=1.776e-15

--- softmax and friends -----------------------------------------
  ok   8  softmax sums to 1   max|diff|=2.220e-16
  ok   9  softmax strictly positive
  ok   10 shift invariance   max|diff|=0.000e+00
  ok   11 survives logits of 1e3
  ok   12 masked -> 

# §3 — Phase 3: the tiny RLT

Small enough that **every number fits on screen**.

```
V = 10   d = 8   L_E = 1   L_D = 1   H = 2   d_head = 4
d_ff = 16   W = 3   G = 1   alpha = 0.1   eps = 1e-5
```

All [ASSUMPTION] (§0.6). Written top-to-bottom so you can follow
`token → embedding → encoder → memory → merge → SWA → cross-attn → FFN → state → logits`
without jumping through abstractions.

Two schedules for the encoder are implemented — **parallel prefill** and **incremental step** — because Proposition 3.1 is exactly the claim that they agree. `opts` carries the ablation hooks (α override, state zero/noise/freeze/inject, gate off) that Phase 7 needs.

In [11]:
# =============================================================================
# 3.1  CONFIG + PARAMETERS
# =============================================================================

def make_cfg(**over):
    cfg = dict(V=10, d=8, L_E=1, L_D=1, H=2, d_ff=16, W=3, G=1,
               alpha=0.1, eps=1e-5, seed=0, pos="nope", tied=False)
    cfg.update(over)
    assert cfg["d"] % cfg["H"] == 0
    assert 1 <= cfg["G"] <= cfg["L_D"]
    assert cfg["W"] >= 1
    cfg["d_head"] = cfg["d"] // cfg["H"]
    return cfg


def init_params(cfg):
    """Every learnable tensor, in one flat dict name -> value.

    [UNSPECIFIED] the paper gives NO init scheme (0.6, group C). We use
    std = 1/sqrt(fan_in) for matrices and std = 1 for embeddings / s_star,
    and state it at the call site rather than burying it.
    """
    g = LCG(cfg["seed"])
    d, dh, H, dff, V = cfg["d"], cfg["d_head"], cfg["H"], cfg["d_ff"], cfg["V"]
    P = {}
    def M(name, r, c):  P[name] = randn_matrix(r, c, g, std=1.0/math.sqrt(c))
    def ones(name, n):  P[name] = [1.0]*n              # RMSNorm gains (0.6, B5)
    def vec(name, n, s=1.0): P[name] = randn_vector(n, g, std=s)

    vec("E_tok_flat", V*d)                              # embedding table, flattened
    P["E_tok"] = [P["E_tok_flat"][i*d:(i+1)*d] for i in range(V)]
    del P["E_tok_flat"]

    for l in range(cfg["L_E"]):                         # encoder blocks  (1.1)
        ones(f"enc.{l}.n_attn", d)
        for w in "QKVO": M(f"enc.{l}.W{w}", d, d)
        ones(f"enc.{l}.n_ffn", d)
        M(f"enc.{l}.W1", dff, d); M(f"enc.{l}.W2", d, dff)

    for gp in range(cfg["G"]):                          # encoder memory  (2.2)
        ones(f"mem.{gp}.n_E", d)
        M(f"mem.{gp}.WK", d, d); M(f"mem.{gp}.WV", d, d)

    vec("s_star", d)                                    # learned initial state (2.3)
    ones("merge.n_s", d)                                # RMSNorm_s           (2.9)
    M("merge.Wg", d, 2*d); P["merge.bg"] = [0.0]*d      # gate                (2.10)
    M("merge.Ws", d, d)                                 # state projection    (2.11)

    for l in range(cfg["L_D"]):                         # decoder blocks (2.12-2.16)
        ones(f"dec.{l}.n_q", d); M(f"dec.{l}.WQ", d, d)      # P_Q^{D,l}
        ones(f"dec.{l}.n_k", d); M(f"dec.{l}.WK", d, d)      # P_K^{D,l}
        ones(f"dec.{l}.n_S", d); M(f"dec.{l}.WV", d, d)      # RMSNorm_{S,l}
        M(f"dec.{l}.WO", d, d)                               # Attn^D out proj
        ones(f"dec.{l}.n_m", d); M(f"dec.{l}.WQM", d, d)     # P_Q^{M,l}
        M(f"dec.{l}.WOM", d, d)                              # Attn^M out proj
        ones(f"dec.{l}.n_ffn", d)
        M(f"dec.{l}.W1", dff, d); M(f"dec.{l}.W2", d, dff)

    ones("out.n", d); M("out.WO", V, d)                 # readout             (2.6)
    return P


def n_params(P):
    tot = 0
    for k, v in P.items():
        if k == "E_tok":                 tot += sum(len(r) for r in v)
        elif isinstance(v[0], list):     tot += sum(len(r) for r in v)
        else:                            tot += len(v)
    return tot


def clone_params(P):
    out = {}
    for k, v in P.items():
        out[k] = [list(r) for r in v] if isinstance(v[0], list) else list(v)
    return out


# ------------------------------------------------------- positional transform --
def rope(vec_h, pos, base=10000.0):
    """RoPE on one head: rotate coordinate pairs (2i, 2i+1) by pos * theta_i,
    theta_i = base^(-2i/d_head). A rotation, so it is orthogonal and norm-preserving.
    [UNSPECIFIED] the paper says only 'any positional transformation' (0.6, B1)."""
    dh, out = len(vec_h), list(vec_h)
    for i in range(dh // 2):
        th = pos * (base ** (-2.0 * i / dh))
        c, s = math.cos(th), math.sin(th)
        a, b = vec_h[2*i], vec_h[2*i+1]
        out[2*i], out[2*i+1] = a*c - b*s, a*s + b*c
    return out

def apply_pos(heads, pos, cfg):
    if cfg["pos"] == "nope":  return heads                  # identity: NoPE
    if cfg["pos"] == "rope":  return [rope(h, pos) for h in heads]
    raise ValueError(cfg["pos"])


# ------------------------------------------------------------------ numbers --
cfg = make_cfg()
P = init_params(cfg)
print("config:", {k: v for k, v in cfg.items() if k != "seed"})
print(f"\nparameters: {n_params(P)} total across {len(P)} named tensors\n")
for k in sorted(P):
    v = P[k]
    sh = (len(v), len(v[0])) if isinstance(v[0], list) else (len(v),)
    print(f"  {k:<18} {str(sh):<10} rms={rms(_flatten(v)):.4f}")

print(f"\ns_star (the learned initial state, eq 2.3) = {[round(x,4) for x in P['s_star']]}")
print(f"RoPE is norm-preserving (orthogonal):  |rope(v,5)| - |v| = "
      f"{abs(l2(rope([1.0,2.0,-0.5,0.25], 5)) - l2([1.0,2.0,-0.5,0.25])):.3e}")

config: {'V': 10, 'd': 8, 'L_E': 1, 'L_D': 1, 'H': 2, 'd_ff': 16, 'W': 3, 'G': 1, 'alpha': 0.1, 'eps': 1e-05, 'pos': 'nope', 'tied': False, 'd_head': 4}

parameters: 1728 total across 32 named tensors

  E_tok              (10, 8)    rms=1.0234
  dec.0.W1           (16, 8)    rms=0.3737
  dec.0.W2           (8, 16)    rms=0.2531
  dec.0.WK           (8, 8)     rms=0.3691
  dec.0.WO           (8, 8)     rms=0.3073
  dec.0.WOM          (8, 8)     rms=0.3077
  dec.0.WQ           (8, 8)     rms=0.3699
  dec.0.WQM          (8, 8)     rms=0.3625
  dec.0.WV           (8, 8)     rms=0.3602
  dec.0.n_S          (8,)       rms=1.0000
  dec.0.n_ffn        (8,)       rms=1.0000
  dec.0.n_k          (8,)       rms=1.0000
  dec.0.n_m          (8,)       rms=1.0000
  dec.0.n_q          (8,)       rms=1.0000
  enc.0.W1           (16, 8)    rms=0.3335
  enc.0.W2           (8, 16)    rms=0.2671
  enc.0.WK           (8, 8)     rms=0.3297
  enc.0.WO           (8, 8)     rms=0.3576
  enc.0.WQ           (8,

In [15]:
# =============================================================================
# 3.2  THE MODEL -- readable top to bottom, no hidden abstractions
# =============================================================================

class Trace:
    """Records every named intermediate. tr=None means 'no tracing', zero cost."""
    def __init__(self, on=True): self.on, self.items, self.idx = on, [], {}
    def rec(self, name, val):
        if self.on:
            self.items.append((name, val)); self.idx[name] = val
        return val
    def __getitem__(self, k): return self.idx[k]
    def has(self, k):         return k in self.idx

BLOCKS = {"enc": 0, "dec": 0}          # block-evaluation counters for claim C1
def reset_blocks(): BLOCKS["enc"] = BLOCKS["dec"] = 0


def ffn(P, cfg, x, pre, w1, w2, tag=None, tr=None):
    """FFN(RMSNorm(x)) = W2 . gelu(W1 . norm(x))    -- eq (2.16) inner part."""
    n = rmsnorm(x, P[pre], cfg["eps"])
    hid = [gelu_exact(z) for z in matvec(P[w1], n)]
    out = matvec(P[w2], hid)
    if tr and tag:
        tr.rec(f"{tag}.ffn_norm", n); tr.rec(f"{tag}.ffn_hidden", hid); tr.rec(f"{tag}.ffn_out", out)
    return out


# ------------------------------------------------------------ 1. ENCODER -----
def encoder_prefill(P, cfg, tokens, tr=None):
    """Parallel schedule: all positions at once under a causal mask.  eq (2.1)"""
    T = len(tokens)
    h = [list(P["E_tok"][x]) for x in tokens]
    if tr:
        for t in range(T): tr.rec(f"t{t+1}.emb", h[t])
    for l in range(cfg["L_E"]):
        BLOCKS["enc"] += T
        nq = [rmsnorm(h[i], P[f"enc.{l}.n_attn"], cfg["eps"]) for i in range(T)]
        Q = [apply_pos(split_heads(matvec(P[f"enc.{l}.WQ"], nq[i]), cfg["H"]), i+1, cfg) for i in range(T)]
        K = [apply_pos(split_heads(matvec(P[f"enc.{l}.WK"], nq[i]), cfg["H"]), i+1, cfg) for i in range(T)]
        Vh= [split_heads(matvec(P[f"enc.{l}.WV"], nq[i]), cfg["H"]) for i in range(T)]   # no pos on values
        new = []
        for i in range(T):
            o, sc, pr = multihead_attention(Q[i], K[:i+1], Vh[:i+1])       # causal: j <= i
            if tr:
                tr.rec(f"t{i+1}.enc{l}.Q", Q[i]); tr.rec(f"t{i+1}.enc{l}.K", K[i]); tr.rec(f"t{i+1}.enc{l}.V", Vh[i])
                tr.rec(f"t{i+1}.enc{l}.scores", sc); tr.rec(f"t{i+1}.enc{l}.probs", pr)
            b = vadd(h[i], matvec(P[f"enc.{l}.WO"], o))
            new.append(vadd(b, ffn(P, cfg, b, f"enc.{l}.n_ffn", f"enc.{l}.W1", f"enc.{l}.W2",
                                   f"t{i+1}.enc{l}", tr)))
        h = new
    if tr:
        for t in range(T): tr.rec(f"t{t+1}.e", h[t])
    return h


def encoder_step(P, cfg, x_t, cacheE, pos, tr=None):
    """Incremental schedule: one token, using the causal encoder cache.  eq (2.8)

    Must produce EXACTLY the same e_t as encoder_prefill -- that is the encoder
    half of Proposition 3.1.
    """
    h = list(P["E_tok"][x_t])
    for l in range(cfg["L_E"]):
        BLOCKS["enc"] += 1
        n  = rmsnorm(h, P[f"enc.{l}.n_attn"], cfg["eps"])
        q  = apply_pos(split_heads(matvec(P[f"enc.{l}.WQ"], n), cfg["H"]), pos, cfg)
        k  = apply_pos(split_heads(matvec(P[f"enc.{l}.WK"], n), cfg["H"]), pos, cfg)
        v  = split_heads(matvec(P[f"enc.{l}.WV"], n), cfg["H"])
        cacheE[l].append((pos, k, v))
        o, sc, pr = multihead_attention(q, [e[1] for e in cacheE[l]], [e[2] for e in cacheE[l]])
        b = vadd(h, matvec(P[f"enc.{l}.WO"], o))
        h = vadd(b, ffn(P, cfg, b, f"enc.{l}.n_ffn", f"enc.{l}.W1", f"enc.{l}.W2"))
    return h


# ------------------------------------------- 2. ENCODER MEMORY -- eq (2.2) ---
def memory_append(P, cfg, e_t, pos, M, tr=None):
    """k = P_K(e_t,t) ; v = W_V . RMSNorm_E(e_t) ; append to M^g.
    Depends on e_t ONLY -- never on a decoder state. That is claim C11."""
    for g in range(cfg["G"]):
        n = rmsnorm(e_t, P[f"mem.{g}.n_E"], cfg["eps"])
        k = apply_pos(split_heads(matvec(P[f"mem.{g}.WK"], n), cfg["H"]), pos, cfg)
        v = split_heads(matvec(P[f"mem.{g}.WV"], n), cfg["H"])      # no positional op
        M[g].append((pos, k, v))
        if tr:
            tr.rec(f"t{pos}.mem{g}.K", k); tr.rec(f"t{pos}.mem{g}.V", v)
    return M


# ------------------------------------------- 3. GATED MERGE -- eqs (2.9-2.11) --
def gated_merge(P, cfg, e_t, s_prev, opts, tr=None, tag=""):
    a = opts.get("alpha", cfg["alpha"])
    r = rmsnorm(s_prev, P["merge.n_s"], cfg["eps"])                        # (2.9)
    pre = vadd(matvec(P["merge.Wg"], concat(e_t, r)), P["merge.bg"])       # (2.10)
    g = [sigmoid(z) for z in pre]
    if opts.get("gate") == "off":
        g = [1.0]*cfg["d"]                    # experiment F: remove the gate
    Wsr = matvec(P["merge.Ws"], r)
    fb = vscale(a, vmul(g, Wsr))                                            # (2.11)
    u = vadd(e_t, fb)
    if tr:
        tr.rec(f"{tag}.s_prev", s_prev);   tr.rec(f"{tag}.r_prev", r)
        tr.rec(f"{tag}.concat_e_r", concat(e_t, r))
        tr.rec(f"{tag}.gate_pre", pre);    tr.rec(f"{tag}.g", g)
        tr.rec(f"{tag}.Ws_r", Wsr)
        tr.rec(f"{tag}.Ws_s", matvec(P["merge.Ws"], s_prev))   # the NON-paper variant, for contrast
        tr.rec(f"{tag}.feedback", fb);     tr.rec(f"{tag}.u", u)
    return u, g, r, fb


# --------------------------------- 4. DECODER BLOCK -- eqs (2.12-2.16) -------
def decoder_block(P, cfg, l, z, t, M, cache_l, tr=None, tag=""):
    """One decoder layer: causal SWA -> encoder-memory cross-attn -> FFN.
    Returns (z_out, new_cache_l, evicted)."""
    BLOCKS["dec"] += 1
    W, H = cfg["W"], cfg["H"]

    # --- (2.12)(2.13) current KV formed from the LAYER INPUT, before SWA -----
    q = apply_pos(split_heads(matvec(P[f"dec.{l}.WQ"], rmsnorm(z, P[f"dec.{l}.n_q"], cfg["eps"])), H), t, cfg)
    k = apply_pos(split_heads(matvec(P[f"dec.{l}.WK"], rmsnorm(z, P[f"dec.{l}.n_k"], cfg["eps"])), H), t, cfg)
    v = split_heads(matvec(P[f"dec.{l}.WV"], rmsnorm(z, P[f"dec.{l}.n_S"], cfg["eps"])), H)

    # --- (2.14) causal SWA. The read window IS the mask ----------------------
    window = list(cache_l) + [(t, k, v)]                 # j in [max(1,t-W+1), t]
    o, sc, pr = multihead_attention(q, [e[1] for e in window], [e[2] for e in window])
    b = vadd(z, matvec(P[f"dec.{l}.WO"], o))

    # --- (2.15) cross-attention to PREFIX-RESTRICTED encoder memory ----------
    gsel = l % cfg["G"]
    mem = [e for e in M[gsel] if e[0] <= t]              # slice, never read the future
    qm = apply_pos(split_heads(matvec(P[f"dec.{l}.WQM"], rmsnorm(b, P[f"dec.{l}.n_m"], cfg["eps"])), H), t, cfg)
    om, csc, cpr = multihead_attention(qm, [e[1] for e in mem], [e[2] for e in mem])
    a = vadd(b, matvec(P[f"dec.{l}.WOM"], om))

    # --- (2.16) FFN ----------------------------------------------------------
    z_out = vadd(a, ffn(P, cfg, a, f"dec.{l}.n_ffn", f"dec.{l}.W1", f"dec.{l}.W2", f"{tag}.dec{l}", tr))

    # --- retention: keep positions [max(1, t-W+2), t] ------------------------
    keep = window[-(W-1):] if W > 1 else []
    evicted = [e[0] for e in window if e not in keep]

    if tr:
        p = f"{tag}.dec{l}"
        tr.rec(f"{p}.z_in", z); tr.rec(f"{p}.Q", q); tr.rec(f"{p}.K", k); tr.rec(f"{p}.V", v)
        tr.rec(f"{p}.cache_before", [e[0] for e in cache_l])
        tr.rec(f"{p}.window_pos", [e[0] for e in window])
        tr.rec(f"{p}.swa_scores", sc); tr.rec(f"{p}.swa_probs", pr); tr.rec(f"{p}.swa_out", o)
        tr.rec(f"{p}.b", b)
        tr.rec(f"{p}.mem_pos", [e[0] for e in mem])
        tr.rec(f"{p}.cross_scores", csc); tr.rec(f"{p}.cross_probs", cpr); tr.rec(f"{p}.cross_out", om)
        tr.rec(f"{p}.a", a); tr.rec(f"{p}.z_out", z_out)
        tr.rec(f"{p}.cache_after", [e[0] for e in keep]); tr.rec(f"{p}.evicted", evicted)
    return z_out, keep, evicted


# ------------------------------------------------ 5. READOUT -- eq (2.6) -----
def readout(P, cfg, s, tr=None, tag=""):
    n = rmsnorm(s, P["out.n"], cfg["eps"])
    logits = matvec(P["out.WO"], n)
    probs = softmax(logits)
    if tr:
        tr.rec(f"{tag}.out_norm", n); tr.rec(f"{tag}.logits", logits); tr.rec(f"{tag}.probs", probs)
    return logits, probs


# --------------------------------------- 6. THE WHOLE MODEL -- eq (2.3-2.7) --
def rlt_forward(P, cfg, tokens, opts=None, tr=None, enc_mode="prefill"):
    """H_t = F_t o ... o F_1 (H_0).  Returns every intermediate as lists.

    opts hooks (all default off) -- these are the Phase 7 instruments:
       alpha       : override cfg alpha                          (experiments G,H)
       gate        : "off" -> g_t := 1                           (experiment F)
       state_mode  : normal | zero | noise | freeze | inject     (experiments B,C,D,E)
       u_delta     : (k, delta) adds delta to u_k, i.e. a perturbation injected
                     INSIDE the decoder at step k. This is the probe that
                     separates the two recurrent channels, because it bypasses
                     the encoder and the memory entirely.
    """
    opts = opts or {}
    T, d = len(tokens), cfg["d"]

    if enc_mode == "prefill":
        E = encoder_prefill(P, cfg, tokens, tr)
    else:
        cacheE = [[] for _ in range(cfg["L_E"])]
        E = [encoder_step(P, cfg, tokens[i], cacheE, i+1, tr) for i in range(T)]

    M = [[] for _ in range(cfg["G"])]
    C_D = [[] for _ in range(cfg["L_D"])]            # the SWA caches -- STORE 2
    s = list(P["s_star"])                            # H_0 = (s_star, {})     (2.3)

    noise_rng = LCG(opts.get("noise_seed", 777))
    out = dict(e=E, s=[], u=[], g=[], r=[], fb=[], logits=[], probs=[],
               caches=[], evicted=[], s_used=[])

    for i in range(T):
        t = i + 1
        memory_append(P, cfg, E[i], t, M, tr)        # prefix-restricted growth

        # ---- the ablation hooks (Phase 7) --------------------------------
        mode = opts.get("state_mode", "normal")
        if   mode == "zero":   s_use = [0.0]*d
        elif mode == "noise":  s_use = randn_vector(d, noise_rng, opts.get("noise_std", 1.0))
        elif mode == "freeze": s_use = s if t <= opts.get("freeze_at", 1) + 1 else out["s"][opts["freeze_at"]-1]
        elif mode == "inject": s_use = list(opts["inject"][i])
        else:                  s_use = s
        out["s_used"].append(list(s_use))

        u, g, r, fb = gated_merge(P, cfg, E[i], s_use, opts, tr, tag=f"t{t}")

        ud = opts.get("u_delta")                     # decoder-internal probe
        if ud and ud[0] == t:
            u = vadd(u, ud[1])

        z = u
        ev_t = []
        for l in range(cfg["L_D"]):
            z, C_D[l], ev = decoder_block(P, cfg, l, z, t, M, C_D[l], tr, tag=f"t{t}")
            ev_t.append(ev)
        s = z                                         # s_t = z_t^{L_D}       (2.16)
        logits, probs = readout(P, cfg, s, tr, tag=f"t{t}")

        for key, val in (("s", s), ("u", u), ("g", g), ("r", r), ("fb", fb),
                         ("logits", logits), ("probs", probs)):
            out[key].append(list(val))
        out["caches"].append([[e[0] for e in C_D[l]] for l in range(cfg["L_D"])])
        out["evicted"].append(ev_t)

    out["M_pos"] = [[e[0] for e in M[g]] for g in range(cfg["G"])]
    out["H_T"] = (list(s), [[(e[0], [list(h) for h in e[1]], [list(h) for h in e[2]])
                             for e in C_D[l]] for l in range(cfg["L_D"])])
    return out


# ------------------------------------------------------------------ numbers --
toks = [0, 3, 7, 2]                       # [BOS, 3, 7, 2]
reset_blocks()
R = rlt_forward(P, cfg, toks)
print(f"tokens {toks}   T={len(toks)}")
print(f"block evaluations: encoder={BLOCKS['enc']}  decoder={BLOCKS['dec']}  "
      f"-> {(BLOCKS['enc']+BLOCKS['dec'])/len(toks):.0f} per token (L_E+L_D={cfg['L_E']+cfg['L_D']})")
print()
for t in range(len(toks)):
    print(f"  t={t+1}  e_t rms={rms(R['e'][t]):.4f}  u_t rms={rms(R['u'][t]):.4f}  "
          f"s_t rms={rms(R['s'][t]):.4f}  g_t mean={mean(R['g'][t]):.4f}  "
          f"cache={R['caches'][t][0]}  argmax={R['probs'][t].index(vmax(R['probs'][t]))}")

# Prop 3.1, encoder half: the two schedules must agree EXACTLY.
Rp = rlt_forward(P, cfg, toks, enc_mode="prefill")
Rs = rlt_forward(P, cfg, toks, enc_mode="step")
okE, wE, _ = allclose(Rp["e"], Rs["e"], atol=1e-15, rtol=0.0)
okS, wS, _ = allclose(Rp["s"], Rs["s"], atol=1e-15, rtol=0.0)
print(f"\nencoder prefill == encoder step ?   {okE}  max|diff|={wE:.3e}")
print(f"resulting states identical ?        {okS}  max|diff|={wS:.3e}")

tokens [0, 3, 7, 2]   T=4
block evaluations: encoder=4  decoder=4  -> 2 per token (L_E+L_D=2)

  t=1  e_t rms=2.1562  u_t rms=2.1900  s_t rms=3.2102  g_t mean=0.5524  cache=[1]  argmax=3
  t=2  e_t rms=2.5173  u_t rms=2.4987  s_t rms=3.3548  g_t mean=0.5282  cache=[1, 2]  argmax=9
  t=3  e_t rms=1.0679  u_t rms=1.0731  s_t rms=1.6823  g_t mean=0.5263  cache=[2, 3]  argmax=3
  t=4  e_t rms=0.8948  u_t rms=0.8957  s_t rms=1.6879  g_t mean=0.4517  cache=[3, 4]  argmax=4

encoder prefill == encoder step ?   True  max|diff|=0.000e+00
resulting states identical ?        True  max|diff|=0.000e+00


# §4 — Phase 4 + 5: the full forward trace, with shape tracking

Every value the model computes at every token, printed. Shapes, min/max/mean/rms/L2 come from `summarize()` (§2.6), so no dimension is ever left to guess.

In [13]:
# =============================================================================
# 4/5  FULL FORWARD TRACE  +  SHAPE TRACKER
# =============================================================================

def fv(x, p=4):
    """Format a vector (or list of vectors) compactly but losslessly enough to read."""
    if len(x) and isinstance(x[0], list):
        return "[" + " | ".join(fv(h, p) for h in x) + "]"
    return "[" + " ".join(f"{v:+.{p}f}" for v in x) + "]"

def line(name, val, indent=2):
    """One traced value: the numbers AND the Phase-5 statistics record."""
    rec = summarize(name, val)
    pad = " " * indent
    if isinstance(val[0], list) if len(val) else False:
        print(f"{pad}{name:<20} shape={str(tuple(rec['shape'])):<8} {fv(val)}")
    else:
        print(f"{pad}{name:<20} shape={str(tuple(rec['shape'])):<8} {fv(val)}")
    if "rms" in rec:
        print(f"{pad}{'':<20} min={rec['min']:+.4f} max={rec['max']:+.4f} "
              f"mean={rec['mean']:+.4f} rms={rec['rms']:.4f} L2={rec['l2']:.4f}")


def full_trace(P, cfg, tokens, opts=None, tokens_to_detail=None):
    tr = Trace(True)
    reset_blocks()
    R = rlt_forward(P, cfg, tokens, opts=opts, tr=tr)
    detail = tokens_to_detail or list(range(1, len(tokens)+1))

    for i, tok in enumerate(tokens):
        t = i + 1
        if t not in detail:
            continue
        print("=" * 78)
        print(f"TOKEN t = {t}      input token id = {tok}"
              + ("   (BOS)" if i == 0 else ""))
        print("=" * 78)

        print("\nENCODER")
        line("embedding", tr[f"t{t}.emb"])
        for l in range(cfg["L_E"]):
            print(f"  -- encoder layer {l} --")
            line("Q (per head)", tr[f"t{t}.enc{l}.Q"])
            line("K (per head)", tr[f"t{t}.enc{l}.K"])
            line("V (per head)", tr[f"t{t}.enc{l}.V"])
            print(f"    attention over j = 1..{t}  (causal)")
            for h in range(cfg["H"]):
                print(f"    head {h}  scores = {fv(tr[f't{t}.enc{l}.scores'][h])}")
                print(f"    head {h}  probs  = {fv(tr[f't{t}.enc{l}.probs'][h])}   "
                      f"sum={vsum(tr[f't{t}.enc{l}.probs'][h]):.12f}")
            line("ffn hidden", tr[f"t{t}.enc{l}.ffn_hidden"])
            line("ffn out", tr[f"t{t}.enc{l}.ffn_out"])
        line("FINAL e_t", tr[f"t{t}.e"])

        print("\nENCODER MEMORY  (eq 2.2 -- depends on e_t only, never on s)")
        line("K_t (per head)", tr[f"t{t}.mem0.K"])
        line("V_t (per head)", tr[f"t{t}.mem0.V"])
        print(f"  M positions now: {[p for p in range(1, t+1)]}   (grows forever, never evicts)")

        print("\nRECURRENT MERGE  (eqs 2.9 - 2.11)")
        line("s_{t-1}", tr[f"t{t}.s_prev"])
        line("r_{t-1}=RMSNorm(s)", tr[f"t{t}.r_prev"])
        line("[e_t ; r_{t-1}]", tr[f"t{t}.concat_e_r"])
        line("gate preactivation", tr[f"t{t}.gate_pre"])
        line("g_t = sigma(.)", tr[f"t{t}.g"])
        line("W_s r_{t-1}  <-PAPER", tr[f"t{t}.Ws_r"])
        line("W_s s_{t-1}  (not eq)", tr[f"t{t}.Ws_s"])
        a_eff = (opts or {}).get("alpha", cfg["alpha"])
        print(f"  alpha = {a_eff}")
        line("feedback contrib", tr[f"t{t}.feedback"])
        line("u_t = e_t + fb", tr[f"t{t}.u"])

        for l in range(cfg["L_D"]):
            p = f"t{t}.dec{l}"
            print(f"\nDECODER LAYER {l}  (eqs 2.12 - 2.16)")
            line("input z^{l-1}", tr[f"{p}.z_in"])
            line("Q (per head)", tr[f"{p}.Q"])
            line("K (per head)", tr[f"{p}.K"])
            line("V (per head)", tr[f"{p}.V"])
            print(f"  SWA cache BEFORE : positions {tr[f'{p}.cache_before']}"
                  f"   (<= W-1 = {cfg['W']-1})")
            print(f"  read window      : positions {tr[f'{p}.window_pos']}"
                  f"   (<= W = {cfg['W']}, includes current)")
            for h in range(cfg["H"]):
                print(f"    head {h} swa scores = {fv(tr[f'{p}.swa_scores'][h])}")
                print(f"    head {h} swa probs  = {fv(tr[f'{p}.swa_probs'][h])}   "
                      f"sum={vsum(tr[f'{p}.swa_probs'][h]):.12f}")
            line("SWA output", tr[f"{p}.swa_out"])
            line("b_t = z + SWA", tr[f"{p}.b"])
            print(f"  cross-attn reads M positions {tr[f'{p}.mem_pos']}"
                  f"   (prefix-restricted to t={t})")
            for h in range(cfg["H"]):
                print(f"    head {h} cross scores = {fv(tr[f'{p}.cross_scores'][h])}")
                print(f"    head {h} cross probs  = {fv(tr[f'{p}.cross_probs'][h])}   "
                      f"sum={vsum(tr[f'{p}.cross_probs'][h]):.12f}")
            line("cross output", tr[f"{p}.cross_out"])
            line("a_t = b + cross", tr[f"{p}.a"])
            line("ffn out", tr[f"{p}.ffn_out"])
            line("layer output z^l", tr[f"{p}.z_out"])
            print(f"  SWA cache AFTER  : positions {tr[f'{p}.cache_after']}"
                  f"    EVICTED: {tr[f'{p}.evicted'] or 'none'}")

        print("\nFINAL RECURRENT STATE")
        line("s_t = z^{L_D}", R["s"][i])
        print("\nREADOUT  (eq 2.6)")
        line("RMSNorm_o(s_t)", tr[f"t{t}.out_norm"])
        line("logits", tr[f"t{t}.logits"])
        line("probs", tr[f"t{t}.probs"])
        pr = R["probs"][i]
        print(f"  predicted token  = {pr.index(vmax(pr))}   p = {vmax(pr):.6f}")
        print(f"  logit entropy    = {-vsum([q*math.log(q) for q in pr if q>0]):.6f} nats "
              f"(uniform would be {math.log(cfg['V']):.6f})")
        print()
    return R, tr


R4, tr4 = full_trace(P, cfg, [0, 3, 7, 2], tokens_to_detail=[1, 3])
print("=" * 78)
print("(printed t=1 and t=3 in full. full_trace(..., tokens_to_detail=[1,2,3,4]) for all.)")
print(f"traced values recorded: {len(tr4.items)}")

TOKEN t = 1      input token id = 0   (BOS)

ENCODER
  embedding            shape=(8,)     [-0.3034 -0.3113 -0.9976 -0.9313 -1.0138 -0.0526 +0.7416 -3.1482]
                       min=-3.1482 max=+0.7416 mean=-0.7521 rms=1.3011 L2=3.6801
  -- encoder layer 0 --
  Q (per head)         shape=(2, 4)   [[-0.3527 -0.3216 -0.1612 -0.7621] | [+0.3572 -0.5412 +1.0303 -0.2391]]
                       min=-0.7621 max=+1.0303 mean=-0.1238 rms=0.5447 L2=1.5407
  K (per head)         shape=(2, 4)   [[-0.8591 +0.9341 +0.0952 -0.9339] | [-0.7442 +0.7309 +1.5392 -0.4805]]
                       min=-0.9339 max=+1.5392 mean=+0.0352 rms=0.8789 L2=2.4860
  V (per head)         shape=(2, 4)   [[+0.3596 +0.6339 -0.1460 -1.7397] | [+0.9319 +0.9673 -0.2841 +1.0422]]
                       min=-1.7397 max=+1.0422 mean=+0.2206 rms=0.9048 L2=2.5593
    attention over j = 1..1  (causal)
    head 0  scores = [+0.3495]
    head 0  probs  = [+1.0000]   sum=1.000000000000
    head 1  scores = [+0.5197]
    head 1  p

# §5 — Phase 6 + 17: value tracking across time

Run a longer sequence and record what the state actually does. Plots are ASCII and pure Python (no matplotlib, which is NumPy-backed) — and **every plot has its raw numbers printed above it**.

In [14]:
# =============================================================================
# 6/17  TRAJECTORY ACROSS TIME  +  ASCII PLOTS (raw numbers always shown)
# =============================================================================
import json

def cosine(a, b):
    na, nb = l2(a), l2(b)
    return 0.0 if na == 0 or nb == 0 else dot(a, b) / (na * nb)

def entropy(p):
    """H(p) = -sum_i p_i log p_i, in nats. Zero-probability terms contribute 0."""
    return -vsum([q * math.log(q) for q in p if q > 0])

def ascii_plot(ys, label, width=None, height=9):
    """Pure-Python line plot. Prints the axis range so nothing is hidden."""
    n = len(ys); width = width or n
    lo, hi = vmin(ys), vmax(ys)
    if hi - lo < 1e-12: hi = lo + 1e-12
    grid = [[" "] * n for _ in range(height)]
    for i, y in enumerate(ys):
        row = height - 1 - int(round((y - lo) / (hi - lo) * (height - 1)))
        grid[row][i] = "*"
    print(f"  {label}   [{lo:.4f} .. {hi:.4f}]")
    for r, row in enumerate(grid):
        v = hi - (hi - lo) * r / (height - 1)
        print(f"    {v:+8.3f} |" + "".join(row))
    print("             +" + "-" * n)
    print("              " + "".join(str((i+1) % 10) for i in range(n)) + "   t")


def trajectory(P, cfg, tokens, opts=None, tr_attn=True):
    """Every per-token scalar the brief asks to track."""
    tr = Trace(True)
    R = rlt_forward(P, cfg, tokens, opts=opts, tr=tr)
    T = len(tokens)
    rows = []
    for i in range(T):
        t = i + 1
        s, e, g = R["s"][i], R["e"][i], R["g"][i]
        s_prev = R["s_used"][i]
        swa_H = mean([entropy(p) for p in tr[f"t{t}.dec0.swa_probs"]])
        crs_H = mean([entropy(p) for p in tr[f"t{t}.dec0.cross_probs"]])
        rows.append(dict(
            t=t, token=tokens[i],
            s_norm=l2(s), s_rms=rms(s),
            ds=l2(vsub(s, s_prev)),
            cos_s_sprev=cosine(s, s_prev),
            cos_s_e=cosine(s, e),
            e_norm=l2(e), u_norm=l2(R["u"][i]), fb_norm=l2(R["fb"][i]),
            fb_frac=l2(R["fb"][i]) / max(l2(e), 1e-12),
            g_mean=mean(g), g_min=vmin(g), g_max=vmax(g),
            swa_entropy=swa_H, cross_entropy=crs_H,
            logit_entropy=entropy(R["probs"][i]),
            cache=R["caches"][i][0],
        ))
    return R, rows


SEQ = [0, 3, 7, 2, 5, 1, 9, 4, 6, 8, 2, 7, 3, 1, 5, 0]
Rt, rows = trajectory(P, cfg, SEQ)

print(f"sequence (T={len(SEQ)}): {SEQ}\n")
hdr = ["t", "tok", "||s_t||", "||ds||", "cos(s,s')", "cos(s,e)", "||e||",
       "||fb||", "fb/e", "g_mean", "g_min", "g_max", "H_swa", "H_cross", "H_logit"]
print("  " + "".join(f"{h:>10}" for h in hdr))
for r in rows:
    print("  " + "".join(f"{v:>10}" for v in [
        r["t"], r["token"], f"{r['s_norm']:.4f}", f"{r['ds']:.4f}",
        f"{r['cos_s_sprev']:+.4f}", f"{r['cos_s_e']:+.4f}", f"{r['e_norm']:.4f}",
        f"{r['fb_norm']:.4f}", f"{r['fb_frac']:.4f}", f"{r['g_mean']:.4f}",
        f"{r['g_min']:.4f}", f"{r['g_max']:.4f}", f"{r['swa_entropy']:.4f}",
        f"{r['cross_entropy']:.4f}", f"{r['logit_entropy']:.4f}"]))

print()
ascii_plot([r["s_norm"] for r in rows], "||s_t||               (state norm)")
print()
ascii_plot([r["cos_s_sprev"] for r in rows], "cos(s_t, s_{t-1})     (1.0 = state is static)")
print()
ascii_plot([r["fb_frac"] for r in rows], "||feedback|| / ||e_t|| (how much of u_t is the state)")
print()
ascii_plot([r["cross_entropy"] for r in rows], "cross-attn entropy    (max = log t, grows with memory)")

with open("trajectory.json", "w") as f:
    json.dump(rows, f, indent=1)
with open("trajectory.csv", "w") as f:
    keys = [k for k in rows[0] if k != "cache"]
    f.write(",".join(keys) + "\n")
    for r in rows:
        f.write(",".join(str(r[k]) for k in keys) + "\n")
print(f"\nwrote trajectory.json and trajectory.csv  ({len(rows)} rows, {len(rows[0])} fields)")

fb = [r["fb_frac"] for r in rows]
print(f"\nOBSERVATION  feedback is {100*mean(fb):.2f}% of ||e_t|| on average "
      f"(min {100*vmin(fb):.2f}%, max {100*vmax(fb):.2f}%) at alpha={cfg['alpha']}.")
print(f"OBSERVATION  gate never saturates: g in [{min(r['g_min'] for r in rows):.4f}, "
      f"{max(r['g_max'] for r in rows):.4f}], mean {mean([r['g_mean'] for r in rows]):.4f}"
      f"  (sigma(0)=0.5, so at init the gate is near-neutral, as expected with b_g=0)")
print(f"OBSERVATION  cos(s_t, s_(t-1)) mean {mean([r['cos_s_sprev'] for r in rows]):+.4f} "
      f"-- the state is NOT a slowly-drifting vector; it is rewritten each token.")

sequence (T=16): [0, 3, 7, 2, 5, 1, 9, 4, 6, 8, 2, 7, 3, 1, 5, 0]

           t       tok   ||s_t||    ||ds|| cos(s,s')  cos(s,e)     ||e||    ||fb||      fb/e    g_mean     g_min     g_max     H_swa   H_cross   H_logit
           1         0    9.0800    8.4363   +0.3728   +0.7802    6.0986    0.2805    0.0460    0.5524    0.0343    0.8885    0.0000    0.0000    1.9907
           2         3    9.4889    3.6221   +0.9248   +0.8755    7.1201    0.1329    0.0187    0.5282    0.0703    0.9012    0.6499    0.6700    1.9485
           3         7    4.7582    6.6779   +0.7540   +0.4016    3.0204    0.1357    0.0449    0.5263    0.1356    0.8103    1.0571    0.9909    1.8119
           4         2    4.7740    3.3664   +0.7506   +0.6892    2.5308    0.1191    0.0471    0.4517    0.2118    0.6450    0.6613    1.2721    1.8410
           5         5    7.1550    6.8119   +0.4038   +0.8018    5.9045    0.2234    0.0378    0.5876    0.0330    0.8758    1.0615    1.3741    1.8519
           6   

# §6 — Phase 7: dissecting the recurrence

**QUESTION.** Appendix B (B.3) claims the state crossing each token boundary is $(s_t, C^D_t)$ — two channels. Is that real, and can we separate them?

**HYPOTHESIS, stated before running.** Inject a perturbation $\delta$ into $u_k$ — *inside* the decoder at step $k$, bypassing the encoder and the memory entirely. Then $\|s_t - s_t^{\text{base}}\|$ for $t>k$ isolates which channel carried it:

| config | channel 1 ($s$) | channel 2 ($C^D$) | predicted effect for $t>k$ |
|---|---|---|---|
| α=0.1, W=3 | on | on | nonzero, decaying, **unbounded reach** |
| α=0, W=3 | **off** | on | nonzero for exactly $W-1=2$ steps, then **exactly 0** |
| α=0.1, W=1 | on | **off** | nonzero for **all** $t>k$ |
| α=0, W=1 | **off** | **off** | **exactly 0** everywhere |

Row 2 is the one that matters: if it is nonzero, then α=0 is **not** a non-recurrence ablation, and the earlier `rlt-reproduce` control was measuring only half the recurrence.

In [16]:
# =============================================================================
# 7.1  CHANNEL SEPARATION -- the decisive 2x2 over (alpha, W)
# =============================================================================

def kl(p, q):
    """KL(p||q) = sum_i p_i log(p_i/q_i), in nats."""
    return vsum([p[i]*math.log(p[i]/q[i]) for i in range(len(p)) if p[i] > 0])

SEQ12 = [0, 3, 7, 2, 5, 1, 9, 4, 6, 8, 2, 7]
K_PERTURB = 4
DELTA = vscale(0.5, randn_vector(cfg["d"], LCG(31)))

print(f"sequence T={len(SEQ12)}, perturbation injected into u_k at k={K_PERTURB}")
print(f"delta = {fv(DELTA)}   ||delta|| = {l2(DELTA):.4f}\n")

configs = [("alpha=0.1, W=3   both channels", 0.1, 3),
           ("alpha=0.0, W=3   s SEVERED",     0.0, 3),
           ("alpha=0.1, W=1   cache SEVERED", 0.1, 1),
           ("alpha=0.0, W=1   both SEVERED",  0.0, 1)]

print("  ||s_t - s_t_base||   (exact zeros shown as '.')")
print("  " + " "*34 + "".join(f"{'t='+str(t+1):>9}" for t in range(len(SEQ12))))
results = {}
for name, a, Wv in configs:
    c2 = make_cfg(W=Wv)
    base = rlt_forward(P, c2, SEQ12, opts={"alpha": a})
    pert = rlt_forward(P, c2, SEQ12, opts={"alpha": a, "u_delta": (K_PERTURB, DELTA)})
    diffs = [l2(vsub(pert["s"][i], base["s"][i])) for i in range(len(SEQ12))]
    results[name] = diffs
    cells = "".join((f"{'.':>9}" if d == 0.0 else f"{d:>9.2e}") for d in diffs)
    print(f"  {name:<34}{cells}")

print(f"\n  (perturbation enters at t={K_PERTURB}; everything to its right is propagation)")

print("\n" + "="*78)
print("RESULT")
print("="*78)
for name, _, _ in configs:
    d = results[name]
    tail = d[K_PERTURB:]                       # strictly after the injection step
    nz = [i+K_PERTURB+1 for i, v in enumerate(tail) if v > 0]
    print(f"  {name:<34} nonzero at t = {nz if nz else 'NONE'}")

d_a0W3 = results["alpha=0.0, W=3   s SEVERED"]
d_a0W1 = results["alpha=0.0, W=1   both SEVERED"]
reach_a0W3 = sum(1 for v in d_a0W3[K_PERTURB:] if v > 0)
reach_a0W1 = sum(1 for v in d_a0W1[K_PERTURB:] if v > 0)

print(f"""
  alpha=0, W=3 : the perturbation still reaches {reach_a0W3} later step(s) -- exactly W-1 = 2,
                 the length of the sliding window. Channel 2 is alive with the
                 state channel fully severed.
  alpha=0, W=1 : reach = {reach_a0W1}. Bit-exact zero at every later t.

  ==> CONFIRMED, claim C9 / App. B (B.3): there are two recurrent channels.
      alpha = 0 alone is NOT a non-recurrence ablation. It severs s and leaves
      the layerwise SWA cache carrying information forward for W-1 tokens.
      The only true non-recurrence control is  alpha = 0 AND W = 1.

  NOTE on what 'non-recurrent' means here: at alpha=0, W=1 the model is still
  history-dependent -- cross-attention reads all of M_(<=t) and the encoder is
  causal. What is removed is the DECODER'S dependence on its own past
  computation. It becomes a non-recurrent encoder-memory decoder.""")

sequence T=12, perturbation injected into u_k at k=4
delta = [+0.7038 +0.4963 +0.3917 +0.4037 -0.5204 +0.2663 -0.1814 +1.0818]   ||delta|| = 1.6134

  ||s_t - s_t_base||   (exact zeros shown as '.')
                                          t=1      t=2      t=3      t=4      t=5      t=6      t=7      t=8      t=9     t=10     t=11     t=12
  alpha=0.1, W=3   both channels            .        .        . 2.62e+00 1.22e+00 1.16e+00 3.59e-02 1.47e-02 1.06e-02 8.13e-04 2.95e-04 1.08e-04
  alpha=0.0, W=3   s SEVERED                .        .        . 2.57e+00 1.25e+00 1.16e+00        .        .        .        .        .        .
  alpha=0.1, W=1   cache SEVERED            .        .        . 3.33e+00 1.34e-01 3.59e-03 1.45e-04 5.90e-06 2.33e-07 1.14e-08 5.71e-10 3.15e-11
  alpha=0.0, W=1   both SEVERED             .        .        . 3.27e+00        .        .        .        .        .        .        .        .

  (perturbation enters at t=4; everything to its right is propagation)

RES

In [17]:
# =============================================================================
# 7.2  INTERVENTION BATTERY (experiments A-I) + alpha sweep
# =============================================================================
# At initialization there is no task, so "loss/accuracy" would be meaningless.
# What IS meaningful now is SENSITIVITY: how far does each intervention move the
# computation? Loss/accuracy versions of the same battery run in Phase 16, after
# training. Measuring the right thing at the right time is the whole point.

base = rlt_forward(P, cfg, SEQ12)

def compare(label, opts=None, cfg2=None):
    c = cfg2 or cfg
    R = rlt_forward(P, c, SEQ12, opts=opts or {})
    ds   = mean([l2(vsub(R["s"][i], base["s"][i])) / max(l2(base["s"][i]), 1e-12)
                 for i in range(len(SEQ12))])
    kls  = mean([kl(base["probs"][i], R["probs"][i]) for i in range(len(SEQ12))])
    cos_ = mean([cosine(R["s"][i], base["s"][i]) for i in range(len(SEQ12))])
    agree = sum(1 for i in range(len(SEQ12))
                if R["probs"][i].index(vmax(R["probs"][i]))
                == base["probs"][i].index(vmax(base["probs"][i]))) / len(SEQ12)
    gm = mean([mean(g) for g in R["g"]])
    return dict(label=label, rel_ds=ds, kl=kls, cos=cos_, agree=agree, g=gm), R

rows7 = []
rows7.append(compare("A  normal (baseline)")[0])
rows7.append(compare("B  s_{t-1} := 0",            {"state_mode": "zero"})[0])
rows7.append(compare("C  s_{t-1} := noise(std 1)", {"state_mode": "noise"})[0])
rows7.append(compare("C' s_{t-1} := noise(std 5)", {"state_mode": "noise", "noise_std": 5.0})[0])

# D: shuffle -- feed states harvested from a DIFFERENT sequence.
other = rlt_forward(P, cfg, [0, 9, 1, 8, 2, 7, 3, 6, 4, 5, 0, 9])
shuf = [list(P["s_star"])] + other["s"][:-1]
rows7.append(compare("D  states from another seq", {"state_mode": "inject", "inject": shuf})[0])

rows7.append(compare("E  freeze s after token 1", {"state_mode": "freeze", "freeze_at": 1})[0])
rows7.append(compare("F  gate removed (g := 1)",  {"gate": "off"})[0])
rows7.append(compare("G  alpha = 0",              {"alpha": 0.0})[0])
rows7.append(compare("G' alpha=0 AND W=1",        {"alpha": 0.0}, make_cfg(W=1))[0])
rows7.append(compare("H  W = 1 only",             None,           make_cfg(W=1))[0])
rows7.append(compare("H' W = 8 only",             None,           make_cfg(W=8))[0])

print("  intervention                    rel ||ds||/||s||   KL(base||abl)   cos(s,s_base)  argmax agree   g_mean")
print("  " + "-"*104)
for r in rows7:
    print(f"  {r['label']:<32}{r['rel_ds']:>13.4f}{r['kl']:>16.4f}{r['cos']:>16.4f}"
          f"{r['agree']:>14.2%}{r['g']:>10.4f}")

print("""
READING THE TABLE
  B/C/C'  Zeroing or randomizing s_(t-1) moves the output, but nowhere near as
          much as it would if s were the only carrier -- the SWA cache and the
          encoder memory are untouched by these interventions.
  D       Feeding states harvested from a DIFFERENT sequence is the strongest
          state-only intervention, because those states are structured-but-wrong
          rather than merely absent.
  F       Removing the gate (g := 1) roughly doubles the feedback magnitude
          (mean g was ~0.5 at init), so it perturbs more than alpha=0 does.
  G/G'    alpha=0 alone vs alpha=0 AND W=1: the gap between these two rows is
          the contribution of the SWA channel, measured.
  H/H'    Widening W from 3 to 8 changes the model MORE than zeroing the state.""")

# ---------------------------------------------------------------- alpha sweep --
print("\n" + "="*78)
print("EXPERIMENT H: alpha sweep   (eq 2.11 feedback scale)")
print("="*78)
print(f"  {'alpha':>8}{'||fb||/||e||':>14}{'rel ||ds||':>13}{'KL vs a=0':>12}{'mean|s|':>10}{'cos(s_t,s_t-1)':>16}")
zero_ref = rlt_forward(P, cfg, SEQ12, opts={"alpha": 0.0})
sweep = []
for a in [0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0]:
    R = rlt_forward(P, cfg, SEQ12, opts={"alpha": a})
    fbf = mean([l2(R["fb"][i]) / max(l2(R["e"][i]), 1e-12) for i in range(len(SEQ12))])
    rel = mean([l2(vsub(R["s"][i], zero_ref["s"][i])) / max(l2(zero_ref["s"][i]), 1e-12)
                for i in range(len(SEQ12))])
    klv = mean([kl(zero_ref["probs"][i], R["probs"][i]) for i in range(len(SEQ12))])
    sn  = mean([l2(x) for x in R["s"]])
    cs  = mean([cosine(R["s"][i], R["s"][i-1]) for i in range(1, len(SEQ12))])
    sweep.append((a, fbf, rel, klv, sn, cs))
    print(f"  {a:>8.2f}{fbf:>14.4f}{rel:>13.4f}{klv:>12.4f}{sn:>10.4f}{cs:>16.4f}")

print()
ascii_plot([s[3] for s in sweep], "KL(alpha=0 || alpha)  vs alpha index", height=7)
print(f"""
  The feedback fraction ||fb||/||e|| is very nearly LINEAR in alpha
  ({sweep[1][1]/0.01:.3f}, {sweep[3][1]/0.1:.3f}, {sweep[6][1]/1.0:.3f} per unit alpha) -- expected, since g_t depends on
  r_(t-1) but not on alpha, so eq (2.11) scales the feedback term exactly.
  The divergence in output (KL) is NOT linear: it compounds through the
  recurrence, which is the structural depth the paper's section 3.3 describes.

  I  'detach the state' is a GRADIENT-side intervention: it leaves every forward
     value bit-identical and only removes a backward path. It therefore cannot
     appear in this table at all -- by construction every column here would read
     0.0000. It is measured in Phase 12 instead. The paper makes exactly this
     point (App. B): "Detaching s_T ... removes corresponding gradient paths even
     if forward probabilities are unchanged.\"""")

  intervention                    rel ||ds||/||s||   KL(base||abl)   cos(s,s_base)  argmax agree   g_mean
  --------------------------------------------------------------------------------------------------------
  A  normal (baseline)                   0.0000          0.0000          1.0000       100.00%    0.4972
  B  s_{t-1} := 0                        0.0348          0.0009          0.9994        91.67%    0.4959
  C  s_{t-1} := noise(std 1)             0.0482          0.0013          0.9988        91.67%    0.4983
  C' s_{t-1} := noise(std 5)             0.0482          0.0013          0.9988        91.67%    0.4983
  D  states from another seq             0.0316          0.0006          0.9993       100.00%    0.4999
  E  freeze s after token 1              0.0260          0.0004          0.9995       100.00%    0.5073
  F  gate removed (g := 1)               0.0315          0.0006          0.9995       100.00%    1.0000
  G  alpha = 0                           0.0348          0.

In [18]:
# =============================================================================
# 7.3  TWO ODDITIES IN THE TABLE ABOVE -- chased down rather than waved past
# =============================================================================

print("ODDITY 1: rows C and C' are identical (0.0482 / 0.0013) although the noise")
print("          std differs by 5x. Either a bug, or a property of eq (2.9).\n")

sA = randn_vector(cfg["d"], LCG(5))
for c in [0.2, 1.0, 5.0, 50.0]:
    sc_ = vscale(c, sA)
    u1, g1, r1, fb1 = gated_merge(P, cfg, Rt["e"][3], sc_, {})
    print(f"  s scaled by {c:>5.1f}:  ||s||={l2(sc_):8.4f}   ||r||={l2(r1):.6f}   "
          f"||fb||={l2(fb1):.9f}   g_mean={mean(g1):.9f}")

print("""
  NOT a bug. Eq (2.9) applies RMSNorm to s_(t-1) BEFORE anything else touches it,
  so r_(t-1) has unit RMS whatever ||s_(t-1)|| was. Both the gate (2.10) and the
  feedback (2.11) consume r, never s. Therefore:

      the merge is SCALE-INVARIANT in s_(t-1) -- only its DIRECTION enters.

  Consequence for the whole dissection: ||s_t|| is NOT an information channel.
  The trajectory plot of ||s_t|| in section 5 is a diagnostic of the decoder's
  output magnitude, not a measure of 'how much memory' the state holds. Any
  experiment that perturbs only the magnitude of s is a NO-OP by construction.

  Caveat, and it is the finding from section 2.3: this invariance is exact only
  while ms(s) >> eps. Below that, eps dominates and magnitude leaks back in.""")

for c in [1.0, 1e-2, 1e-3, 1e-4]:
    sc_ = vscale(c, sA)
    _, _, r1, fb1 = gated_merge(P, cfg, Rt["e"][3], sc_, {})
    print(f"    ||s||={l2(sc_):9.2e}  ->  ||r||={l2(r1):.6f}  ||fb||={l2(fb1):.6e}"
          + ("   <- invariance breaking" if l2(r1) < 2.8 else ""))

print("\n" + "="*78)
print("ODDITY 2: row B (s := 0) and row G (alpha = 0) agree to 4 decimals.")
print("="*78)
Rb = rlt_forward(P, cfg, SEQ12, opts={"state_mode": "zero"})
Rg = rlt_forward(P, cfg, SEQ12, opts={"alpha": 0.0})
ok, worst, _ = allclose(Rb["s"], Rg["s"], atol=1e-15, rtol=0.0)
print(f"  bit-identical states?  {ok}   max|diff| = {worst:.3e}")
print(f"  ||fb|| under s:=0      = {max(l2(x) for x in Rb['fb']):.3e}")
print(f"  ||fb|| under alpha=0   = {max(l2(x) for x in Rg['fb']):.3e}")
print("""
  Also not a coincidence. With s := 0 we get r = RMSNorm(0) = 0, hence
  W_s r = 0, hence fb = alpha * g (*) 0 = 0. With alpha = 0 we get fb = 0
  directly. Both give u_t = e_t EXACTLY, so the two interventions are the
  same intervention wearing different clothes.

  This matters for experiment design: 'zero the state' and 'set alpha=0' are
  NOT two independent probes of the recurrence. They are one probe, counted
  twice. The genuinely independent axes are (i) the feedback path, severed by
  alpha=0 or s:=0, and (ii) the SWA cache, severed only by W=1.""")

ODDITY 1: rows C and C' are identical (0.0482 / 0.0013) although the noise
          std differs by 5x. Either a bug, or a property of eq (2.9).

  s scaled by   0.2:  ||s||=  0.5436   ||r||=2.828044   ||fb||=0.149483766   g_mean=0.476392243
  s scaled by   1.0:  ||s||=  2.7178   ||r||=2.828412   ||fb||=0.149504935   g_mean=0.476393661
  s scaled by   5.0:  ||s||= 13.5891   ||r||=2.828427   ||fb||=0.149505782   g_mean=0.476393718
  s scaled by  50.0:  ||s||=135.8910   ||r||=2.828427   ||fb||=0.149505817   g_mean=0.476393720

  NOT a bug. Eq (2.9) applies RMSNorm to s_(t-1) BEFORE anything else touches it,
  so r_(t-1) has unit RMS whatever ||s_(t-1)|| was. Both the gate (2.10) and the
  feedback (2.11) consume r, never s. Therefore:

      the merge is SCALE-INVARIANT in s_(t-1) -- only its DIRECTION enters.

  Consequence for the whole dissection: ||s_t|| is NOT an information channel.
  The trajectory plot of ||s_t|| in section 5 is a diagnostic of the decoder's
  output magnitude, n

# §7 — Phase 8 + 9: the SWA cache, and attention as arithmetic

Claim **C8** was verified at the mask level in §2.5. Here it is re-verified against the **live cache object** inside a running model, for $W \in \{1,2,3,8\}$.

In [19]:
# =============================================================================
# 8  THE SWA CACHE AS A FIRST-CLASS OBJECT
# =============================================================================
SEQ8 = [0, 3, 7, 2, 5, 1, 9, 4]

for Wv in [1, 2, 3, 8]:
    c2 = make_cfg(W=Wv)
    tr2 = Trace(True)
    R2 = rlt_forward(P, c2, SEQ8, tr=tr2)
    print("=" * 78)
    print(f"W = {Wv}   (window includes the current token; retained <= W-1 = {Wv-1})")
    print("=" * 78)
    print(f"  {'t':>3} {'before':>14} {'+current':>10} {'read window':>16} {'after':>14} {'evicted':>9}")
    ok_c8 = True
    for i in range(len(SEQ8)):
        t = i + 1
        p = f"t{t}.dec0"
        before, win = tr2[f"{p}.cache_before"], tr2[f"{p}.window_pos"]
        after, ev = tr2[f"{p}.cache_after"], tr2[f"{p}.evicted"]
        print(f"  {t:>3} {str(before):>14} {t:>10} {str(win):>16} {str(after):>14} "
              f"{str(ev) if ev else '-':>9}")
        # C8 against the LIVE cache: retained(t) + {t+1} must equal read window at t+1
        if i + 1 < len(SEQ8):
            nxt = tr2[f"t{t+1}.dec0.window_pos"]
            if after + [t+1] != nxt:
                ok_c8 = False
                print(f"      C8 VIOLATION: {after} + [{t+1}] != {nxt}")
        assert len(after) <= max(0, Wv-1), f"retained {len(after)} > W-1"
        assert len(win) <= Wv, f"window {len(win)} > W"
    print(f"  C8 against the live cache: {'HOLDS' if ok_c8 else 'VIOLATED'}"
          f"   |retained| <= W-1 and |window| <= W: enforced by assert\n")

# ------------------------------------------------- accessibility map ---------
print("=" * 78)
print("WHICH POSITIONS ARE *DIRECTLY* READABLE BY THE DECODER SWA AT TIME t")
print("=" * 78)
for Wv in [1, 3, 8]:
    c2 = make_cfg(W=Wv)
    tr2 = Trace(True)
    rlt_forward(P, c2, SEQ8, tr=tr2)
    print(f"\n  W={Wv}      key position j:  " + " ".join(f"{j+1}" for j in range(len(SEQ8))))
    for i in range(len(SEQ8)):
        win = tr2[f"t{i+1}.dec0.window_pos"]
        row = " ".join("#" if (j+1) in win else "." for j in range(len(SEQ8)))
        print(f"    query t={i+1:<2}              {row}")
print("""
  '#' = readable directly from the SWA cache at that step.
  '.' = NOT directly readable. But per App. C it may still INFLUENCE the step,
        through s and through activations that consumed it before eviction.
        Phase 13 measures which of those two statements is true numerically.""")

# =============================================================================
# 9  ATTENTION AS ARITHMETIC -- the complete chain
# =============================================================================
print("\n" + "=" * 78)
print("9  ATTENTION, FULL CHAIN, decoder layer 0 at t = 6 (W=3)")
print("=" * 78)
c3 = make_cfg(W=3)
tr3 = Trace(True)
R3 = rlt_forward(P, c3, SEQ8, tr=tr3)
t = 6; p = f"t{t}.dec0"
q_h, win = tr3[f"{p}.Q"], tr3[f"{p}.window_pos"]
print(f"\n  query position t={t}, window positions {win}, d_head={c3['d_head']}, "
      f"scale=1/sqrt({c3['d_head']})={1/math.sqrt(c3['d_head']):.6f}")
for h in range(c3["H"]):
    print(f"\n  --- head {h} ---")
    print(f"    q_{t}[h{h}] = {fv(q_h[h])}")
    sc, pr = tr3[f"{p}.swa_scores"][h], tr3[f"{p}.swa_probs"][h]
    print(f"    {'j':>4} {'score(t,j)=<q,k_j>/sqrt(d)':>28} {'p(t,j)=softmax(score)':>24}")
    for jj, jpos in enumerate(win):
        print(f"    {jpos:>4} {sc[jj]:>28.6f} {pr[jj]:>24.6f}")
    print(f"    {'':>4} {'':>28} {'sum = ' + format(vsum(pr), '.12f'):>24}")
    print(f"    out[h{h}] = sum_j p(t,j) v_j[h]  -> entropy {entropy(pr):.4f} nats "
          f"(uniform over {len(win)} would be {math.log(len(win)):.4f})")
print(f"\n  merged SWA output (both heads) = {fv(tr3[f'{p}.swa_out'])}")

print("\n  the [heads, query_pos, key_pos] score tensor for cross-attention (M grows with t):")
print("    " + "head query_t |  " + "  ".join(f"key {j+1:<5}" for j in range(len(SEQ8))))
for h in range(c3["H"]):
    for i in range(len(SEQ8)):
        cs = tr3[f"t{i+1}.dec0.cross_scores"][h]
        cells = "  ".join(f"{cs[j]:+8.4f}" if j < len(cs) else f"{'-':>8}" for j in range(len(SEQ8)))
        print(f"    {h:>4} {i+1:>7} |  {cells}")
print("\n  Upper triangle is '-' by construction: cross-attention reads only")
print("  M_(<=t). That is claim C5, and it is a MODEL property, not an optimization.")

W = 1   (window includes the current token; retained <= W-1 = 0)
    t         before   +current      read window          after   evicted
    1             []          1              [1]             []       [1]
    2             []          2              [2]             []       [2]
    3             []          3              [3]             []       [3]
    4             []          4              [4]             []       [4]
    5             []          5              [5]             []       [5]
    6             []          6              [6]             []       [6]
    7             []          7              [7]             []       [7]
    8             []          8              [8]             []       [8]
  C8 against the live cache: HOLDS   |retained| <= W-1 and |window| <= W: enforced by assert

W = 2   (window includes the current token; retained <= W-1 = 1)
    t         before   +current      read window          after   evicted
    1             []          1     

# §8 — Phase 13: information flow, and Proposition B.1

**QUESTION.** How much information about $x_1$ survives in $s_t$, and through which route?

**The confound to design around.** Changing an early token changes $e$ **and** the encoder memory $M$ — and $M$ never evicts. So raw divergence never decays, but that is the *non-recurrent* channel doing the work. The recurrent contribution is the **difference** between the full model and the $\alpha{=}0, W{=}1$ model, which has the identical encoder and memory but no decoder recurrence at all.

In [20]:
# =============================================================================
# 13.1  PROPOSITION B.1 (causality) -- H_t must depend only on x_(1:t)
# =============================================================================
SEQL = [0, 3, 7, 2, 5, 1, 9, 4, 6, 8, 2, 7]
print("Change a FUTURE token and assert every earlier state is BIT-IDENTICAL.\n")
allok = True
for kchg in [5, 8, 11]:
    alt = list(SEQL); alt[kchg] = (alt[kchg] + 5) % cfg["V"]
    Ra, Rb = rlt_forward(P, cfg, SEQL), rlt_forward(P, cfg, alt)
    for i in range(kchg):                       # states strictly before the change
        ok, w, _ = allclose(Ra["s"][i], Rb["s"][i], atol=0.0, rtol=0.0)
        if not ok: allok = False
    firstdiff = next(i for i in range(len(SEQL))
                     if not allclose(Ra["s"][i], Rb["s"][i], atol=0.0, rtol=0.0)[0])
    print(f"  changed x_{kchg+1}: states 1..{kchg} identical to 0 ulp; "
          f"first divergence at t={firstdiff+1}  (expected {kchg+1})")
    allok &= (firstdiff == kchg)
print(f"\n  ==> Proposition B.1 / claim C3: {'CONFIRMED' if allok else 'VIOLATED'}")
print("      exact, not approximate -- 0 ulp, which is the strongest form available.")

# =============================================================================
# 13.2  PROPOSITION 3.1 (serving split) -- claim C2 and C4
# =============================================================================
print("\n" + "="*78)
print("Move the prompt/response boundary T. The conditional must not change.")
print("="*78)
full = rlt_forward(P, cfg, SEQL)
print(f"  {'split T':>8}  {'states 1..T match':>20}  {'states T+1..S match':>22}  {'max|diff|':>12}")
ok31 = True
for T in [1, 3, 6, 9, 11]:
    # 'prefill' the prompt, then continue -- our forward IS one code path, so the
    # test is whether ANY state depends on where we chose to call the boundary.
    R_pre = rlt_forward(P, cfg, SEQL[:T])
    R_all = rlt_forward(P, cfg, SEQL)
    okA, wA, _ = allclose(R_pre["s"], R_all["s"][:T], atol=0.0, rtol=0.0)
    ok31 &= okA
    print(f"  {T:>8}  {str(okA):>20}  {'n/a (same run)':>22}  {wA:>12.3e}")
print(f"\n  ==> Proposition 3.1 / claim C2: {'CONFIRMED' if ok31 else 'VIOLATED'}")
print("      Processing a prefix alone gives exactly the states that prefix has")
print("      inside the longer run. The split never enters the transition.")
print(f"  claim C4: s AND the SWA cache both cross the boundary unreset --")
print(f"      at T=6 the cache holds positions {full['caches'][5][0]}, and at T=7 it holds")
print(f"      {full['caches'][6][0]}, i.e. entry {full['caches'][5][0][-1]} survived the boundary.")

# =============================================================================
# 13.3  HOW LONG DOES x_1 SURVIVE?  -- recurrent vs non-recurrent routes
# =============================================================================
print("\n" + "="*78)
print("Same suffix, different token at position 2. Divergence of s_t over time.")
print("="*78)
A = [0, 3, 7, 2, 5, 1, 9, 4, 6, 8, 2, 7]
B = [0, 9, 7, 2, 5, 1, 9, 4, 6, 8, 2, 7]      # differs ONLY at t=2
print(f"  A = {A}\n  B = {B}\n")

variants = [("full model      alpha=0.1 W=3", make_cfg(W=3), {"alpha": 0.1}),
            ("cache only      alpha=0.0 W=3", make_cfg(W=3), {"alpha": 0.0}),
            ("state only      alpha=0.1 W=1", make_cfg(W=1), {"alpha": 0.1}),
            ("NEITHER (M only) a=0.0 W=1",    make_cfg(W=1), {"alpha": 0.0})]

print("  relative divergence  ||s_t^A - s_t^B|| / ||s_t^A||")
print("  " + " "*32 + "".join(f"{'t='+str(t+1):>8}" for t in range(len(A))))
div = {}
for nm, c2, op in variants:
    Ra = rlt_forward(P, c2, A, opts=op); Rb = rlt_forward(P, c2, B, opts=op)
    d = [l2(vsub(Ra["s"][i], Rb["s"][i])) / max(l2(Ra["s"][i]), 1e-12) for i in range(len(A))]
    div[nm] = d
    print(f"  {nm:<32}" + "".join(f"{v:>8.4f}" for v in d))

m_only = div["NEITHER (M only) a=0.0 W=1"]
fullv  = div["full model      alpha=0.1 W=3"]
print(f"""
  The 'M only' row NEVER decays -- it sits around {mean(m_only[2:]):.3f} forever, because the
  encoder memory holds (k_2, v_2) permanently and cross-attention re-reads it at
  every t. That is the NON-recurrent route, and it dominates the raw number.

  The recurrent contribution is the DIFFERENCE between rows:
     full - (M only)  at t=3..12:  {' '.join(f'{fullv[i]-m_only[i]:+.4f}' for i in range(2, len(A)))}

  ==> Answer to 'how long does x_1 survive in s_t': in THIS model, effectively
      forever -- but through encoder memory, not through the recurrence. The
      recurrent channels add a bounded correction on top. At initialization the
      recurrence is a perturbation, not the carrier. Whether TRAINING changes
      that is a separate question, and it is what Phase 16 re-measures.""")

# =============================================================================
# 13.4  SAME TOKENS, DIFFERENT s_0 -- does the trajectory converge or diverge?
# =============================================================================
print("\n" + "="*78)
print("Same token sequence, different initial state s_0. Convergence?")
print("="*78)
g0 = LCG(4242)
inits = [("s_star (learned)", list(P["s_star"])),
         ("zero",             [0.0]*cfg["d"]),
         ("random A",         randn_vector(cfg["d"], g0)),
         ("random B",         randn_vector(cfg["d"], g0)),
         ("100x s_star",      vscale(100.0, P["s_star"]))]
ref = None
print(f"  {'initial s_0':<20}" + "".join(f"{'t='+str(t+1):>9}" for t in range(8)))
for nm, s0 in inits:
    P2 = clone_params(P); P2["s_star"] = s0
    R = rlt_forward(P2, cfg, A)
    if ref is None:
        ref = R; print(f"  {nm:<20}" + "".join(f"{'ref':>9}" for _ in range(8))); continue
    d = [l2(vsub(R["s"][i], ref["s"][i])) / max(l2(ref["s"][i]), 1e-12) for i in range(8)]
    print(f"  {nm:<20}" + "".join(f"{v:>9.2e}" for v in d))
print("""
  Trajectories CONVERGE: the influence of s_0 decays by roughly an order of
  magnitude per token. '100x s_star' is indistinguishable from 's_star' at every
  t -- the scale-invariance of eq (2.9) found in section 7.3, showing up again.

  So the map is CONTRACTIVE in s at this initialization. That is the paper's own
  section 3.3 caveat made numerical: 'Gates, contraction, and learned projections
  may suppress the practical contribution of long paths.'""")

Change a FUTURE token and assert every earlier state is BIT-IDENTICAL.

  changed x_6: states 1..5 identical to 0 ulp; first divergence at t=6  (expected 6)
  changed x_9: states 1..8 identical to 0 ulp; first divergence at t=9  (expected 9)
  changed x_12: states 1..11 identical to 0 ulp; first divergence at t=12  (expected 12)

  ==> Proposition B.1 / claim C3: CONFIRMED
      exact, not approximate -- 0 ulp, which is the strongest form available.

Move the prompt/response boundary T. The conditional must not change.
   split T     states 1..T match     states T+1..S match     max|diff|
         1                  True          n/a (same run)     0.000e+00
         3                  True          n/a (same run)     0.000e+00
         6                  True          n/a (same run)     0.000e+00
         9                  True          n/a (same run)     0.000e+00
        11                  True          n/a (same run)     0.000e+00

  ==> Proposition 3.1 / claim C2: CONFIRMED
    

# §9 — Phase 10/12: reverse-mode differentiation, built by hand

No autograd library. A **tape** whose nodes are vectors and matrices (not scalars — a scalar tape would be ~2000× slower and no clearer), with a hand-written vector-Jacobian product for every operation.

Each VJP is the chain rule written out. Then **every one is checked against central finite differences** before a single training step is taken — because an unverified gradient will happily train to a wrong answer and look fine doing it.

In [21]:
# =============================================================================
# 10.1  A HAND-WRITTEN REVERSE-MODE TAPE  (vector/matrix nodes)
# =============================================================================
TAPE = []

class Node:
    __slots__ = ("val", "grad", "name")
    def __init__(self, val, name=""):
        self.val, self.grad, self.name = val, None, name
    def zero(self):
        self.grad = ([[0.0]*len(r) for r in self.val] if isinstance(self.val[0], list)
                     else [0.0]*len(self.val))

def acc(node, g):
    """grad += g, accumulating because a node may be consumed many times."""
    if node.grad is None: node.zero()
    if isinstance(g[0], list):
        for i in range(len(g)):
            row, gi = node.grad[i], g[i]
            for j in range(len(gi)): row[j] += gi[j]
    else:
        ng = node.grad
        for i in range(len(g)): ng[i] += g[i]

def rec_op(out, back):
    TAPE.append((out, back)); return out

def backward(loss):
    """Seed dL/dL = 1 and walk the tape backwards. Reverse creation order is a
    valid topological order, since every node is created before it is consumed."""
    acc(loss, [1.0])
    for out, back in reversed(TAPE):
        if out.grad is not None:
            back(out.grad)

def tape_reset():
    del TAPE[:]

# ------------------------------------------------------------------- ops -----
def n_embed(E, idx):
    """row lookup.   dE[idx] += g"""
    out = Node(list(E.val[idx]))
    def back(g):
        if E.grad is None: E.zero()
        row = E.grad[idx]
        for i in range(len(g)): row[i] += g[i]
    return rec_op(out, back)

def n_matvec(Wn, xn):
    """y = Wx.    dW += g x^T (outer product),   dx += W^T g"""
    out = Node(matvec(Wn.val, xn.val))
    def back(g):
        xv = xn.val
        acc(Wn, [[g[i]*xv[j] for j in range(len(xv))] for i in range(len(g))])
        Wv = Wn.val
        acc(xn, [vsum([Wv[i][j]*g[i] for i in range(len(g))]) for j in range(len(xv))])
    return rec_op(out, back)

def n_add(a, b):
    """y = a + b.   da += g,  db += g"""
    out = Node(vadd(a.val, b.val))
    def back(g): acc(a, g); acc(b, g)
    return rec_op(out, back)

def n_mul(a, b):
    """y = a (*) b.   da += g (*) b,   db += g (*) a"""
    out = Node(vmul(a.val, b.val))
    def back(g): acc(a, vmul(g, b.val)); acc(b, vmul(g, a.val))
    return rec_op(out, back)

def n_scale(c, a):
    """y = c a, c a CONSTANT (this is how alpha enters).   da += c g"""
    out = Node(vscale(c, a.val))
    def back(g): acc(a, vscale(c, g))
    return rec_op(out, back)

def n_cat(a, b):
    out = Node(concat(a.val, b.val))
    na = len(a.val)
    def back(g): acc(a, g[:na]); acc(b, g[na:])
    return rec_op(out, back)

def n_slice(a, i, j):
    out = Node(a.val[i:j])
    n = len(a.val)
    def back(g): acc(a, [0.0]*i + list(g) + [0.0]*(n-j))
    return rec_op(out, back)

def n_rmsnorm(xn, gn, eps):
    """y_i = gain_i * x_i * r,   r = (mean(x^2)+eps)^(-1/2)

    dL/dx_i = r*gain_i*g_i - (r^3 x_i / n) * S,   S = sum_j g_j gain_j x_j
    dL/dgain_i = g_i * x_i * r
    (the second term of dL/dx is the dependence of r itself on x -- the part
     that is easy to forget, and the reason we finite-difference check it)
    """
    x, gain, n = xn.val, gn.val, len(xn.val)
    ms = vsum([v*v for v in x]) / n
    r = 1.0 / math.sqrt(ms + eps)
    out = Node([gain[i]*x[i]*r for i in range(n)])
    def back(g):
        S = vsum([g[i]*gain[i]*x[i] for i in range(n)])
        acc(xn, [r*gain[i]*g[i] - (r**3)*x[i]*S/n for i in range(n)])
        acc(gn, [g[i]*x[i]*r for i in range(n)])
    return rec_op(out, back)

def n_sigmoid(xn):
    """y = sigma(x).   dx += g * y * (1-y)"""
    y = [sigmoid(v) for v in xn.val]
    out = Node(y)
    def back(g): acc(xn, [g[i]*y[i]*(1.0-y[i]) for i in range(len(y))])
    return rec_op(out, back)

_SQRT2PI = math.sqrt(2.0*math.pi)
def n_gelu(xn):
    """y = x*Phi(x).   dy/dx = Phi(x) + x*phi(x),  phi(x)=exp(-x^2/2)/sqrt(2pi)"""
    x = xn.val
    out = Node([gelu_exact(v) for v in x])
    def back(g):
        acc(xn, [g[i]*(0.5*(1.0+math.erf(x[i]/math.sqrt(2.0)))
                       + x[i]*math.exp(-0.5*x[i]*x[i])/_SQRT2PI) for i in range(len(x))])
    return rec_op(out, back)

def n_scores(qn, key_nodes, scale):
    """score_j = scale * <q, k_j>.
       dq += scale * sum_j g_j k_j ;   dk_j += scale * g_j * q"""
    out = Node([scale*dot(qn.val, k.val) for k in key_nodes])
    def back(g):
        dq = vzeros(len(qn.val))
        for j, k in enumerate(key_nodes):
            dq = vadd(dq, vscale(scale*g[j], k.val))
            acc(k, vscale(scale*g[j], qn.val))
        acc(qn, dq)
    return rec_op(out, back)

def n_softmax(zn):
    """p = softmax(z).   dz_i = p_i (g_i - sum_j g_j p_j)"""
    p = softmax(zn.val)
    out = Node(p)
    def back(g):
        s = vsum([g[j]*p[j] for j in range(len(p))])
        acc(zn, [p[i]*(g[i]-s) for i in range(len(p))])
    return rec_op(out, back)

def n_weighted(pn, val_nodes):
    """out = sum_j p_j v_j.   dp_j = <g, v_j> ;   dv_j += p_j g"""
    dv = len(val_nodes[0].val)
    o = vzeros(dv)
    for j, v in enumerate(val_nodes): o = vadd(o, vscale(pn.val[j], v.val))
    out = Node(o)
    def back(g):
        acc(pn, [dot(g, v.val) for v in val_nodes])
        for j, v in enumerate(val_nodes): acc(v, vscale(pn.val[j], g))
    return rec_op(out, back)

def n_cross_entropy(logits_n, target):
    """L = -log softmax(logits)[target].   dlogits = p - onehot(target)"""
    p = softmax(logits_n.val)
    out = Node([-math.log(max(p[target], 1e-300))])
    def back(g):
        d = list(p); d[target] -= 1.0
        acc(logits_n, vscale(g[0], d))
    return rec_op(out, back)


# =============================================================================
# 10.2  GRADIENT CHECK -- every VJP against central finite differences
# =============================================================================
def fd_check(name, build, inputs, h=1e-6):
    """build(inputs) -> scalar Node. Compare analytic grads to (f(x+h)-f(x-h))/2h."""
    tape_reset()
    for n in inputs: n.grad = None
    loss = build(inputs)
    backward(loss)
    worst, worst_where = 0.0, ""
    for ni, node in enumerate(inputs):
        flat_idx = ([(i, j) for i in range(len(node.val)) for j in range(len(node.val[i]))]
                    if isinstance(node.val[0], list) else [(i,) for i in range(len(node.val))])
        for idx in flat_idx:
            def peek(delta):
                if len(idx) == 2: node.val[idx[0]][idx[1]] += delta
                else:             node.val[idx[0]] += delta
                tape_reset()
                v = build(inputs).val[0]
                if len(idx) == 2: node.val[idx[0]][idx[1]] -= delta
                else:             node.val[idx[0]] -= delta
                return v
            num = (peek(h) - peek(-h)) / (2*h)
            ana = node.grad[idx[0]][idx[1]] if len(idx) == 2 else node.grad[idx[0]]
            rel = abs(num-ana) / max(1.0, abs(num), abs(ana))
            if rel > worst: worst, worst_where = rel, f"{ni}{idx}"
    status = "ok  " if worst < 2e-5 else "FAIL"
    print(f"  {status} {name:<34} worst rel err = {worst:.3e}  at input{worst_where}")
    return worst < 2e-5

g = LCG(11)
def mk(*shape): return Node(randn_matrix(*shape, g) if len(shape) == 2 else randn_vector(shape[0], g))

print("finite-difference check of every hand-written VJP (h=1e-6):\n")
allg = True
W1_, x1_ = mk(6, 5), mk(5)
allg &= fd_check("n_matvec",   lambda I: n_matvec(I[0], I[1]), [W1_, x1_])
a_, b_ = mk(5), mk(5)
allg &= fd_check("n_add",      lambda I: n_matvec(Node([[1.0]*5]), n_add(I[0], I[1])), [a_, b_])
allg &= fd_check("n_mul",      lambda I: n_matvec(Node([[1.0]*5]), n_mul(I[0], I[1])), [a_, b_])
allg &= fd_check("n_scale",    lambda I: n_matvec(Node([[1.0]*5]), n_scale(0.37, I[0])), [a_])
allg &= fd_check("n_cat",      lambda I: n_matvec(Node([[1.0]*10]), n_cat(I[0], I[1])), [a_, b_])
allg &= fd_check("n_slice",    lambda I: n_matvec(Node([[1.0]*3]), n_slice(I[0], 1, 4)), [a_])
gain_ = Node([1.0, 0.7, 1.3, 0.9, 1.1])
allg &= fd_check("n_rmsnorm (x and gain)",
                 lambda I: n_matvec(Node([[1.0, -2.0, 0.5, 3.0, -1.0]]), n_rmsnorm(I[0], I[1], 1e-5)),
                 [a_, gain_])
allg &= fd_check("n_sigmoid",  lambda I: n_matvec(Node([[1.0]*5]), n_sigmoid(I[0])), [a_])
allg &= fd_check("n_gelu",     lambda I: n_matvec(Node([[1.0]*5]), n_gelu(I[0])), [a_])
q_, k1_, k2_, k3_ = mk(4), mk(4), mk(4), mk(4)
allg &= fd_check("n_scores",   lambda I: n_matvec(Node([[1.0]*3]),
                 n_scores(I[0], [I[1], I[2], I[3]], 0.5)), [q_, k1_, k2_, k3_])
z_ = mk(4)
allg &= fd_check("n_softmax",  lambda I: n_matvec(Node([[1.0, -2.0, 0.3, 0.7]]), n_softmax(I[0])), [z_])
p_, v1_, v2_ = Node(softmax(randn_vector(2, g))), mk(4), mk(4)
allg &= fd_check("n_weighted", lambda I: n_matvec(Node([[1.0, 2.0, -1.0, 0.5]]),
                 n_weighted(I[0], [I[1], I[2]])), [p_, v1_, v2_])
lg_ = mk(7)
allg &= fd_check("n_cross_entropy", lambda I: n_cross_entropy(I[0], 3), [lg_])
# full attention chain end-to-end
allg &= fd_check("attention chain (scores->softmax->weighted)",
                 lambda I: n_matvec(Node([[1.0, -1.0, 0.5, 2.0]]),
                     n_weighted(n_softmax(n_scores(I[0], [I[1], I[2]], 0.5)), [I[3], I[4]])),
                 [q_, k1_, k2_, v1_, v2_])

print(f"\n  ==> all VJPs correct: {allg}")
tape_reset()

finite-difference check of every hand-written VJP (h=1e-6):

  ok   n_matvec                           worst rel err = 1.081e-10  at input0(0, 1)
  ok   n_add                              worst rel err = 1.398e-10  at input0(0,)
  ok   n_mul                              worst rel err = 3.507e-10  at input0(0,)
  ok   n_scale                            worst rel err = 2.822e-11  at input0(0,)
  ok   n_cat                              worst rel err = 3.043e-10  at input0(2,)
  ok   n_slice                            worst rel err = 8.227e-11  at input0(1,)
  ok   n_rmsnorm (x and gain)             worst rel err = 1.886e-10  at input0(1,)
  ok   n_sigmoid                          worst rel err = 1.718e-10  at input0(3,)
  ok   n_gelu                             worst rel err = 1.643e-11  at input0(2,)
  ok   n_scores                           worst rel err = 1.395e-10  at input0(1,)
  ok   n_softmax                          worst rel err = 6.975e-11  at input0(1,)
  ok   n_weighted       

In [22]:
# =============================================================================
# 10.3  THE TAPED FORWARD  (same model, now differentiable)
# =============================================================================

def make_param_nodes(P):
    return {k: Node([list(r) for r in v] if isinstance(v[0], list) else list(v), k)
            for k, v in P.items()}

def n_detach(node):
    """stopgrad: identical value, no tape edge.  This is eq (C.1)."""
    return Node(list(node.val))

def n_concat_heads(hs):
    o = hs[0]
    for x in hs[1:]: o = n_cat(o, x)
    return o

def n_split(node, H, dh):
    return [n_slice(node, h*dh, (h+1)*dh) for h in range(H)]

def n_mha(qh, k_entries, v_entries, H, scale):
    outs = []
    for h in range(H):
        sc = n_scores(qh[h], [k[h] for k in k_entries], scale)
        pr = n_softmax(sc)
        outs.append(n_weighted(pr, [v[h] for v in v_entries]))
    return n_concat_heads(outs)

def n_ffn(PN, x, pre, w1, w2, eps):
    return n_matvec(PN[w2], n_gelu(n_matvec(PN[w1], n_rmsnorm(x, PN[pre], eps))))


def rlt_forward_ad(PN, cfg, tokens, targets=None, opts=None):
    """Differentiable twin of rlt_forward. Must give bit-identical values.

    opts:  alpha, detach_s, detach_kv  -- the last two are the TBPTT knobs of
           Appendix C, and they change ONLY the backward graph.
    """
    opts = opts or {}
    assert cfg["pos"] == "nope", "taped forward implements NoPE (0.6, B1)"
    alpha = opts.get("alpha", cfg["alpha"])
    det_s, det_kv = opts.get("detach_s", False), opts.get("detach_kv", False)
    H, dh, eps, W = cfg["H"], cfg["d_head"], cfg["eps"], cfg["W"]
    scale, T = 1.0/math.sqrt(dh), len(tokens)

    # ---- encoder, causal, parallel schedule -------------------------------
    h = [n_embed(PN["E_tok"], x) for x in tokens]
    for l in range(cfg["L_E"]):
        nq = [n_rmsnorm(h[i], PN[f"enc.{l}.n_attn"], eps) for i in range(T)]
        Qh = [n_split(n_matvec(PN[f"enc.{l}.WQ"], nq[i]), H, dh) for i in range(T)]
        Kh = [n_split(n_matvec(PN[f"enc.{l}.WK"], nq[i]), H, dh) for i in range(T)]
        Vh = [n_split(n_matvec(PN[f"enc.{l}.WV"], nq[i]), H, dh) for i in range(T)]
        new = []
        for i in range(T):
            o = n_mha(Qh[i], Kh[:i+1], Vh[:i+1], H, scale)
            b = n_add(h[i], n_matvec(PN[f"enc.{l}.WO"], o))
            new.append(n_add(b, n_ffn(PN, b, f"enc.{l}.n_ffn", f"enc.{l}.W1", f"enc.{l}.W2", eps)))
        h = new
    E = h

    # ---- memory (eq 2.2) ---------------------------------------------------
    M = [[] for _ in range(cfg["G"])]
    for i in range(T):
        for gi in range(cfg["G"]):
            n = n_rmsnorm(E[i], PN[f"mem.{gi}.n_E"], eps)
            M[gi].append((i+1, n_split(n_matvec(PN[f"mem.{gi}.WK"], n), H, dh),
                                n_split(n_matvec(PN[f"mem.{gi}.WV"], n), H, dh)))

    # ---- the recurrence ----------------------------------------------------
    C_D = [[] for _ in range(cfg["L_D"])]
    s = PN["s_star"]
    losses, states, logits_all = [], [], []
    for i in range(T):
        t = i + 1
        s_in = n_detach(s) if det_s else s                       # App. C truncation
        r = n_rmsnorm(s_in, PN["merge.n_s"], eps)                          # (2.9)
        gate = n_sigmoid(n_add(n_matvec(PN["merge.Wg"], n_cat(E[i], r)), PN["merge.bg"]))  # (2.10)
        u = n_add(E[i], n_scale(alpha, n_mul(gate, n_matvec(PN["merge.Ws"], r))))          # (2.11)

        z = u
        for l in range(cfg["L_D"]):
            q = n_split(n_matvec(PN[f"dec.{l}.WQ"], n_rmsnorm(z, PN[f"dec.{l}.n_q"], eps)), H, dh)
            k = n_split(n_matvec(PN[f"dec.{l}.WK"], n_rmsnorm(z, PN[f"dec.{l}.n_k"], eps)), H, dh)
            v = n_split(n_matvec(PN[f"dec.{l}.WV"], n_rmsnorm(z, PN[f"dec.{l}.n_S"], eps)), H, dh)
            win = list(C_D[l]) + [(t, k, v)]
            o = n_mha(q, [e[1] for e in win], [e[2] for e in win], H, scale)      # (2.14)
            b = n_add(z, n_matvec(PN[f"dec.{l}.WO"], o))
            gsel = l % cfg["G"]
            mem = [e for e in M[gsel] if e[0] <= t]
            qm = n_split(n_matvec(PN[f"dec.{l}.WQM"], n_rmsnorm(b, PN[f"dec.{l}.n_m"], eps)), H, dh)
            om = n_mha(qm, [e[1] for e in mem], [e[2] for e in mem], H, scale)    # (2.15)
            a = n_add(b, n_matvec(PN[f"dec.{l}.WOM"], om))
            z = n_add(a, n_ffn(PN, a, f"dec.{l}.n_ffn", f"dec.{l}.W1", f"dec.{l}.W2", eps))  # (2.16)
            keep = win[-(W-1):] if W > 1 else []
            if det_kv:                                            # detach only the KV path
                keep = [(e[0], [n_detach(x) for x in e[1]], [n_detach(x) for x in e[2]]) for e in keep]
            C_D[l] = keep
        s = z
        states.append(s)
        lg = n_matvec(PN["out.WO"], n_rmsnorm(s, PN["out.n"], eps))                  # (2.6)
        logits_all.append(lg)
        if targets is not None:
            losses.append(n_cross_entropy(lg, targets[i]))

    total = None
    if targets is not None:
        total = losses[0]
        for L in losses[1:]: total = n_add(total, L)
        total = n_scale(1.0/len(losses), total)
    return dict(loss=total, states=states, logits=logits_all, per_tok=losses)


# ---- the twin must agree with the plain forward, bit for bit ---------------
tape_reset()
PN = make_param_nodes(P)
seq_chk = [0, 3, 7, 2, 5, 1]
Aad = rlt_forward_ad(PN, cfg, seq_chk)
Apl = rlt_forward(P, cfg, seq_chk)
ok, worst, _ = allclose([n.val for n in Aad["states"]], Apl["s"], atol=0.0, rtol=0.0)
print(f"taped forward == plain forward ?  {ok}   max|diff| = {worst:.3e}")
ok2, w2, _ = allclose([n.val for n in Aad["logits"]], Apl["logits"], atol=0.0, rtol=0.0)
print(f"logits identical ?                {ok2}   max|diff| = {w2:.3e}")
print(f"tape length for T={len(seq_chk)}: {len(TAPE)} nodes "
      f"({len(TAPE)/len(seq_chk):.0f} per token)")

# ---- and its gradients must match finite differences on the REAL model ----
print("\nfinite-difference check of the FULL model gradient (a few parameters):")
def model_loss(Pd, cfgd, toks, tgts, opts=None):
    tape_reset()
    PNd = make_param_nodes(Pd)
    return rlt_forward_ad(PNd, cfgd, toks, tgts, opts)["loss"].val[0], PNd

toks_g, tgts_g = [0, 1, 2, 1], [3, 4, 3, 4]
tape_reset(); PNg = make_param_nodes(P)
Lg = rlt_forward_ad(PNg, cfg, toks_g, tgts_g)["loss"]
backward(Lg)
hfd, worst_fd = 1e-6, 0.0
for pname, idx in [("merge.Ws", (0, 0)), ("merge.Wg", (2, 5)), ("s_star", (3,)),
                   ("dec.0.WQ", (1, 4)), ("enc.0.W1", (7, 2)), ("out.WO", (4, 6)),
                   ("merge.n_s", (5,)), ("E_tok", (1, 3)), ("mem.0.WV", (6, 1))]:
    P2 = clone_params(P)
    tgt = P2[pname]
    if len(idx) == 2: tgt[idx[0]][idx[1]] += hfd
    else:             tgt[idx[0]] += hfd
    lp, _ = model_loss(P2, cfg, toks_g, tgts_g)
    if len(idx) == 2: tgt[idx[0]][idx[1]] -= 2*hfd
    else:             tgt[idx[0]] -= 2*hfd
    lm, _ = model_loss(P2, cfg, toks_g, tgts_g)
    num = (lp - lm) / (2*hfd)
    ana = PNg[pname].grad[idx[0]][idx[1]] if len(idx) == 2 else PNg[pname].grad[idx[0]]
    rel = abs(num-ana)/max(1e-8, abs(num), abs(ana))
    worst_fd = max(worst_fd, rel)
    print(f"  {pname:<12}{str(idx):<8} numeric={num:+.9f}  analytic={ana:+.9f}  rel={rel:.2e}")
print(f"\n  ==> full-model BPTT gradient verified, worst rel err = {worst_fd:.3e}")
tape_reset()

taped forward == plain forward ?  False   max|diff| = 8.882e-16
logits identical ?                False   max|diff| = 1.665e-16
tape length for T=6: 492 nodes (82 per token)

finite-difference check of the FULL model gradient (a few parameters):
  merge.Ws    (0, 0)   numeric=-0.002053948  analytic=-0.002053948  rel=4.74e-08
  merge.Wg    (2, 5)   numeric=-0.002138073  analytic=-0.002138073  rel=5.84e-08
  s_star      (3,)     numeric=-0.000977957  analytic=-0.000977957  rel=2.40e-07
  dec.0.WQ    (1, 4)   numeric=-0.015644614  analytic=-0.015644614  rel=3.77e-09
  enc.0.W1    (7, 2)   numeric=+0.073038515  analytic=+0.073038515  rel=5.40e-10
  out.WO      (4, 6)   numeric=-0.224867457  analytic=-0.224867457  rel=8.87e-10
  merge.n_s   (5,)     numeric=-0.000085875  analytic=-0.000085875  rel=8.71e-07
  E_tok       (1, 3)   numeric=+0.271658176  analytic=+0.271658176  rel=1.02e-09
  mem.0.WV    (6, 1)   numeric=+0.001528109  analytic=+0.001528109  rel=5.87e-09

  ==> full-model BPTT gr

# §10 — Phase 10/11: training a tiny RLT, and watching which parameters move

**Task (ours, not the paper's).** *Running parity*: input is `[BOS, b_1 … b_n]` with bits as tokens 1/2; the target at every position $t$ is token 3 (even) or 4 (odd) for the parity of the bits seen so far. Chance is 50%.

Parity is chosen because it **requires accumulating state across tokens** and is famously hard for attention to shortcut — so it is a task where a recurrent channel could actually pay for itself. Whether it does is the measurement, not the assumption.

**One thing to note first.** The taped and plain forwards agree to $8.9\times10^{-16}$, not bit-exactly. The cause is real and worth naming: `rmsnorm` divides by $\sqrt{\text{ms}+\epsilon}$ while `n_rmsnorm` multiplies by its reciprocal. Same mathematics, different rounding. This is precisely the caveat the paper attaches to Proposition 3.1 — *"Different kernels and precision choices can still cause numerical discrepancies"* — reproduced accidentally, in pure Python, at 4 ulp.

In [23]:
# =============================================================================
# 10.4  THE PARITY TASK + MANUAL ADAM
# =============================================================================
import time

BOS, BIT0, BIT1, EVEN, ODD = 0, 1, 2, 3, 4

def parity_example(rng, n_bits):
    """[BOS, b1..bn] -> targets = running parity at every position."""
    bits = [1 if rng.uniform() < 0.5 else 0 for _ in range(n_bits)]
    toks = [BOS] + [BIT1 if b else BIT0 for b in bits]
    tgts, par = [EVEN], 0                       # position 1 is BOS: parity of {} = even
    for b in bits:
        par ^= b
        tgts.append(ODD if par else EVEN)
    return toks, tgts

def make_data(n, n_bits, seed):
    r = LCG(seed)
    return [parity_example(r, n_bits) for _ in range(n)]

N_BITS = 8
train_data = make_data(256, N_BITS, seed=1)
test_data  = make_data(128, N_BITS, seed=99)
ex = train_data[0]
print(f"example: tokens {ex[0]}\n         targets{ex[1]}   (3=even, 4=odd)")
print(f"train {len(train_data)} / test {len(test_data)} sequences, T={N_BITS+1}, chance = 50%\n")


class Adam:
    """m_t = b1 m + (1-b1) g ;  v_t = b2 v + (1-b2) g^2
       update = lr * m_hat / (sqrt(v_hat) + eps),  with bias correction."""
    def __init__(self, P, lr=0.02, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
        self.m = {k: ([[0.0]*len(r) for r in v] if isinstance(v[0], list) else [0.0]*len(v))
                  for k, v in P.items()}
        self.v = {k: ([[0.0]*len(r) for r in v] if isinstance(v[0], list) else [0.0]*len(v))
                  for k, v in P.items()}
    def step(self, P, G):
        self.t += 1
        bc1 = 1.0 - self.b1**self.t
        bc2 = 1.0 - self.b2**self.t
        for k in P:
            p, g, m, v = P[k], G[k], self.m[k], self.v[k]
            if isinstance(p[0], list):
                for i in range(len(p)):
                    for j in range(len(p[i])):
                        m[i][j] = self.b1*m[i][j] + (1-self.b1)*g[i][j]
                        v[i][j] = self.b2*v[i][j] + (1-self.b2)*g[i][j]*g[i][j]
                        p[i][j] -= self.lr*(m[i][j]/bc1)/(math.sqrt(v[i][j]/bc2)+self.eps)
            else:
                for i in range(len(p)):
                    m[i] = self.b1*m[i] + (1-self.b1)*g[i]
                    v[i] = self.b2*v[i] + (1-self.b2)*g[i]*g[i]
                    p[i] -= self.lr*(m[i]/bc1)/(math.sqrt(v[i]/bc2)+self.eps)


def zero_grads(P):
    return {k: ([[0.0]*len(r) for r in v] if isinstance(v[0], list) else [0.0]*len(v))
            for k, v in P.items()}

def accumulate(G, PN):
    for k, node in PN.items():
        if node.grad is None: continue
        g, ng = G[k], node.grad
        if isinstance(g[0], list):
            for i in range(len(g)):
                for j in range(len(g[i])): g[i][j] += ng[i][j]
        else:
            for i in range(len(g)): g[i] += ng[i]

def evaluate(P, cfg, data, opts=None):
    tot, corr, n = 0.0, 0, 0
    for toks, tgts in data:
        R = rlt_forward(P, cfg, toks, opts=opts or {})
        for i in range(len(toks)):
            p = softmax(R["logits"][i])
            tot += -math.log(max(p[tgts[i]], 1e-300))
            corr += 1 if p.index(vmax(p)) == tgts[i] else 0
            n += 1
    return tot/n, corr/n

def train(cfg_, opts=None, steps=150, bs=16, lr=0.02, seed=0, log_every=25,
          track_params=False, data=None, label=""):
    opts = opts or {}
    Pt = init_params(make_cfg(seed=seed, **{k: v for k, v in cfg_.items()
                                            if k in ("V","d","L_E","L_D","H","d_ff","W","G","alpha","eps","pos","tied")}))
    P0 = clone_params(Pt)
    opt = Adam(Pt, lr=lr)
    dat = data or train_data
    hist, pstats = [], []
    r = LCG(seed + 555)
    t0 = time.time()
    for st in range(steps):
        G = zero_grads(Pt)
        bl = 0.0
        for _ in range(bs):
            toks, tgts = dat[int(r.uniform()*len(dat)) % len(dat)]
            tape_reset()
            PN = make_param_nodes(Pt)
            out = rlt_forward_ad(PN, cfg_, toks, tgts, opts)
            backward(out["loss"])
            accumulate(G, PN)
            bl += out["loss"].val[0]
        for k in G:                                   # mean over the batch
            gk = G[k]
            if isinstance(gk[0], list):
                for i in range(len(gk)):
                    for j in range(len(gk[i])): gk[i][j] /= bs
            else:
                for i in range(len(gk)): gk[i] /= bs
        gnorm = math.sqrt(sum(x*x for k in G for x in _flatten(G[k])))
        opt.step(Pt, G)
        hist.append(dict(step=st, loss=bl/bs, gnorm=gnorm))
        if track_params:
            pstats.append({k: l2(_flatten([[Pt[k][i][j]-P0[k][i][j] for j in range(len(Pt[k][i]))]
                                           for i in range(len(Pt[k]))] if isinstance(Pt[k][0], list)
                                          else [Pt[k][i]-P0[k][i] for i in range(len(Pt[k]))]))
                           for k in Pt})
        if log_every and (st+1) % log_every == 0:
            te_l, te_a = evaluate(Pt, cfg_, test_data[:48], opts)
            print(f"    step {st+1:>4}  train loss {bl/bs:.4f}  |g| {gnorm:8.4f}  "
                  f"test loss {te_l:.4f}  test acc {te_a:6.2%}  [{time.time()-t0:5.1f}s]")
    return Pt, P0, hist, pstats

print("timing probe: one step of batch 4 ...")
_t = time.time()
_ = train(cfg, steps=1, bs=4, log_every=0)
print(f"  {time.time()-_t:.2f}s for 1 step x 4 examples "
      f"-> ~{(time.time()-_t)/4*16:.1f}s per step at batch 16\n")

print("TRAINING: full RLT (alpha=0.1, W=3)")
P_trained, P_init, hist, pstats = train(cfg, steps=150, bs=16, lr=0.02, seed=0,
                                        log_every=25, track_params=True)
tr_l, tr_a = evaluate(P_trained, cfg, train_data[:64])
te_l, te_a = evaluate(P_trained, cfg, test_data)
print(f"\n  FINAL   train loss {tr_l:.4f} acc {tr_a:.2%}   test loss {te_l:.4f} acc {te_a:.2%}")
print()
ascii_plot([h["loss"] for h in hist][::5], "training loss (every 5th step)", height=9)

example: tokens [0, 2, 2, 1, 1, 2, 2, 1, 1]
         targets[3, 4, 3, 3, 3, 4, 3, 3, 3]   (3=even, 4=odd)
train 256 / test 128 sequences, T=9, chance = 50%

timing probe: one step of batch 4 ...
  0.08s for 1 step x 4 examples -> ~0.3s per step at batch 16

TRAINING: full RLT (alpha=0.1, W=3)
    step   25  train loss 0.5975  |g|   0.3637  test loss 0.6120  test acc 64.81%  [  4.0s]
    step   50  train loss 0.5185  |g|   0.4955  test loss 0.5323  test acc 72.69%  [  8.0s]
    step   75  train loss 0.3820  |g|   0.2206  test loss 0.4242  test acc 77.08%  [ 12.1s]
    step  100  train loss 0.3469  |g|   0.2203  test loss 0.3223  test acc 83.56%  [ 16.1s]
    step  125  train loss 0.2546  |g|  10.2731  test loss 0.8757  test acc 76.39%  [ 20.1s]
    step  150  train loss 0.1800  |g|   0.3582  test loss 0.2505  test acc 87.50%  [ 24.2s]

  FINAL   train loss 0.2139 acc 89.58%   test loss 0.2557 acc 87.24%

  training loss (every 5th step)   [0.2180 .. 1.5695]
      +1.570 |*              

In [25]:
# =============================================================================
# 11  PARAMETER DISSECTION -- which parameters actually learned?
# =============================================================================
# NOTE ON THE METRIC. The first draft ranked by ||W_t - W_0|| / ||W_0|| and put
# merge.bg on top at 1.0e14. That is not a result, it is a divide-by-zero:
# b_g is initialized to EXACTLY 0 (it is the only zero-initialized tensor), so
# the denominator was the 1e-12 floor. Relative change is undefined for a
# zero-initialized parameter. Reported as absolute movement instead, and
# ranked separately, rather than left in the table looking like a finding.

for toks, tgts in train_data[:1]:
    tape_reset(); PNf = make_param_nodes(P_trained)
    o = rlt_forward_ad(PNf, cfg, toks, tgts)
    backward(o["loss"])

rows11 = []
for k in sorted(P_trained):
    w0, w1 = _flatten(P_init[k]), _flatten(P_trained[k])
    d = [w1[i]-w0[i] for i in range(len(w0))]
    gnorm = l2(_flatten(PNf[k].grad)) if PNf[k].grad is not None else 0.0
    n0 = l2(w0)
    rows11.append(dict(name=k, n=len(w0), n0=n0, n1=l2(w1), dn=l2(d),
                       rel=(l2(d)/n0 if n0 > 1e-12 else None), g=gnorm,
                       rms0=rms(w0), rms1=rms(w1)))

defined = [r for r in rows11 if r["rel"] is not None]
undef   = [r for r in rows11 if r["rel"] is None]
defined.sort(key=lambda r: -r["rel"])

print("  parameters ranked by RELATIVE movement  ||W_t - W_0|| / ||W_0||\n")
print(f"  {'parameter':<16}{'n':>5}{'||W_0||':>10}{'||W_t||':>10}{'||dW||':>10}"
      f"{'rel change':>12}{'|grad|':>10}{'rms_0':>8}{'rms_t':>8}")
print("  " + "-"*89)
for r in defined:
    print(f"  {r['name']:<16}{r['n']:>5}{r['n0']:>10.4f}{r['n1']:>10.4f}{r['dn']:>10.4f}"
          f"{r['rel']:>12.4f}{r['g']:>10.4f}{r['rms0']:>8.4f}{r['rms1']:>8.4f}")
for r in undef:
    print(f"  {r['name']:<16}{r['n']:>5}{r['n0']:>10.4f}{r['n1']:>10.4f}{r['dn']:>10.4f}"
          f"{'n/a (W_0=0)':>12}{r['g']:>10.4f}{r['rms0']:>8.4f}{r['rms1']:>8.4f}")

print(f"""
  The merge parameters -- the ONLY parameters that exist because of the
  recurrence -- move as follows:""")
for r in rows11:
    if r["name"].startswith("merge") or r["name"] == "s_star":
        rel = f"{r['rel']:.4f}" if r["rel"] is not None else "n/a (zero init)"
        print(f"    {r['name']:<14} rel change {rel:<16} ||dW|| {r['dn']:.4f}   |grad| {r['g']:.4f}")

top = defined[0]
print(f"""
  Largest well-defined mover: {top['name']} at {top['rel']:.1%}.

  Two things worth reading off this table:

  1. merge.Wg and merge.Ws are the 2nd and 3rd largest movers in the whole
     model. The gate and the state projection are NOT dead weight -- training
     pushes them hard. (Whether that MOVEMENT buys accuracy is a different
     question, answered in section 12 by ablation, not by this table.)

  2. Relative movement and gradient norm rank DIFFERENTLY: out.WO has the
     largest gradient ({[r['g'] for r in defined if r['name']=='out.WO'][0]:.4f}) but only the 6th largest movement, while
     merge.Wg has a gradient of {[r['g'] for r in defined if r['name']=='merge.Wg'][0]:.4f} and the 2nd largest movement.
     Adam divides by the running second moment, so a persistently small
     gradient still travels far. Ranking 'what learned' by gradient magnitude
     under an adaptive optimizer is simply the wrong statistic.""")

# ------------------------------------------------- movement over time --------
if pstats:
    print("\n  relative movement over training:")
    watch = ["merge.Ws", "merge.Wg", "s_star", "out.WO", "dec.0.WQ", "E_tok"]
    print(f"    {'step':>6}" + "".join(f"{w:>13}" for w in watch))
    for st in [0, 24, 49, 74, 99, 124, 149]:
        if st < len(pstats):
            print(f"    {st+1:>6}" + "".join(
                f"{pstats[st][w]/max(l2(_flatten(P_init[w])),1e-12):>13.4f}" for w in watch))
    print()
    ascii_plot([pstats[i]["merge.Ws"]/max(l2(_flatten(P_init["merge.Ws"])),1e-12)
                for i in range(0, len(pstats), 5)],
               "relative movement of merge.Ws (the state projection)", height=7)

  parameters ranked by RELATIVE movement  ||W_t - W_0|| / ||W_0||

  parameter           n   ||W_0||   ||W_t||    ||dW||  rel change    |grad|   rms_0   rms_t
  -----------------------------------------------------------------------------------------
  merge.Wg          128    2.7217    5.2548    4.9244      1.8093    0.0013  0.2406  0.4645
  merge.Ws           64    2.7923    4.6639    3.7482      1.3423    0.0732  0.3490  0.5830
  dec.0.WK           64    2.9531    4.9620    3.9435      1.3354    0.0149  0.3691  0.6202
  dec.0.WQM          64    2.8997    4.0494    2.9014      1.0006    0.0384  0.3625  0.5062
  out.WO             80    3.3519    5.2731    3.3126      0.9883    1.3068  0.3748  0.5896
  dec.0.WQ           64    2.9588    4.4108    2.9049      0.9818    0.0247  0.3699  0.5513
  dec.0.W1          128    4.2275    5.5269    3.6971      0.8745    0.2565  0.3737  0.4885
  enc.0.W1          128    3.7732    5.2364    3.2641      0.8651    0.6217  0.3335  0.4628
  dec.0.W2   

# §11 — Phase 12: gradient dissection, BPTT, and the Jacobian claim

The paper's sharpest unproven statement, Appendix B:

> *"A product involving only $\partial s_t/\partial s_{t-1}$ generally **misses paths through decoder KV**. Normalization alone does not bound products of these Jacobians."*

**Design.** The tape gives the *true* total derivative $\partial s_t / \partial s_j$ — every path, both channels. Separately we compute each single-step $\partial s_k/\partial s_{k-1}$ and multiply them, which is the naive chain the paper warns about. If the paper is right, the two differ, and the gap **is** the KV contribution.

$$J_t = \frac{\partial H_t}{\partial H_{t-1}},\qquad \frac{\partial H_t}{\partial H_j} = J_t J_{t-1}\cdots J_{j+1}$$

In [27]:
# =============================================================================
# 12.1  JACOBIANS -- true total derivative vs the s-only product (claim C9)
# =============================================================================

def backward_from(seed_node, seed_vec, PNd):
    """Seed an arbitrary node and propagate. Clears all grads first so the
    result is a pure VJP from that node, not a mixture with an earlier pass."""
    for out, _ in TAPE: out.grad = None
    for n in PNd.values(): n.grad = None
    acc(seed_node, seed_vec)
    for out, back in reversed(TAPE):
        if out.grad is not None:
            back(out.grad)

def jacobian(out_node, in_node, PNd, d):
    """J[i][j] = d out_i / d in_j, by seeding each unit vector at the output."""
    J = []
    for i in range(d):
        e = [0.0]*d; e[i] = 1.0
        backward_from(out_node, e, PNd)
        J.append(list(in_node.grad) if in_node.grad is not None else [0.0]*d)
    return J

def spectral_norm(A, iters=100):
    """Largest singular value by power iteration on A^T A. Pure Python."""
    n = len(A[0])
    v = [1.0/math.sqrt(n)]*n
    for _ in range(iters):
        Av = matvec(A, v)
        ATAv = matvec(transpose(A), Av)
        nv = l2(ATAv)
        if nv < 1e-300: return 0.0
        v = vscale(1.0/nv, ATAv)
    return l2(matvec(A, v))

def fro(A): return l2(_flatten(A))

d = cfg["d"]
SEQJ, TGTJ = train_data[3]
TJ = len(SEQJ)

for tag, Puse in [("AT INITIALIZATION", P_init), ("AFTER TRAINING", P_trained)]:
    print("=" * 78)
    print(f"{tag}   (alpha={cfg['alpha']}, W={cfg['W']}, T={TJ})")
    print("=" * 78)
    tape_reset()
    PNj = make_param_nodes(Puse)
    O = rlt_forward_ad(PNj, cfg, SEQJ, TGTJ)
    S = O["states"]                      # S[i] is s_(i+1)

    Jstep = {}                           # dict, not list -- index IS the step k
    for k in range(2, TJ+1):
        Jstep[k] = jacobian(S[k-1], S[k-2], PNj, d)

    print(f"  single-step ||ds_k/ds_(k-1)||   (the channel-1 Jacobian alone)")
    print(f"    {'k':>4}{'spectral':>12}{'frobenius':>12}")
    for k in range(2, TJ+1):
        print(f"    {k:>4}{spectral_norm(Jstep[k]):>12.6f}{fro(Jstep[k]):>12.6f}")

    print(f"\n  TRUE ds_t/ds_j (all paths, from the tape)  vs  PRODUCT of single steps")
    print(f"    {'j':>3}{'t':>4}{'||true||_F':>14}{'||prod||_F':>14}{'||true-prod||_F':>17}"
          f"{'ratio':>12}")
    gaps = []
    for j in [1, 2, 3]:
        for t in sorted({j+1, j+2, j+3, min(j+5, TJ)}):
            if t > TJ or t <= j: continue
            Jtrue = jacobian(S[t-1], S[j-1], PNj, d)
            Jprod = Jstep[j+1]
            for k in range(j+2, t+1):
                Jprod = matmul(Jstep[k], Jprod)
            diff = [[Jtrue[a][b]-Jprod[a][b] for b in range(d)] for a in range(d)]
            ratio = fro(Jtrue)/max(fro(Jprod), 1e-300)
            gaps.append((j, t, fro(Jtrue), fro(Jprod), fro(diff), ratio))
            print(f"    {j:>3}{t:>4}{fro(Jtrue):>14.6e}{fro(Jprod):>14.6e}"
                  f"{fro(diff):>17.6e}{ratio:>12.2f}")

    one_step = [g for g in gaps if g[1] == g[0]+1]
    multi    = [g for g in gaps if g[1] > g[0]+1]
    print(f"\n  one-step (t=j+1): max ||true-prod||_F = {max(g[4] for g in one_step):.3e}"
          f"   <- must be ~0: the product IS the single step there")
    print(f"  multi-step      : mean ||true||/||prod|| = {mean([g[5] for g in multi]):.2f}"
          f"   max {max(g[5] for g in multi):.2f}")
    print()

print("""==============================================================================
RESULT -- claim C9
==============================================================================
  At t = j+1 the two agree to machine precision, as they must: with one decoder
  layer the cache entry written at step j is built from z^0 = u_j, which does
  not depend on s_j. So there the single-step Jacobian is already complete.

  For t > j+1 they DIVERGE and the true derivative is LARGER. The excess is the
  gradient flowing  s_j -> u_(j+1) -> KV written at j+1 -> read by SWA at j+2
  -> ... -> s_t.  That path is present in the true derivative and absent from
  the product of ds/ds factors.

  ==> CONFIRMED. A product of only ds_t/ds_(t-1) UNDERSTATES the true
      sensitivity. Any vanishing-gradient or stability argument built on that
      product alone is measuring the wrong object -- which is exactly what
      Appendix B says, and it is a large effect, not a rounding detail.""")

AT INITIALIZATION   (alpha=0.1, W=3, T=9)
  single-step ||ds_k/ds_(k-1)||   (the channel-1 Jacobian alone)
       k    spectral   frobenius
       2    0.032823    0.048807
       3    0.039798    0.065256
       4    0.043439    0.067681
       5    0.050866    0.077579
       6    0.051478    0.086694
       7    0.062379    0.095227
       8    0.060972    0.094532
       9    0.073564    0.103884

  TRUE ds_t/ds_j (all paths, from the tape)  vs  PRODUCT of single steps
      j   t    ||true||_F    ||prod||_F  ||true-prod||_F       ratio
      1   2  4.880661e-02  4.880661e-02     0.000000e+00        1.00
      1   3  1.205368e-02  1.044907e-03     1.172765e-02       11.54
      1   4  1.154773e-02  2.831885e-05     1.154431e-02      407.78
      1   6  2.296235e-04  2.953905e-08     2.296194e-04     7773.56
      2   3  6.525589e-02  6.525589e-02     0.000000e+00        1.00
      2   4  1.444270e-02  1.767531e-03     1.440984e-02        8.17
      2   5  1.700295e-02  6.021547e-05

In [28]:
# =============================================================================
# 12.2  TBPTT VARIANTS (App. C) -- claims C7 and C10
# =============================================================================
# App. C, eq (C.1):  H~_t = (stopgrad(s_t), stopgrad(C^D_t))
# "Detaching only s_t leaves possible paths through decoder KV; detaching only
#  decoder KV leaves paths through the recurrent output."
# Both are implemented as the detach_s / detach_kv flags in rlt_forward_ad.

variants12 = [("full BPTT (reference)", {}),
              ("detach s only",        {"detach_s": True}),
              ("detach KV only",       {"detach_kv": True}),
              ("detach BOTH  (C.1)",   {"detach_s": True, "detach_kv": True})]

print("  forward loss must be IDENTICAL in all four -- detaching changes only the")
print("  backward graph. Gradients must NOT be identical.\n")
print(f"  {'variant':<24}{'loss':>12}{'|grad| all':>13}{'|g| merge.Ws':>15}"
      f"{'|g| s_star':>13}{'|g| dec.WK':>13}")
print("  " + "-"*90)
ref_loss, ref_g = None, None
res12 = []
for nm, op in variants12:
    tape_reset()
    PNv = make_param_nodes(P_trained)
    O = rlt_forward_ad(PNv, cfg, SEQJ, TGTJ, op)
    backward(O["loss"])
    gall = math.sqrt(sum(x*x for k in PNv for x in
                         (_flatten(PNv[k].grad) if PNv[k].grad is not None else [0.0])))
    gws = l2(_flatten(PNv["merge.Ws"].grad)) if PNv["merge.Ws"].grad else 0.0
    gss = l2(PNv["s_star"].grad) if PNv["s_star"].grad else 0.0
    gwk = l2(_flatten(PNv["dec.0.WK"].grad)) if PNv["dec.0.WK"].grad else 0.0
    res12.append((nm, O["loss"].val[0], gall, gws, gss, gwk))
    print(f"  {nm:<24}{O['loss'].val[0]:>12.9f}{gall:>13.6f}{gws:>15.6f}"
          f"{gss:>13.6f}{gwk:>13.6f}")

losses12 = [r[1] for r in res12]
print(f"\n  max spread in forward loss: {max(losses12)-min(losses12):.3e}  <- zero, as required")
print(f"  full-BPTT |grad| = {res12[0][2]:.6f}")
for nm, _, gall, gws, gss, gwk in res12[1:]:
    print(f"    {nm:<22} keeps {100*gall/res12[0][2]:6.2f}% of the total gradient norm, "
          f"{100*gws/max(res12[0][3],1e-12):6.2f}% of merge.Ws")

print(f"""
  ==> claim C7 CONFIRMED. All four give bit-identical forward probabilities and
      materially different gradients. 'Detach s only' is NOT full BPTT: it still
      carries {100*res12[1][2]/res12[0][2]:.1f}% of the gradient norm through the KV path. 'Detach
      both' is the only one that truly cuts the boundary, and even it leaves the
      encoder-side path (App. B, B.4) intact -- notice s_star's gradient goes to
      {res12[3][4]:.0e} while the model-wide gradient stays at {res12[3][2]:.3f}, because the
      encoder and memory still receive gradient from every token's loss.""")

# =============================================================================
# 12.3  GRADIENT REACH vs CACHE REACH -- claim C10
# =============================================================================
print("\n" + "=" * 78)
print("C10: 'SWA eviction ... is not itself a stop-gradient operation'")
print("=" * 78)
tape_reset()
PNr = make_param_nodes(P_trained)
Or = rlt_forward_ad(PNr, cfg, SEQJ, TGTJ)
Sr = Or["states"]
T_last = len(SEQJ)

print(f"  W={cfg['W']}, so position 1's KV is EVICTED from the cache at t=3 and is")
print(f"  never directly readable again. Does gradient still reach it?\n")
print(f"  {'j':>4}{'in SWA window at t=9?':>24}{'||d s_9 / d s_j||_F':>22}")
tr_r = Trace(True); rlt_forward(P_trained, cfg, SEQJ, tr=tr_r)
win_last = tr_r[f"t{T_last}.dec0.window_pos"]
for j in range(1, T_last):
    Jj = jacobian(Sr[T_last-1], Sr[j-1], PNr, d)
    print(f"  {j:>4}{('YES' if j in win_last else 'no  (evicted)'):>24}{fro(Jj):>22.6e}")

# gradient of the FULL loss w.r.t. the first token's embedding row
tape_reset()
PNr = make_param_nodes(P_trained)
Or = rlt_forward_ad(PNr, cfg, SEQJ, TGTJ)
backward(Or["loss"])
row0 = PNr["E_tok"].grad[SEQJ[0]]
print(f"""
  gradient w.r.t. the embedding row of token x_1 (BOS): ||g|| = {l2(row0):.6e}
  cache reach at t={T_last}: positions {win_last}   ({cfg['W']} of {T_last})
  gradient reach at t={T_last}: positions {list(range(1, T_last+1))}   (all {T_last})

  ==> claim C10 CONFIRMED. Every ||ds_9/ds_j|| above is strictly nonzero,
      including j=1..6 whose KV left the cache long ago. Eviction bounds what a
      later step can READ; it does not bound what the backward pass must
      DIFFERENTIATE, because those entries already shaped the states that
      survived. The paper's consequence follows directly: "The inference cache
      size therefore does not bound full-BPTT activation storage.\"""")

  forward loss must be IDENTICAL in all four -- detaching changes only the
  backward graph. Gradients must NOT be identical.

  variant                         loss   |grad| all   |g| merge.Ws   |g| s_star   |g| dec.WK
  ------------------------------------------------------------------------------------------
  full BPTT (reference)    0.229472430     0.533978       0.020688     0.000011     0.028458
  detach s only            0.229472430     0.537102       0.015191     0.000000     0.028049
  detach KV only           0.229472430     0.566386       0.017819     0.000014     0.086035
  detach BOTH  (C.1)       0.229472430     0.499257       0.017227     0.000000     0.087207

  max spread in forward loss: 0.000e+00  <- zero, as required
  full-BPTT |grad| = 0.533978
    detach s only          keeps 100.59% of the total gradient norm,  73.43% of merge.Ws
    detach KV only         keeps 106.07% of the total gradient norm,  86.13% of merge.Ws
    detach BOTH  (C.1)     keeps  93.50% of 

In [29]:
# =============================================================================
# 12.4  CORRECTION to the wording in 12.2
# =============================================================================
print("""The table in 12.2 reports 'detach s only' at 100.59% and 'detach KV only' at
106.07% of the full-BPTT gradient norm. Removing a gradient path made the norm
go UP. The percentages are right; calling them 'the fraction of the gradient
that survives' was wrong, and that phrasing should not stand.

Why it happens: ||g|| is not monotone under path removal. The total gradient is
a SUM of path contributions, and contributions can have opposite signs. Deleting
a partially-cancelling path leaves a larger residual:

    ||a + b||  <  ||a||     whenever  <a,b>  <  -||b||^2 / 2

So a truncation scheme can be GRADIENT-INFLATING, not merely gradient-shrinking.
The correct statement of claim C7 does not depend on the direction of the change
-- only on the fact that the gradients differ at all while the forward values do
not. Let us measure the thing that actually settles it: the angle between each
truncated gradient and the true one.""")

tape_reset(); PNa = make_param_nodes(P_trained)
Oa = rlt_forward_ad(PNa, cfg, SEQJ, TGTJ, {}); backward(Oa["loss"])
gref = {k: (_flatten(PNa[k].grad) if PNa[k].grad is not None else [0.0]*len(_flatten(P_trained[k])))
        for k in P_trained}
gref_flat = [x for k in sorted(gref) for x in gref[k]]

print(f"\n  {'variant':<24}{'cos(g, g_full)':>17}{'||g||/||g_full||':>19}"
      f"{'||g - g_full||':>17}")
print("  " + "-"*77)
for nm, op in variants12:
    tape_reset(); PNv = make_param_nodes(P_trained)
    Ov = rlt_forward_ad(PNv, cfg, SEQJ, TGTJ, op); backward(Ov["loss"])
    gv = {k: (_flatten(PNv[k].grad) if PNv[k].grad is not None else [0.0]*len(gref[k]))
          for k in P_trained}
    gv_flat = [x for k in sorted(gv) for x in gv[k]]
    c = cosine(gv_flat, gref_flat)
    print(f"  {nm:<24}{c:>17.9f}{l2(gv_flat)/l2(gref_flat):>19.6f}"
          f"{l2(vsub(gv_flat, gref_flat)):>17.6f}")

print("""
  cos = 1.000000000 exactly would mean 'same gradient direction'. None of the
  truncations achieve it. That is the claim, stated in the form that cannot be
  confounded by a norm going the wrong way:

  ==> C7: truncated BPTT produces a gradient that points in a DIFFERENT
      DIRECTION from full BPTT, while leaving every forward probability
      bit-identical. A scheme validated by checking forward values -- which is
      the natural thing to check -- would pass while training a different model.
      The paper says exactly this in section 5.3: "matching forward
      probabilities at one parameter value is not sufficient to establish
      matching policy gradients.\"""")

The table in 12.2 reports 'detach s only' at 100.59% and 'detach KV only' at
106.07% of the full-BPTT gradient norm. Removing a gradient path made the norm
go UP. The percentages are right; calling them 'the fraction of the gradient
that survives' was wrong, and that phrasing should not stand.

Why it happens: ||g|| is not monotone under path removal. The total gradient is
a SUM of path contributions, and contributions can have opposite signs. Deleting
a partially-cancelling path leaves a larger residual:

    ||a + b||  <  ||a||     whenever  <a,b>  <  -||b||^2 / 2

So a truncation scheme can be GRADIENT-INFLATING, not merely gradient-shrinking.
The correct statement of claim C7 does not depend on the direction of the change
-- only on the fact that the gradients differ at all while the forward values do
not. Let us measure the thing that actually settles it: the angle between each
truncated gradient and the true one.

  variant                    cos(g, g_full)   ||g||/||g_full||   |

# §12 — Phase 15/16: the ablation lab

**QUESTION.** Section 7 showed the two channels exist and carry signal. Does either of them *buy accuracy* on a task that needs state?

**SETUP.** Running parity, 8 bits, chance 50%. Four configurations × 3 seeds, each trained identically. The only differences are $\alpha$ and $W$ — same parameter count, same block count, same data, same optimizer, same steps. Then a depth-split sweep at fixed total blocks, which is the experiment §3.3 raises and declines to run.

**Reporting rule.** Mean ± spread over seeds, and no claim of a difference that the seed spread does not support.

In [30]:
# =============================================================================
# 15/16  ABLATION LAB -- every run reproducible from a config
# =============================================================================
STEPS, BS, LR, SEEDS = 100, 12, 0.02, [0, 1, 2]
EVAL_N = 64
t_lab = time.time()

def run_config(name, cfg_over, opts, seed):
    c = make_cfg(seed=seed, **cfg_over)
    Pt, P0, hist, _ = train(c, opts, steps=STEPS, bs=BS, lr=LR, seed=seed, log_every=0)
    trl, tra = evaluate(Pt, c, train_data[:EVAL_N], opts)
    tel, tea = evaluate(Pt, c, test_data[:EVAL_N], opts)
    return dict(name=name, seed=seed, config=dict(c), opts=dict(opts),
                train_loss=trl, train_acc=tra, test_loss=tel, test_acc=tea,
                final_batch_loss=hist[-1]["loss"],
                loss_curve=[h["loss"] for h in hist])

GRID = [("full        a=0.1 W=3", {"W": 3}, {"alpha": 0.1}),
        ("state cut   a=0.0 W=3", {"W": 3}, {"alpha": 0.0}),
        ("cache cut   a=0.1 W=1", {"W": 1}, {"alpha": 0.1}),
        ("BOTH cut    a=0.0 W=1", {"W": 1}, {"alpha": 0.0})]

print(f"main grid: {len(GRID)} configs x {len(SEEDS)} seeds, {STEPS} steps, batch {BS}\n")
runs = []
for name, cov, op in GRID:
    accs = []
    for sd in SEEDS:
        r = run_config(name, cov, op, sd)
        runs.append(r); accs.append(r["test_acc"])
        print(f"  {name}  seed {sd}: test acc {r['test_acc']:6.2%}  "
              f"loss {r['test_loss']:.4f}   [{time.time()-t_lab:5.1f}s]")
    print(f"    -> mean {mean(accs):.2%}  spread {vmax(accs)-vmin(accs):.2%}\n")

print("=" * 78)
print("MAIN GRID RESULT   (running parity, chance = 50%)")
print("=" * 78)
print(f"  {'config':<24}{'test acc mean':>15}{'min':>9}{'max':>9}{'spread':>9}{'test loss':>12}")
print("  " + "-"*78)
summary = {}
for name, _, _ in GRID:
    rs = [r for r in runs if r["name"] == name]
    a = [r["test_acc"] for r in rs]; L = [r["test_loss"] for r in rs]
    summary[name] = (mean(a), vmin(a), vmax(a), mean(L))
    print(f"  {name:<24}{mean(a):>15.2%}{vmin(a):>9.2%}{vmax(a):>9.2%}"
          f"{vmax(a)-vmin(a):>9.2%}{mean(L):>12.4f}")

full_a  = summary["full        a=0.1 W=3"]
both_a  = summary["BOTH cut    a=0.0 W=1"]
state_a = summary["state cut   a=0.0 W=3"]
cache_a = summary["cache cut   a=0.1 W=1"]
gap = full_a[0] - both_a[0]
maxspread = max(s[2]-s[1] for s in summary.values())
print(f"""
  full vs fully-non-recurrent:  {full_a[0]:.2%} vs {both_a[0]:.2%}   gap = {gap:+.2%}
  largest within-config seed spread: {maxspread:.2%}
  verdict: the gap is {"LARGER" if gap > maxspread else "SMALLER"} than the seed spread""")

# ------------------------------------------------- depth-split sweep ---------
print("\n" + "=" * 78)
print("DEPTH SPLIT at fixed total logical blocks -- the experiment sec 3.3 defers")
print("=" * 78)
print("  [OUR DESIGN, not the paper's: the paper reports no experiments at all.]\n")
DEPTH = [("1+1  (2 blocks/token)", {"L_E": 1, "L_D": 1}),
         ("2+1  (3 blocks/token)", {"L_E": 2, "L_D": 1}),
         ("1+2  (3 blocks/token)", {"L_E": 1, "L_D": 2}),
         ("3+1  (4 blocks/token)", {"L_E": 3, "L_D": 1}),
         ("1+3  (4 blocks/token)", {"L_E": 1, "L_D": 3})]
print(f"  {'split':<24}{'test acc':>11}{'test loss':>12}{'params':>9}{'state path':>13}")
print("  " + "-"*70)
depth_runs = []
for name, cov in DEPTH:
    r = run_config(name, cov, {"alpha": 0.1}, seed=0)
    depth_runs.append(r)
    c = make_cfg(**cov)
    npar = n_params(init_params(c))
    print(f"  {name:<24}{r['test_acc']:>11.2%}{r['test_loss']:>12.4f}{npar:>9}"
          f"{str(N_BITS+1)+' x '+str(c['L_D']):>13}   [{time.time()-t_lab:5.1f}s]")

print(f"""
  'state path' is the paper's section 3.3 quantity: T x L_D decoder blocks along
  the recurrent chain. 1+2 and 2+1 execute the SAME 3 blocks per token but the
  1+2 split has TWICE the recurrent depth.""")

# ---------------------------------------------------------- persist ----------
with open("config.json", "w") as f:
    json.dump({"grid": [{"name": n, "cfg_over": c, "opts": o} for n, c, o in GRID],
               "steps": STEPS, "batch": BS, "lr": LR, "seeds": SEEDS,
               "task": "running parity, 8 bits, chance 0.5"}, f, indent=1)
with open("metrics.json", "w") as f:
    json.dump([{k: v for k, v in r.items() if k != "loss_curve"} for r in runs + depth_runs],
              f, indent=1)
print(f"\nwrote config.json and metrics.json ({len(runs)+len(depth_runs)} runs)")
print(f"total lab time: {time.time()-t_lab:.1f}s")

main grid: 4 configs x 3 seeds, 100 steps, batch 12

  full        a=0.1 W=3  seed 0: test acc 86.63%  loss 0.4135   [ 12.3s]
  full        a=0.1 W=3  seed 1: test acc 95.66%  loss 0.1378   [ 24.7s]
  full        a=0.1 W=3  seed 2: test acc 69.97%  loss 0.4783   [ 37.0s]
    -> mean 84.09%  spread 25.69%

  state cut   a=0.0 W=3  seed 0: test acc 76.91%  loss 0.3806   [ 49.4s]
  state cut   a=0.0 W=3  seed 1: test acc 64.93%  loss 0.5366   [ 61.7s]
  state cut   a=0.0 W=3  seed 2: test acc 64.24%  loss 0.5073   [ 74.1s]
    -> mean 68.69%  spread 12.67%

  cache cut   a=0.1 W=1  seed 0: test acc 62.85%  loss 0.6364   [ 86.3s]
  cache cut   a=0.1 W=1  seed 1: test acc 72.22%  loss 0.4940   [ 98.5s]
  cache cut   a=0.1 W=1  seed 2: test acc 64.58%  loss 0.5285   [110.6s]
    -> mean 66.55%  spread 9.38%

  BOTH cut    a=0.0 W=1  seed 0: test acc 61.11%  loss 0.5198   [122.8s]
  BOTH cut    a=0.0 W=1  seed 1: test acc 62.33%  loss 0.5418   [134.9s]
  BOTH cut    a=0.0 W=1  seed 2: test ac

In [32]:
# =============================================================================
# 16.2  READING THE GRID PROPERLY
# =============================================================================
# The cell above printed "the gap is SMALLER than the seed spread" and therefore
# implied nothing was shown. That comparison is the wrong test: it puts a
# BETWEEN-group difference of means against a WITHIN-group RANGE, which are not
# commensurable. A range grows with n; a difference of means does not. Redone
# with an exact test.
from itertools import combinations

groups = {}
for name, _, _ in GRID:
    groups[name] = [r["test_acc"] for r in runs if r["name"] == name]

def exact_p(a, b):
    """One-sided exact permutation test on the difference of means.
    n=3 vs n=3 -> 20 distinct splits, so the smallest achievable p is 1/20 = 0.05."""
    obs = mean(a) - mean(b)
    pool = a + b
    ge = 0
    for idx in combinations(range(len(pool)), len(a)):
        ga = [pool[i] for i in idx]
        gb = [pool[i] for i in range(len(pool)) if i not in idx]
        if mean(ga) - mean(gb) >= obs - 1e-12: ge += 1
    return obs, ge / len(list(combinations(range(len(pool)), len(a))))

print("  per-seed test accuracies\n")
for name in groups:
    print(f"  {name:<24}{'  '.join(f'{a:.2%}' for a in sorted(groups[name], reverse=True))}"
          f"     mean {mean(groups[name]):.2%}")

full_g = groups["full        a=0.1 W=3"]
both_g = groups["BOTH cut    a=0.0 W=1"]

print("\n  exact one-sided permutation test, each ablation vs the full model")
print(f"  {'comparison':<40}{'diff of means':>15}{'exact p':>11}{'separated?':>13}")
print("  " + "-"*79)
for name in groups:
    if name.startswith("full"): continue
    obs, p = exact_p(full_g, groups[name])
    sep = vmin(full_g) > vmax(groups[name])
    print(f"  {'full  vs  ' + name.strip():<40}{obs:>15.2%}{p:>11.3f}{str(sep):>13}")

print(f"""
  full model:      min = {vmin(full_g):.2%}
  fully-cut model: max = {vmax(both_g):.2%}
  -> COMPLETE SEPARATION: every full-model seed beats every non-recurrent seed.

  With n=3 vs n=3 there are only 20 possible relabelings, so complete separation
  gives exact one-sided p = 1/20 = 0.050. That is the SMALLEST p this design can
  produce. The result is therefore at the edge of what three seeds can show:
  suggestive, consistent, and underpowered. Calling it 'significant' would be
  overclaiming; calling it 'nothing' -- as the previous cell's heuristic did --
  was wrong in the other direction.

  What IS solid, because it does not depend on a p-value:
    the four configurations order MONOTONICALLY in both accuracy and loss,
    in the direction the architecture predicts --

      full {mean(groups['full        a=0.1 W=3']):.2%}  >  state cut {mean(groups['state cut   a=0.0 W=3']):.2%}  >  cache cut {mean(groups['cache cut   a=0.1 W=1']):.2%}  >  both cut {mean(groups['BOTH cut    a=0.0 W=1']):.2%}

    and cutting BOTH channels is worse than cutting either one alone. That
    ordering is exactly what section 7.1 predicted from the forward-pass
    structure, now reproduced in a downstream task metric.

  Honest limitation: one 8-bit parity task, d=8, 100 steps, 3 seeds. This
  establishes a direction, not an effect size.""")

# ------------------------------------------------- depth split, honestly -----
print("\n" + "="*78)
print("THE DEPTH-SPLIT SWEEP IS UNDERPOWERED AND CONFOUNDED -- reported as such")
print("="*78)
for r in depth_runs:
    print(f"  {r['name']:<24}test acc {r['test_acc']:>7.2%}   loss {r['test_loss']:.4f}")
print(f"""
  Two problems, both fatal to any conclusion:

  1. CONFOUNDED BY TRAINING BUDGET. Every run got the same {STEPS} steps, but
     the models differ in size (1728 to 3088 parameters). Accuracy falls
     MONOTONICALLY as the model grows, which is the signature of underfitting
     at a fixed step count, not of depth being harmful.

  2. n = 1. The 2+1 vs 1+2 comparison is the one that isolates recurrent depth
     -- same blocks per token, 2x the state path -- and it reads
     {[r['test_acc'] for r in depth_runs if r['name'].startswith('2+1')][0]:.2%} vs {[r['test_acc'] for r in depth_runs if r['name'].startswith('1+2')][0]:.2%}. From section 16.2 we know the seed spread on this
     task reaches 25 points. A 0.7-point difference at n=1 is noise.

  ==> NO EFFECT DETECTED, and the design could not have detected one. This is
      reported as a null result about the EXPERIMENT, not about the
      architecture. Fixing it needs equal-compute budgets (or train-to-
      convergence) and several seeds -- more compute than pure Python affords
      here. The paper itself declines to run this experiment and writes
      'structural depth alone is not a reasoning guarantee'; nothing here
      contradicts or supports that.""")

  per-seed test accuracies

  full        a=0.1 W=3   95.66%  86.63%  69.97%     mean 84.09%
  state cut   a=0.0 W=3   76.91%  64.93%  64.24%     mean 68.69%
  cache cut   a=0.1 W=1   72.22%  64.58%  62.85%     mean 66.55%
  BOTH cut    a=0.0 W=1   65.10%  62.33%  61.11%     mean 62.85%

  exact one-sided permutation test, each ablation vs the full model
  comparison                                diff of means    exact p   separated?
  -------------------------------------------------------------------------------
  full  vs  state cut   a=0.0 W=3                  15.39%      0.100        False
  full  vs  cache cut   a=0.1 W=1                  17.53%      0.100        False
  full  vs  BOTH cut    a=0.0 W=1                  21.24%      0.050         True

  full model:      min = 69.97%
  fully-cut model: max = 65.10%
  -> COMPLETE SEPARATION: every full-model seed beats every non-recurrent seed.

  With n=3 vs n=3 there are only 20 possible relabelings, so complete separation
  give

# §13 — Phase 14: paper claim verification

Every major technical claim, tested rather than assumed. The remaining ones (C1, C5, C6, C11) are checked here; C2, C3, C4 were settled in §8, C7/C9/C10 in §11, C8 in §2.5 and §7.

In [31]:
# =============================================================================
# 14  REMAINING CLAIMS: C1, C5, C6, C11
# =============================================================================
VERDICTS = {}

# ---- C1: per-token block count fixed; state path grows as t * L_D -----------
print("C1  'per-token block count stays fixed; the state path is t*L_D blocks'  (sec 3.3, Fig 2)")
print(f"  {'L_E':>4}{'L_D':>5}{'T':>4}{'enc blocks':>12}{'dec blocks':>12}"
      f"{'per token':>11}{'state path':>12}{'predicted':>11}")
c1ok = True
for LE, LD in [(1,1), (2,1), (1,2), (3,2)]:
    for T in [4, 9]:
        c = make_cfg(L_E=LE, L_D=LD)
        Pc = init_params(c)
        reset_blocks()
        rlt_forward(Pc, c, [0]+[1+(i%2) for i in range(T-1)])
        per = (BLOCKS["enc"]+BLOCKS["dec"])/T
        path, pred = BLOCKS["dec"], T*LD
        c1ok &= (abs(per-(LE+LD)) < 1e-9) and (path == pred)
        print(f"  {LE:>4}{LD:>5}{T:>4}{BLOCKS['enc']:>12}{BLOCKS['dec']:>12}"
              f"{per:>11.1f}{path:>12}{pred:>11}")
VERDICTS["C1"] = ("CONFIRMED" if c1ok else "VIOLATED",
                  "block count per token = L_E+L_D exactly; decoder blocks on the chain = t*L_D exactly")
print(f"  ==> {VERDICTS['C1'][0]}: counted by instrumenting the actual calls, not re-derived.\n")

# ---- C5: cross-attention is prefix-restricted PER DECODER POSITION ----------
print("C5  'at decoder position t, attention is restricted to M_(<=t)'  (sec 2.4)")
trc = Trace(True)
rlt_forward(P, cfg, SEQ8, tr=trc)
c5ok = all(trc[f"t{t}.dec0.mem_pos"] == list(range(1, t+1)) for t in range(1, len(SEQ8)+1))
for t in [1, 4, 8]:
    print(f"    t={t}: cross-attn reads M positions {trc[f't{t}.dec0.mem_pos']}"
          f"   (available: {list(range(1, len(SEQ8)+1))})")
# the destructive test: would reading the whole prefill change the model?
def leaky_forward(P, cfg, tokens):
    """A 'faster kernel' that lets every decoder position read ALL of M."""
    E = encoder_prefill(P, cfg, tokens)
    M = [[] for _ in range(cfg["G"])]
    for i in range(len(tokens)): memory_append(P, cfg, E[i], i+1, M)
    C_D = [[] for _ in range(cfg["L_D"])]
    s, out = list(P["s_star"]), []
    for i in range(len(tokens)):
        u, _, _, _ = gated_merge(P, cfg, E[i], s, {})
        z = u
        for l in range(cfg["L_D"]):
            # NOTE the bug we are deliberately introducing: t -> len(tokens)
            z, C_D[l], _ = decoder_block(P, cfg, l, z, i+1, M, C_D[l])
        s = z; out.append(list(s))
    return out
Mfull = [[] for _ in range(cfg["G"])]
Efull = encoder_prefill(P, cfg, SEQ8)
for i in range(len(SEQ8)): memory_append(P, cfg, Efull[i], i+1, Mfull)
C_leak = [[] for _ in range(cfg["L_D"])]
s_l, leak = list(P["s_star"]), []
for i in range(len(SEQ8)):
    u, _, _, _ = gated_merge(P, cfg, Efull[i], s_l, {})
    z = u
    for l in range(cfg["L_D"]):
        z, C_leak[l], _ = decoder_block(P, cfg, l, z, len(SEQ8), Mfull, C_leak[l])  # reads ALL of M
    s_l = z; leak.append(list(s_l))
Rok = rlt_forward(P, cfg, SEQ8)
same, wdiff, _ = allclose(leak, Rok["s"], atol=1e-12, rtol=1e-9)
print(f"    a kernel that reads the whole prefill instead: states differ by "
      f"max {wdiff:.3e}  (identical? {same})")
VERDICTS["C5"] = ("CONFIRMED" if c5ok and not same else "VIOLATED",
                  f"slicing is enforced per position; ignoring it changes states by {wdiff:.2e}")
print(f"  ==> {VERDICTS['C5'][0]}: prefix restriction is a MODEL property. The paper's")
print(f"      'a faster kernel that reads future entries changes the model' is literally true.\n")

# ---- C11: encoder memory never depends on decoder states -------------------
print("C11 'this global memory depends on encoder representations, not decoder states'  (sec 2.2)")
P_alt = clone_params(P)
P_alt["s_star"] = vscale(-3.0, P["s_star"])          # a big change to the decoder side
E1 = encoder_prefill(P, cfg, SEQ8)
E2 = encoder_prefill(P_alt, cfg, SEQ8)
M1 = [[] for _ in range(cfg["G"])]; M2 = [[] for _ in range(cfg["G"])]
for i in range(len(SEQ8)):
    memory_append(P, cfg, E1[i], i+1, M1); memory_append(P_alt, cfg, E2[i], i+1, M2)
k1 = [[h for h in e[1]] for e in M1[0]]; k2 = [[h for h in e[1]] for e in M2[0]]
v1 = [[h for h in e[2]] for e in M1[0]]; v2 = [[h for h in e[2]] for e in M2[0]]
okK, wK, _ = allclose(k1, k2, atol=0.0, rtol=0.0)
okV, wV, _ = allclose(v1, v2, atol=0.0, rtol=0.0)
# and the states DID change, proving the perturbation was real
Rs1, Rs2 = rlt_forward(P, cfg, SEQ8), rlt_forward(P_alt, cfg, SEQ8)
moved = l2(vsub(Rs1["s"][0], Rs2["s"][0]))
print(f"    s_star scaled by -3.  encoder memory K identical? {okK} (max|d|={wK:.1e})")
print(f"                          encoder memory V identical? {okV} (max|d|={wV:.1e})")
print(f"    control -- did anything change at all?  ||s_1 - s_1'|| = {moved:.6f}  (yes)")
VERDICTS["C11"] = ("CONFIRMED" if okK and okV and moved > 1e-9 else "VIOLATED",
                   "M is bit-identical under a decoder-side perturbation that does move the states")
print(f"  ==> {VERDICTS['C11'][0]}\n")

# ---- C6: cached states under old parameters are not current-policy states ---
print("C6  'after an optimizer update, previously computed KV/states cease to be")
print("     current-policy values'  (sec 5.4, App. C)")
R_old = rlt_forward(P_init,    cfg, SEQJ)     # 'sampler' states, old weights
R_new = rlt_forward(P_trained, cfg, SEQJ)     # 'trainer' states, current weights
drift_s = [l2(vsub(R_old["s"][i], R_new["s"][i]))/max(l2(R_new["s"][i]),1e-12)
           for i in range(len(SEQJ))]
kl_pol = [kl(softmax(R_new["logits"][i]), softmax(R_old["logits"][i])) for i in range(len(SEQJ))]
print(f"    {'t':>4}{'rel ||s_old - s_new||':>24}{'KL(new || old)':>18}")
for i in range(len(SEQJ)):
    print(f"    {i+1:>4}{drift_s[i]:>24.4f}{kl_pol[i]:>18.4f}")
VERDICTS["C6"] = ("CONFIRMED", f"mean relative state drift {mean(drift_s):.3f}, "
                              f"mean KL {mean(kl_pol):.3f} after 150 optimizer steps")
print(f"""  ==> CONFIRMED. Reusing the sampler's states would evaluate a policy that
      differs from the current one by KL {mean(kl_pol):.3f} nats per token. This is the
      quantitative content of the paper's replay contract: "A sampler's old
      hidden states cannot replace current-policy replay.\"""")

print("\n" + "=" * 78)
for cid in ["C1", "C5", "C6", "C11"]:
    print(f"  {cid:<5}{VERDICTS[cid][0]:<12}{VERDICTS[cid][1]}")

C1  'per-token block count stays fixed; the state path is t*L_D blocks'  (sec 3.3, Fig 2)
   L_E  L_D   T  enc blocks  dec blocks  per token  state path  predicted
     1    1   4           4           4        2.0           4          4
     1    1   9           9           9        2.0           9          9
     2    1   4           8           4        3.0           4          4
     2    1   9          18           9        3.0           9          9
     1    2   4           4           8        3.0           8          8
     1    2   9           9          18        3.0          18         18
     3    2   4          12           8        5.0           8          8
     3    2   9          27          18        5.0          18         18
  ==> CONFIRMED: counted by instrumenting the actual calls, not re-derived.

C5  'at decoder position t, attention is restricted to M_(<=t)'  (sec 2.4)
    t=1: cross-attn reads M positions [1]   (available: [1, 2, 3, 4, 5, 6, 7, 8])
    t=4: c

# §14 — Phase 18: findings

A dissection of *Recurrent Looped Transformer* (Zhang, Feng, Qin, Sept 2026), in pure Python — `math` and nothing else, across every cell above.

---

## 0. The frame

**The paper reports no experiments** (§1, §3.2, §8; grep confirms zero hits for `512`, `1365`, `4+4`, `accuracy`, `we train`). So "reproduction" here means **verifying propositions and structural claims exactly**, which is the one thing a pure-Python implementation does better than a framework: we control every float, so claims can be settled at 0 ulp rather than "within tolerance."

The config often attributed to this paper — width 512, FFN 1365, 4 heads, splits 4+4…8+0 — is **not in it**. Every task and configuration in this notebook is ours and labelled as ours.

---

## 1. Claim verification — all 11

| ID | Claim | Verdict | Evidence |
|---|---|---|---|
| **C1** | per-token blocks fixed at $L_E{+}L_D$; state path $= t\,L_D$ | ✅ CONFIRMED | counted by instrumenting actual calls, 8 configs, exact |
| **C2** | moving the prompt/response split changes nothing | ✅ CONFIRMED | 0 ulp at every split $T \in \{1,3,6,9,11\}$ |
| **C3** | $H_t$ depends only on $x_{1:t}$ (Prop B.1) | ✅ CONFIRMED | change $x_{k}$, all earlier states bit-identical; first divergence exactly at $k$ |
| **C4** | both $s$ and $C^D$ cross the boundary unreset | ✅ CONFIRMED | cache entry 6 survives the $T{=}6$ boundary |
| **C5** | encoder memory prefix-restricted **per decoder position** | ✅ CONFIRMED | a "faster kernel" reading all of $M$ moves states by 1.65 |
| **C6** | cached states under old params ≠ current-policy states | ✅ CONFIRMED | KL 1.523 nats/token drift after 150 steps |
| **C7** | detaching $s$ alone is not full BPTT | ✅ CONFIRMED | forward bit-identical, $\cos(g, g_{\text{full}}) = 0.925$ |
| **C8** | retention + current == next read window | ✅ CONFIRMED | exact, all $W \in \{1,2,3,8\}$, mask level **and** live cache |
| **C9** | an $\partial s/\partial s$-only product misses the KV paths | ✅ **CONFIRMED, large** | true derivative up to **8904×** the product |
| **C10** | gradient reach > cache reach | ✅ CONFIRMED | $\|\partial s_9/\partial s_j\| \neq 0$ for all $j$, though $W{=}3$ |
| **C11** | $M$ never depends on decoder states | ✅ CONFIRMED | bit-identical under a decoder perturbation that does move states |

---

## 2. The result that changes how you'd design experiments on this architecture

**There are two recurrent channels, and $\alpha = 0$ only severs one.**

Injecting $\delta$ into $u_k$ — inside the decoder, bypassing encoder and memory — and measuring $\|s_t - s_t^{\text{base}}\|$:

| config | reach for $t > k$ |
|---|---|
| α=0.1, W=3 | all 8 later steps, decaying |
| **α=0, W=3** | **exactly 2 steps = $W{-}1$, then bit-exact zero** |
| α=0.1, W=1 | all 8 later steps |
| α=0, W=1 | **bit-exact zero everywhere** |

Row 2 settles it. With the state channel fully severed, the layerwise SWA cache still carries information forward for $W-1$ tokens. **The only true non-recurrence control is α=0 ∧ W=1.**

The two channels also have different *characters*: the cache channel is **strong but bounded** (2.57 → 1.25 → 1.16 → hard zero), the state channel **weak but unbounded** (3.33 → 0.13 → 3.6e-3 → … ≈40× decay per step, never zero).

---

## 3. Claim C9, quantified

The paper says a product of $\partial s_t/\partial s_{t-1}$ factors "generally misses paths through decoder KV." Measured against the true total derivative from the tape:

| | $t = j{+}1$ | $t = j{+}2$ | $t = j{+}3$ | $t = j{+}5$ |
|---|---|---|---|---|
| ratio $\|J_{\text{true}}\|/\|J_{\text{prod}}\|$, at init | **1.00** | 8–12× | 283–586× | **6186–8904×** |
| after training | **1.00** | 1.05–1.45× | 12–22× | 22–35× |

Exactly 1.00 at one step — as it must be, since with $L_D{=}1$ the cache entry written at step $j$ is built from $z^0 = u_j$, which does not depend on $s_j$. Beyond one step the naive product understates the true sensitivity by **three to four orders of magnitude**. Any vanishing-gradient or stability argument built on the $\partial s/\partial s$ product alone is measuring the wrong object.

---

## 4. Findings the paper does not state

**RMSNorm's $\epsilon$ is not scale-invariant near zero.** Output RMS falls to 0.457 at $\mathrm{rms}(x)=1.6\times10^{-3}$ and 0.051 at $1.6\times10^{-4}$. A contracting state keeps shrinking *through* eq (2.9). This is a concrete mechanism behind the paper's own §3.3 caveat about contraction.

**The merge is scale-invariant in $s_{t-1}$.** Eq (2.9) normalizes before anything else touches the state, and both (2.10) and (2.11) consume $r$, never $s$. So **only the direction of $s_{t-1}$ enters** — $\|s_t\|$ is not an information channel, and any experiment perturbing only the magnitude of $s$ is a no-op by construction. Found because noise at std 1 and std 5 gave identical results, which looked like a bug.

**`s := 0` and `α = 0` are the same intervention.** $r = \mathrm{RMSNorm}(0) = 0 \Rightarrow \mathrm{fb} = 0$, exactly as $\alpha{=}0$ does. Verified bit-identical. They are not two independent probes of the recurrence; they are one, counted twice.

**Truncated BPTT can be gradient-*inflating*.** "Detach s only" has 100.6% of the full gradient norm — removing a path made the norm grow, because path contributions partially cancel. The norm is the wrong statistic; the **angle** is the right one: $\cos(g,g_{\text{full}}) = 0.925$, 0.835, 0.859 for the three truncations. A truncation validated by checking forward values — the natural thing to check — passes while training a different model.

**Proposition 3.1's own caveat, reproduced by accident.** The taped and plain forwards agree to $8.9\times10^{-16}$, not bit-exactly, because `rmsnorm` divides by $\sqrt{\text{ms}+\epsilon}$ while `n_rmsnorm` multiplies by the reciprocal. Same mathematics, different rounding — *"Different kernels and precision choices can still cause numerical discrepancies"*, at 4 ulp.

---

## 5. Does the recurrence buy accuracy?

Running parity, 8 bits, chance 50%. Four configs × 3 seeds, identical in every other respect:

| config | test acc (mean) | per-seed | exact $p$ vs full |
|---|---|---|---|
| full α=0.1 W=3 | **84.09%** | 95.66 / 86.63 / 69.97 | — |
| state cut α=0 W=3 | 68.69% | 76.91 / 64.93 / 64.24 | 0.100 |
| cache cut α=0.1 W=1 | 66.55% | 72.22 / 64.58 / 62.85 | 0.100 |
| both cut α=0 W=1 | 62.85% | 65.10 / 62.33 / 61.11 | **0.050** |

**Complete separation** between full and fully-cut: every full seed beats every cut seed. With $n{=}3$ vs $3$ there are 20 relabelings, so $p = 1/20 = 0.050$ — the *smallest this design can produce*. Suggestive, consistent, and underpowered.

What does not depend on a $p$-value: the four configs order **monotonically** in both accuracy and loss, in the direction §7.1 predicted from the forward-pass structure alone, and cutting both channels is worse than cutting either.

**The depth-split sweep is a null result about the experiment, not the architecture.** Accuracy falls monotonically with model size at a fixed 100-step budget — the signature of underfitting, not of depth being harmful. The one comparison that isolates recurrent depth (2+1 vs 1+2: same blocks/token, 2× state path) reads 72.57% vs 73.26% at $n{=}1$, against a known seed spread of 25 points. **No effect detected, and the design could not have detected one.**

---

## 6. Limitations, stated plainly

- $d{=}8$, $L{=}1{+}1$, $V{=}10$, one synthetic task, 100–150 steps, 3 seeds. This establishes **directions and exact structural facts**, not effect sizes.
- NoPE throughout (RoPE implemented and norm-checked, but not used in training).
- Untied $E$/$D$; the paper's tied "looped" configuration (§2.6) is implemented as a flag but not swept.
- The structural claims (C1–C11) are **not** limited by any of this — they are exact, and several are verified at 0 ulp.

---

## 7. What would come next

1. Equal-**compute** depth splits with ≥5 seeds — the experiment §3.3 raises and declines.
2. A task where the cache channel provably cannot substitute for the state channel (needs dependency range > $W$).
3. The tied configuration, to test whether parameter reuse changes the channel balance.
4. The RL replay contract of §5.3 — C6 measures the drift; the importance-ratio machinery is untouched here.

In [33]:
#@title  utility: sync this notebook -> GitHub  (source-only, outputs stripped)
# Not part of the dissection. Run this BY HAND from the Colab UI -- Colab refuses
# to hand out Secrets to an automated caller ("Secrets can only be fetched when
# running from the Colab UI"), so it must be you who clicks it.
# Add the token under the key icon in the left sidebar, named GITHUB_TOKEN.
# Nothing secret is ever printed or written into the notebook.

import json, os, subprocess

REPO = "Maverick-Ansh/recurrent_looped_transformer_scratch"
NB_NAME = "RLT_dissection.ipynb"
MSG = "Colab notebook: full dissection, Phases 0-18"

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as e:
    TOKEN = None
    print("no GITHUB_TOKEN available:", type(e).__name__, "-", str(e)[:200])

from google.colab import _message
nb = _message.blocking_request("get_ipynb", timeout_sec=120)["ipynb"]

# strip outputs + execution counts: the repo copy is source-only
for c in nb.get("cells", []):
    if c.get("cell_type") == "code":
        c["outputs"] = []
        c["execution_count"] = None
n_code = sum(1 for c in nb["cells"] if c["cell_type"] == "code")
n_md   = sum(1 for c in nb["cells"] if c["cell_type"] == "markdown")
print(f"notebook: {len(nb['cells'])} cells ({n_code} code, {n_md} markdown)")

if not TOKEN:
    os.makedirs("/content/out", exist_ok=True)
    with open(f"/content/out/{NB_NAME}", "w") as f:
        json.dump(nb, f, indent=1)
    print(f"\nwrote /content/out/{NB_NAME}")
    print("add GITHUB_TOKEN to Colab Secrets and re-run to push, or download it")
    print("from the file browser on the left and commit it yourself.")
else:
    subprocess.run("rm -rf /content/_sync", shell=True)
    r = subprocess.run(f"git clone -q https://{TOKEN}@github.com/{REPO}.git /content/_sync",
                       shell=True, capture_output=True, text=True)
    if r.returncode:
        print("clone failed:", r.stderr[-400:])
    else:
        with open(f"/content/_sync/{NB_NAME}", "w") as f:
            json.dump(nb, f, indent=1)
        # the experiment artefacts the notebook wrote, if they are present
        for art in ["trajectory.json", "trajectory.csv", "config.json", "metrics.json"]:
            if os.path.exists(art):
                subprocess.run(f"cp {art} /content/_sync/{art}", shell=True)
        cmds = [
            'git -C /content/_sync config user.email "anshvivek2003@gmail.com"',
            'git -C /content/_sync config user.name  "Maverick-Ansh"',
            'git -C /content/_sync add -A',
            f'git -C /content/_sync commit -q -m "{MSG}\n\n'
            'Co-Authored-By: Claude Opus 5 <noreply@anthropic.com>" || echo "nothing to commit"',
            'git -C /content/_sync push -q origin main',
        ]
        for c in cmds:
            rr = subprocess.run(c, shell=True, capture_output=True, text=True)
            if rr.returncode and "nothing to commit" not in (rr.stdout + rr.stderr):
                print("FAILED:", c.split("-C /content/_sync")[-1][:60], "->", (rr.stderr or rr.stdout)[-300:])
                break
        else:
            print(f"pushed {NB_NAME} + artefacts to https://github.com/{REPO}")

notebook: 44 cells (27 code, 17 markdown)
pushed RLT_dissection.ipynb + artefacts to https://github.com/Maverick-Ansh/recurrent_looped_transformer_scratch
